In [1]:
# ============================================================
# NOTEBOOK 11
# INDICADORES E PRIORIZAÇÃO AGROAMBIENTAL
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


BASE_DIR = Path(
    r"C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao"
)


ARQUIVO_BASE_FINAL = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "integracao_agroambiental"
    / "base_agroambiental_final_soja_centro_oeste_sul_2019_2024.csv"
)


base = pd.read_csv(
    ARQUIVO_BASE_FINAL,
    dtype={
        "codigo_ibge": "string"
    },
    encoding="utf-8-sig"
)


base[
    "codigo_ibge"
] = (
    base[
        "codigo_ibge"
    ]
    .str.zfill(7)
)


print(
    "Dimensão:",
    base.shape
)

print(
    "Municípios:",
    base[
        "codigo_ibge"
    ]
    .nunique()
)

print(
    "Anos:",
    sorted(
        base[
            "ano"
        ]
        .unique()
    )
)

print(
    "Duplicatas:",
    base
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)

Dimensão: (8674, 134)
Municípios: 1505
Anos: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Duplicatas: 0


In [2]:
# ============================================================
# INVENTÁRIO DE VARIÁVEIS
# ============================================================

inventario_colunas = pd.DataFrame(
    {
        "ordem":
            range(
                1,
                len(
                    base.columns
                )
                + 1
            ),

        "coluna":
            base.columns,

        "tipo":
            base
            .dtypes
            .astype("string")
            .values,

        "null":
            base
            .isna()
            .sum()
            .values
    }
)


print(
    "Total de variáveis:",
    len(
        inventario_colunas
    )
)


display(
    inventario_colunas
)

Total de variáveis: 134


,ordem,coluna,tipo,null
0,1,codigo_ibge,string,0
1,2,municipio,object,0
2,3,uf,object,0
3,4,regiao,object,0
4,5,ano,int64,0
...,...,...,...,...
129,130,ctotal_t1_incerteza_pct,float64,0
130,131,soc_t0_incerteza_pct,float64,0
131,132,soc_t1_incerteza_pct,float64,0
132,133,cveg_t0_incerteza_pct,float64,0


In [3]:
# ============================================================
# AGRUPAMENTO PRELIMINAR DAS VARIÁVEIS
# ============================================================

GRUPOS_VARIAVEIS = {
    "producao_agricola": [
        "area_plantada",
        "area_colhida",
        "area_nao_colhida",
        "quantidade_produzida",
        "rendimento",
        "valor_producao",
        "aproveitamento"
    ],

    "clima": [
        "precip",
        "temperatura",
        "umidade"
    ],

    "carbono_solo": [
        "carbono",
        "soil",
        "soc"
    ],

    "cobertura_terra": [
        "cobertura",
        "vegetacao",
        "pastagem",
        "agricultura",
        "floresta"
    ],

    "seeg": [
        "seeg",
        "n2o",
        "co2e_gwp"
    ],

    "brluc": [
        "brluc",
        "origem",
        "persistencia",
        "conversao_para_soja",
        "emissao_absoluta_co2",
        "taxa_emissao_co2",
        "ctotal",
        "cveg",
        "classes_brluc"
    ]
}


for grupo, termos in GRUPOS_VARIAVEIS.items():

    colunas_encontradas = [
        coluna
        for coluna in base.columns
        if any(
            termo.lower()
            in
            coluna.lower()
            for termo in termos
        )
    ]


    print(
        "\n" + "=" * 70
    )

    print(
        grupo.upper()
    )

    print("=" * 70)

    print(
        "Quantidade:",
        len(
            colunas_encontradas
        )
    )


    for coluna in colunas_encontradas:

        print(
            "->",
            coluna
        )


PRODUCAO_AGRICOLA
Quantidade: 7
-> area_plantada_ha
-> area_colhida_ha
-> area_nao_colhida_ha
-> aproveitamento_area_pct
-> quantidade_produzida_t
-> rendimento_medio_kg_ha
-> valor_producao_mil_reais

CLIMA
Quantidade: 9
-> precipitacao_anual_mm
-> temperatura_media_anual_c
-> umidade_media_anual_pct
-> origem_precipitacao
-> origem_temperatura
-> origem_umidade
-> qualidade_espacial_precipitacao
-> qualidade_espacial_temperatura
-> qualidade_espacial_umidade

CARBONO_SOLO
Quantidade: 14
-> carbono_solo_t_ha
-> soc_t0_t_c_ha
-> soc_t0_se
-> soc_t0_ic95_inf
-> soc_t0_ic95_sup
-> soc_t0_incerteza_relativa
-> soc_t1_t_c_ha
-> soc_t1_se
-> soc_t1_ic95_inf
-> soc_t1_ic95_sup
-> soc_t1_incerteza_relativa
-> delta_soc_classes_brluc_t1_t0_t_c_ha
-> soc_t0_incerteza_pct
-> soc_t1_incerteza_pct

COBERTURA_TERRA
Quantidade: 10
-> area_cobertura_natural_ha
-> pct_cobertura_natural
-> area_floresta_ha
-> pct_floresta
-> area_pastagem_ha
-> pct_pastagem
-> area_agricultura_ha
-> pct_agricultura
->

In [4]:
# ============================================================
# CLASSIFICAÇÃO DAS VARIÁVEIS POR FONTE REAL
# ============================================================

colunas = list(
    base.columns
)


indice_inicio_seeg = (
    colunas.index(
        "quantidade_biomas_seeg"
    )
)


indice_inicio_brluc = (
    colunas.index(
        "periodo_inicio_brluc"
    )
)


COLUNAS_BASE_07 = (
    colunas[
        :indice_inicio_seeg
    ]
)


COLUNAS_SEEG = (
    colunas[
        indice_inicio_seeg:
        indice_inicio_brluc
    ]
)


COLUNAS_BRLUC = (
    colunas[
        indice_inicio_brluc:
    ]
)


print("=" * 70)
print("CLASSIFICAÇÃO POR FONTE")
print("=" * 70)


print(
    "\nBase anterior — PAM + Solo + INMET + Cobertura:"
)

print(
    len(
        COLUNAS_BASE_07
    )
)


print(
    "\nSEEG:"
)

print(
    len(
        COLUNAS_SEEG
    )
)


print(
    "\nBRLUC:"
)

print(
    len(
        COLUNAS_BRLUC
    )
)


print(
    "\nTotal:"
)

print(
    len(COLUNAS_BASE_07)
    +
    len(COLUNAS_SEEG)
    +
    len(COLUNAS_BRLUC)
)


print(
    "\nTotal original:"
)

print(
    len(
        base.columns
    )
)

CLASSIFICAÇÃO POR FONTE

Base anterior — PAM + Solo + INMET + Cobertura:
45

SEEG:
21

BRLUC:
68

Total:
134

Total original:
134


In [5]:
# ============================================================
# DECOMPOSIÇÃO DA BASE 07
# PAM + SOLO + INMET + COBERTURA
# ============================================================

COLUNAS_IDENTIFICACAO = [
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",
    "ano",
    "cultura"
]


COLUNAS_PAM_ANALITICAS = [
    "area_plantada_ha",
    "area_colhida_ha",
    "area_nao_colhida_ha",
    "aproveitamento_area_pct",
    "quantidade_produzida_t",
    "rendimento_medio_kg_ha",
    "valor_producao_mil_reais"
]


COLUNAS_CLIMA_ANALITICAS = [
    coluna
    for coluna in COLUNAS_BASE_07
    if (
        "precipitacao" in coluna
        or
        "temperatura" in coluna
        or
        "umidade" in coluna
    )
    and
    not coluna.startswith(
        "origem_"
    )
    and
    not coluna.startswith(
        "qualidade_"
    )
]


COLUNAS_CLIMA_METADATA = [
    coluna
    for coluna in COLUNAS_BASE_07
    if (
        coluna.startswith(
            "origem_"
        )
        or
        coluna.startswith(
            "qualidade_espacial_"
        )
    )
]


COLUNAS_SOLO_ANALITICAS = [
    coluna
    for coluna in COLUNAS_BASE_07
    if (
        "carbono_solo" in coluna
    )
]


COLUNAS_COBERTURA_ANALITICAS = [
    coluna
    for coluna in COLUNAS_BASE_07
    if (
        coluna.startswith(
            "area_cobertura_"
        )
        or
        coluna.startswith(
            "pct_cobertura_"
        )
        or
        coluna.startswith(
            "area_floresta"
        )
        or
        coluna.startswith(
            "pct_floresta"
        )
        or
        coluna.startswith(
            "area_pastagem"
        )
        or
        coluna.startswith(
            "pct_pastagem"
        )
        or
        coluna.startswith(
            "area_agricultura"
        )
        or
        coluna.startswith(
            "pct_agricultura"
        )
    )
]


COLUNAS_CLASSIFICADAS_BASE07 = set(
    COLUNAS_IDENTIFICACAO
    +
    COLUNAS_PAM_ANALITICAS
    +
    COLUNAS_CLIMA_ANALITICAS
    +
    COLUNAS_CLIMA_METADATA
    +
    COLUNAS_SOLO_ANALITICAS
    +
    COLUNAS_COBERTURA_ANALITICAS
)


COLUNAS_BASE07_NAO_CLASSIFICADAS = [
    coluna
    for coluna in COLUNAS_BASE_07
    if coluna not in COLUNAS_CLASSIFICADAS_BASE07
]


grupos_base07 = {
    "IDENTIFICAÇÃO":
        COLUNAS_IDENTIFICACAO,

    "PAM — ANALÍTICAS":
        COLUNAS_PAM_ANALITICAS,

    "MAPBIOMAS SOLO — ANALÍTICAS":
        COLUNAS_SOLO_ANALITICAS,

    "INMET — ANALÍTICAS":
        COLUNAS_CLIMA_ANALITICAS,

    "INMET — METADADOS/QUALIDADE":
        COLUNAS_CLIMA_METADATA,

    "MAPBIOMAS COBERTURA — ANALÍTICAS":
        COLUNAS_COBERTURA_ANALITICAS,

    "AINDA NÃO CLASSIFICADAS":
        COLUNAS_BASE07_NAO_CLASSIFICADAS
}


for grupo, lista in grupos_base07.items():

    print(
        "\n" + "=" * 70
    )

    print(
        grupo
    )

    print("=" * 70)

    print(
        "Quantidade:",
        len(lista)
    )


    for coluna in lista:

        print(
            "->",
            coluna
        )


IDENTIFICAÇÃO
Quantidade: 6
-> codigo_ibge
-> municipio
-> uf
-> regiao
-> ano
-> cultura

PAM — ANALÍTICAS
Quantidade: 7
-> area_plantada_ha
-> area_colhida_ha
-> area_nao_colhida_ha
-> aproveitamento_area_pct
-> quantidade_produzida_t
-> rendimento_medio_kg_ha
-> valor_producao_mil_reais

MAPBIOMAS SOLO — ANALÍTICAS
Quantidade: 1
-> carbono_solo_t_ha

INMET — ANALÍTICAS
Quantidade: 3
-> precipitacao_anual_mm
-> temperatura_media_anual_c
-> umidade_media_anual_pct

INMET — METADADOS/QUALIDADE
Quantidade: 6
-> origem_precipitacao
-> origem_temperatura
-> origem_umidade
-> qualidade_espacial_precipitacao
-> qualidade_espacial_temperatura
-> qualidade_espacial_umidade

MAPBIOMAS COBERTURA — ANALÍTICAS
Quantidade: 8
-> area_cobertura_natural_ha
-> pct_cobertura_natural
-> area_floresta_ha
-> pct_floresta
-> area_pastagem_ha
-> pct_pastagem
-> area_agricultura_ha
-> pct_agricultura

AINDA NÃO CLASSIFICADAS
Quantidade: 14
-> area_km2
-> status_dado
-> tipo_representacao_climatica
-> score_

In [6]:
# ============================================================
# INDICADORES CANDIDATOS — PRIMEIRA SELEÇÃO
# ============================================================

indicadores_candidatos = pd.DataFrame(
    [
        # ----------------------------------------------------
        # PRODUÇÃO
        # ----------------------------------------------------
        {
            "tema": "Produção",
            "variavel": "area_colhida_ha",
            "temporalidade": "anual",
            "papel": "escala produtiva"
        },

        {
            "tema": "Produção",
            "variavel": "quantidade_produzida_t",
            "temporalidade": "anual",
            "papel": "volume de produção"
        },

        {
            "tema": "Produção",
            "variavel": "rendimento_medio_kg_ha",
            "temporalidade": "anual",
            "papel": "eficiência produtiva"
        },

        {
            "tema": "Produção",
            "variavel": "aproveitamento_area_pct",
            "temporalidade": "anual",
            "papel": "aproveitamento da área plantada"
        },

        # ----------------------------------------------------
        # CLIMA
        # ----------------------------------------------------
        {
            "tema": "Clima",
            "variavel": "precipitacao_anual_mm",
            "temporalidade": "anual",
            "papel": "condição hídrica"
        },

        {
            "tema": "Clima",
            "variavel": "temperatura_media_anual_c",
            "temporalidade": "anual",
            "papel": "condição térmica"
        },

        {
            "tema": "Clima",
            "variavel": "umidade_media_anual_pct",
            "temporalidade": "anual",
            "papel": "condição atmosférica"
        },

        # ----------------------------------------------------
        # CARBONO DO SOLO
        # ----------------------------------------------------
        {
            "tema": "Carbono do solo",
            "variavel": "carbono_solo_t_ha",
            "temporalidade": "conforme produto MapBiomas Solo",
            "papel": "estoque de carbono no solo"
        },

        # ----------------------------------------------------
        # COBERTURA
        # ----------------------------------------------------
        {
            "tema": "Cobertura da terra",
            "variavel": "pct_cobertura_natural",
            "temporalidade": "anual",
            "papel": "conservação da cobertura natural"
        },

        {
            "tema": "Cobertura da terra",
            "variavel": "pct_floresta",
            "temporalidade": "anual",
            "papel": "presença de cobertura florestal"
        },

        {
            "tema": "Cobertura da terra",
            "variavel": "pct_pastagem",
            "temporalidade": "anual",
            "papel": "ocupação por pastagem"
        },

        {
            "tema": "Cobertura da terra",
            "variavel": "pct_agricultura",
            "temporalidade": "anual",
            "papel": "ocupação agrícola"
        },

        # ----------------------------------------------------
        # SEEG
        # ----------------------------------------------------
        {
            "tema": "Emissões SEEG",
            "variavel": "emissao_total_co2e_gwp_ar6_t",
            "temporalidade": "anual",
            "papel": "N2O associado a resíduos agrícolas da soja"
        },

        # ----------------------------------------------------
        # BRLUC
        # ----------------------------------------------------
        {
            "tema": "BRLUC",
            "variavel": "percentual_conversao_para_soja_pct",
            "temporalidade": "estática 2000-2019",
            "papel": "conversão de outras classes para soja"
        },

        {
            "tema": "BRLUC",
            "variavel": "area_origem_natural_ha",
            "temporalidade": "estática 2000-2019",
            "papel": "origem natural associada à transição para soja"
        },

        {
            "tema": "BRLUC",
            "variavel": "taxa_emissao_co2_t_ha_ano",
            "temporalidade": "estática 2000-2019",
            "papel": "taxa BRLUC de CO2 por área de referência"
        },

        {
            "tema": "BRLUC",
            "variavel": "emissao_absoluta_co2_t_ano",
            "temporalidade": "estática 2000-2019",
            "papel": "balanço absoluto de CO2 associado à mudança de uso"
        },

        {
            "tema": "BRLUC",
            "variavel": "delta_soc_classes_brluc_t1_t0_t_c_ha",
            "temporalidade": "comparação de classes BRLUC",
            "papel": "diferença de SOC entre as classes associadas a t1 e t0"
        },

        {
            "tema": "BRLUC",
            "variavel": "ctotal_t1_t_c_ha",
            "temporalidade": "classe BRLUC t1",
            "papel": "estoque total de carbono da classe associada a t1"
        }
    ]
)


indicadores_candidatos[
    "existe_na_base"
] = (
    indicadores_candidatos[
        "variavel"
    ]
    .isin(
        base.columns
    )
)


print("=" * 70)
print("INDICADORES CANDIDATOS")
print("=" * 70)


print(
    "\nTodos existem na base:"
)

print(
    indicadores_candidatos[
        "existe_na_base"
    ]
    .all()
)


display(
    indicadores_candidatos
)

INDICADORES CANDIDATOS

Todos existem na base:
True


,tema,variavel,temporalidade,papel,existe_na_base
0,Produção,area_colhida_ha,anual,escala produtiva,True
1,Produção,quantidade_produzida_t,anual,volume de produção,True
2,Produção,rendimento_medio_kg_ha,anual,eficiência produtiva,True
3,Produção,aproveitamento_area_pct,anual,aproveitamento da área plantada,True
4,Clima,precipitacao_anual_mm,anual,condição hídrica,True
5,Clima,temperatura_media_anual_c,anual,condição térmica,True
6,Clima,umidade_media_anual_pct,anual,condição atmosférica,True
7,Carbono do solo,carbono_solo_t_ha,conforme produto MapBiomas Solo,estoque de carbono no solo,True
8,Cobertura da terra,pct_cobertura_natural,anual,conservação da cobertura natural,True
9,Cobertura da terra,pct_floresta,anual,presença de cobertura florestal,True


In [7]:
# ============================================================
# CLASSIFICAÇÃO DEFINITIVA — BASE 07
# ============================================================

COLUNAS_IDENTIFICACAO = [
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",
    "ano",
    "cultura"
]


COLUNAS_TERRITORIAIS = [
    "area_km2"
]


COLUNAS_PAM = [
    "area_plantada_ha",
    "area_colhida_ha",
    "area_nao_colhida_ha",
    "aproveitamento_area_pct",
    "quantidade_produzida_t",
    "rendimento_medio_kg_ha",
    "valor_producao_mil_reais"
]


COLUNAS_MAPBIOMAS_SOLO = [
    "carbono_solo_t_ha"
]


COLUNAS_MAPBIOMAS_SOLO_METADATA = [
    "status_dado"
]


COLUNAS_INMET = [
    "precipitacao_anual_mm",
    "temperatura_media_anual_c",
    "umidade_media_anual_pct"
]


COLUNAS_INMET_METADATA = [
    "origem_precipitacao",
    "origem_temperatura",
    "origem_umidade",
    "qualidade_espacial_precipitacao",
    "qualidade_espacial_temperatura",
    "qualidade_espacial_umidade",
    "tipo_representacao_climatica",
    "score_qualidade_climatica",
    "qualidade_climatica_geral"
]


COLUNAS_MAPBIOMAS_COBERTURA = [
    "area_total_mapeada_ha",

    "area_cobertura_natural_ha",
    "pct_cobertura_natural",

    "area_floresta_ha",
    "pct_floresta",

    "area_pastagem_ha",
    "pct_pastagem",

    "area_agricultura_ha",
    "pct_agricultura",

    "area_agropecuaria_ha",
    "pct_agropecuaria",

    "area_soja_mapbiomas_ha",
    "pct_soja_mapbiomas"
]


COLUNAS_MAPBIOMAS_COBERTURA_METADATA = [
    "quantidade_biomas",
    "biomas_presentes",
    "diferenca_area_abs_pct",
    "faixa_diferenca_area_ibge"
]


GRUPOS_BASE07_DEFINITIVOS = {
    "Identificação":
        COLUNAS_IDENTIFICACAO,

    "Contexto territorial":
        COLUNAS_TERRITORIAIS,

    "IBGE/PAM":
        COLUNAS_PAM,

    "MapBiomas Solo":
        COLUNAS_MAPBIOMAS_SOLO,

    "MapBiomas Solo — metadata":
        COLUNAS_MAPBIOMAS_SOLO_METADATA,

    "INMET":
        COLUNAS_INMET,

    "INMET — metadata/qualidade":
        COLUNAS_INMET_METADATA,

    "MapBiomas Cobertura":
        COLUNAS_MAPBIOMAS_COBERTURA,

    "MapBiomas Cobertura — metadata/qualidade":
        COLUNAS_MAPBIOMAS_COBERTURA_METADATA
}


for grupo, lista in GRUPOS_BASE07_DEFINITIVOS.items():

    print(
        "\n" + "=" * 70
    )

    print(grupo.upper())

    print("=" * 70)

    print(
        "Quantidade:",
        len(lista)
    )

    for coluna in lista:

        print(
            "->",
            coluna
        )


IDENTIFICAÇÃO
Quantidade: 6
-> codigo_ibge
-> municipio
-> uf
-> regiao
-> ano
-> cultura

CONTEXTO TERRITORIAL
Quantidade: 1
-> area_km2

IBGE/PAM
Quantidade: 7
-> area_plantada_ha
-> area_colhida_ha
-> area_nao_colhida_ha
-> aproveitamento_area_pct
-> quantidade_produzida_t
-> rendimento_medio_kg_ha
-> valor_producao_mil_reais

MAPBIOMAS SOLO
Quantidade: 1
-> carbono_solo_t_ha

MAPBIOMAS SOLO — METADATA
Quantidade: 1
-> status_dado

INMET
Quantidade: 3
-> precipitacao_anual_mm
-> temperatura_media_anual_c
-> umidade_media_anual_pct

INMET — METADATA/QUALIDADE
Quantidade: 9
-> origem_precipitacao
-> origem_temperatura
-> origem_umidade
-> qualidade_espacial_precipitacao
-> qualidade_espacial_temperatura
-> qualidade_espacial_umidade
-> tipo_representacao_climatica
-> score_qualidade_climatica
-> qualidade_climatica_geral

MAPBIOMAS COBERTURA
Quantidade: 13
-> area_total_mapeada_ha
-> area_cobertura_natural_ha
-> pct_cobertura_natural
-> area_floresta_ha
-> pct_floresta
-> area_pastag

In [8]:
# ============================================================
# AUDITORIA DA CLASSIFICAÇÃO — BASE 07
# ============================================================

todas_colunas_classificadas = []

for lista in GRUPOS_BASE07_DEFINITIVOS.values():

    todas_colunas_classificadas.extend(
        lista
    )


colunas_classificadas_set = set(
    todas_colunas_classificadas
)


colunas_base07_set = set(
    COLUNAS_BASE_07
)


nao_classificadas = sorted(
    colunas_base07_set
    -
    colunas_classificadas_set
)


classificadas_indevidamente = sorted(
    colunas_classificadas_set
    -
    colunas_base07_set
)


contagem_classificacao = (
    pd.Series(
        todas_colunas_classificadas
    )
    .value_counts()
)


classificadas_mais_de_uma_vez = (
    contagem_classificacao[
        contagem_classificacao
        > 1
    ]
)


print("=" * 70)
print("AUDITORIA DA CLASSIFICAÇÃO — BASE 07")
print("=" * 70)


print(
    "\nColunas Base 07:",
    len(
        COLUNAS_BASE_07
    )
)


print(
    "Colunas classificadas:",
    len(
        todas_colunas_classificadas
    )
)


print(
    "Colunas únicas classificadas:",
    len(
        colunas_classificadas_set
    )
)


print(
    "\nNão classificadas:"
)

print(
    nao_classificadas
)


print(
    "\nClassificadas mas inexistentes na Base 07:"
)

print(
    classificadas_indevidamente
)


print(
    "\nClassificadas mais de uma vez:"
)

if len(
    classificadas_mais_de_uma_vez
) == 0:

    print(
        "Nenhuma"
    )

else:

    display(
        classificadas_mais_de_uma_vez
    )

AUDITORIA DA CLASSIFICAÇÃO — BASE 07

Colunas Base 07: 45
Colunas classificadas: 45
Colunas únicas classificadas: 45

Não classificadas:
[]

Classificadas mas inexistentes na Base 07:
[]

Classificadas mais de uma vez:
Nenhuma


In [9]:
# ============================================================
# COMPORTAMENTO TEMPORAL DAS VARIÁVEIS
# ============================================================

VARIAVEIS_CONTROLE_TEMPORAL = [
    # Produção
    "area_colhida_ha",
    "quantidade_produzida_t",
    "rendimento_medio_kg_ha",

    # Clima
    "precipitacao_anual_mm",
    "temperatura_media_anual_c",
    "umidade_media_anual_pct",

    # Solo
    "carbono_solo_t_ha",

    # Cobertura
    "pct_cobertura_natural",
    "pct_agricultura",
    "pct_soja_mapbiomas",

    # SEEG
    "emissao_total_co2e_gwp_ar6_t",

    # BRLUC
    "percentual_conversao_para_soja_pct",
    "taxa_emissao_co2_t_ha_ano",
    "delta_soc_classes_brluc_t1_t0_t_c_ha"
]


resultados_temporais = []


for coluna in VARIAVEIS_CONTROLE_TEMPORAL:

    valores_distintos_municipio = (
        base
        .groupby(
            "codigo_ibge"
        )[
            coluna
        ]
        .nunique(
            dropna=True
        )
    )


    resultados_temporais.append(
        {
            "variavel":
                coluna,

            "linhas_nao_null":
                base[
                    coluna
                ]
                .notna()
                .sum(),

            "municipios_com_dado":
                base
                .loc[
                    base[
                        coluna
                    ]
                    .notna(),
                    "codigo_ibge"
                ]
                .nunique(),

            "municipios_com_mais_de_um_valor":
                (
                    valores_distintos_municipio
                    > 1
                )
                .sum(),

            "max_valores_distintos_por_municipio":
                valores_distintos_municipio
                .max()
        }
    )


auditoria_temporal = pd.DataFrame(
    resultados_temporais
)


print("=" * 70)
print("COMPORTAMENTO TEMPORAL")
print("=" * 70)


display(
    auditoria_temporal
)

COMPORTAMENTO TEMPORAL


,variavel,linhas_nao_null,municipios_com_dado,municipios_com_mais_de_um_valor,max_valores_distintos_por_municipio
0,area_colhida_ha,8669,1504,1439,6
1,quantidade_produzida_t,8669,1504,1478,6
2,rendimento_medio_kg_ha,8669,1504,1473,6
3,precipitacao_anual_mm,8674,1505,1481,6
4,temperatura_media_anual_c,8674,1505,1481,6
5,umidade_media_anual_pct,8674,1505,1481,6
6,carbono_solo_t_ha,8674,1505,1481,6
7,pct_cobertura_natural,8674,1505,1481,6
8,pct_agricultura,8674,1505,1479,6
9,pct_soja_mapbiomas,8674,1505,1468,6


In [10]:
# ============================================================
# AUDITORIA DOS DENOMINADORES DOS INDICADORES
# ============================================================

VARIAVEIS_DENOMINADOR = [
    "area_plantada_ha",
    "area_colhida_ha",
    "quantidade_produzida_t"
]


print("=" * 70)
print("AUDITORIA DOS DENOMINADORES")
print("=" * 70)


for coluna in VARIAVEIS_DENOMINADOR:

    serie = base[
        coluna
    ]

    print(
        f"\n{coluna}"
    )

    print(
        "NULL:",
        serie
        .isna()
        .sum()
    )

    print(
        "Zero:",
        (
            serie
            .eq(0)
        )
        .sum()
    )

    print(
        "Negativos:",
        (
            serie
            < 0
        )
        .sum()
    )

    print(
        "Mínimo não NULL:",
        serie
        .min()
    )

    print(
        "Máximo:",
        serie
        .max()
    )

AUDITORIA DOS DENOMINADORES

area_plantada_ha
NULL: 0
Zero: 0
Negativos: 0
Mínimo não NULL: 1.0
Máximo: 605000.0

area_colhida_ha
NULL: 5
Zero: 0
Negativos: 0
Mínimo não NULL: 1.0
Máximo: 605000.0

quantidade_produzida_t
NULL: 5
Zero: 0
Negativos: 0
Mínimo não NULL: 2.0
Máximo: 2283300.0


In [11]:
# ============================================================
# PRIMEIROS INDICADORES DERIVADOS ANUAIS
# ============================================================

base_indicadores = (
    base
    .copy()
)


# ------------------------------------------------------------
# Função de divisão segura
# ------------------------------------------------------------

def divisao_segura(
    numerador,
    denominador
):

    resultado = pd.Series(
        np.nan,
        index=numerador.index,
        dtype="float64"
    )

    mascara_valida = (
        numerador.notna()
        &
        denominador.notna()
        &
        denominador.gt(0)
    )

    resultado.loc[
        mascara_valida
    ] = (
        numerador.loc[
            mascara_valida
        ]
        /
        denominador.loc[
            mascara_valida
        ]
    )

    return resultado


# ============================================================
# 1. PRODUTIVIDADE CALCULADA
# ============================================================

base_indicadores[
    "produtividade_calculada_t_ha"
] = divisao_segura(
    base_indicadores[
        "quantidade_produzida_t"
    ],
    base_indicadores[
        "area_colhida_ha"
    ]
)


# ============================================================
# 2. INTENSIDADE SEEG POR TONELADA DE SOJA
#
# Escopo:
# N2O associado aos resíduos agrícolas da soja,
# direto + indireto, representado em CO2e GWP-AR6.
# ============================================================

base_indicadores[
    "intensidade_seeg_residuos_soja_co2e_completa_t_por_t_soja"
] = divisao_segura(
    base_indicadores[
        "emissao_total_co2e_gwp_ar6_t"
    ],
    base_indicadores[
        "quantidade_produzida_t"
    ]
)


base_indicadores[
    "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja"
] = divisao_segura(
    base_indicadores[
        "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
    ],
    base_indicadores[
        "quantidade_produzida_t"
    ]
)


# ============================================================
# 3. INTENSIDADE SEEG POR HECTARE COLHIDO
# ============================================================

base_indicadores[
    "intensidade_seeg_residuos_soja_co2e_completa_t_por_ha_colhido"
] = divisao_segura(
    base_indicadores[
        "emissao_total_co2e_gwp_ar6_t"
    ],
    base_indicadores[
        "area_colhida_ha"
    ]
)


base_indicadores[
    "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_ha_colhido"
] = divisao_segura(
    base_indicadores[
        "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
    ],
    base_indicadores[
        "area_colhida_ha"
    ]
)


print("=" * 70)
print("INDICADORES DERIVADOS CRIADOS")
print("=" * 70)


NOVOS_INDICADORES = [
    "produtividade_calculada_t_ha",

    "intensidade_seeg_residuos_soja_co2e_completa_t_por_t_soja",
    "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja",

    "intensidade_seeg_residuos_soja_co2e_completa_t_por_ha_colhido",
    "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_ha_colhido"
]


for coluna in NOVOS_INDICADORES:

    print(
        "\n",
        coluna
    )

    print(
        "NULL:",
        base_indicadores[
            coluna
        ]
        .isna()
        .sum()
    )

INDICADORES DERIVADOS CRIADOS

 produtividade_calculada_t_ha
NULL: 5

 intensidade_seeg_residuos_soja_co2e_completa_t_por_t_soja
NULL: 20

 intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja
NULL: 5

 intensidade_seeg_residuos_soja_co2e_completa_t_por_ha_colhido
NULL: 20

 intensidade_seeg_residuos_soja_co2e_disponivel_t_por_ha_colhido
NULL: 5


In [12]:
# ============================================================
# VALIDAÇÃO DOS INDICADORES DERIVADOS
# ============================================================

# ------------------------------------------------------------
# Rendimento calculado em kg/ha
# ------------------------------------------------------------

base_indicadores[
    "rendimento_calculado_kg_ha"
] = (
    base_indicadores[
        "produtividade_calculada_t_ha"
    ]
    *
    1000
)


diferenca_rendimento = (
    base_indicadores[
        "rendimento_calculado_kg_ha"
    ]
    -
    base_indicadores[
        "rendimento_medio_kg_ha"
    ]
).abs()


print("=" * 70)
print("VALIDAÇÃO DOS INDICADORES")
print("=" * 70)


print(
    "\nRendimento PAM × rendimento calculado"
)


print(
    "Registros comparáveis:",
    diferenca_rendimento
    .notna()
    .sum()
)


print(
    "Diferença média absoluta:",
    diferenca_rendimento
    .mean()
)


print(
    "Diferença mediana absoluta:",
    diferenca_rendimento
    .median()
)


print(
    "Maior diferença absoluta:",
    diferenca_rendimento
    .max()
)


print(
    "\nDiferenças > 1 kg/ha:"
)

print(
    (
        diferenca_rendimento
        > 1
    )
    .sum()
)


print(
    "\nDiferenças > 10 kg/ha:"
)

print(
    (
        diferenca_rendimento
        > 10
    )
    .sum()
)


# ------------------------------------------------------------
# Valores negativos
# ------------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "VALORES NEGATIVOS"
)

print("=" * 70)


for coluna in NOVOS_INDICADORES:

    print(
        coluna,
        "->",
        (
            base_indicadores[
                coluna
            ]
            < 0
        )
        .sum()
    )


# ------------------------------------------------------------
# Registros SEEG parciais
# ------------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "REGISTROS SEEG PARCIAIS"
)

print("=" * 70)


display(
    base_indicadores.loc[
        base_indicadores[
            "status_dado_seeg"
        ]
        .eq(
            "parcial_nao_captado"
        ),
        [
            "codigo_ibge",
            "municipio",
            "ano",

            "quantidade_produzida_t",
            "area_colhida_ha",

            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
            "emissao_total_co2e_gwp_ar6_t",

            "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja",
            "intensidade_seeg_residuos_soja_co2e_completa_t_por_t_soja"
        ]
    ]
)

VALIDAÇÃO DOS INDICADORES

Rendimento PAM × rendimento calculado
Registros comparáveis: 8669
Diferença média absoluta: 0.0652971073658508
Diferença mediana absoluta: 0.0
Maior diferença absoluta: 0.5000000000004547

Diferenças > 1 kg/ha:
0

Diferenças > 10 kg/ha:
0

VALORES NEGATIVOS
produtividade_calculada_t_ha -> 0
intensidade_seeg_residuos_soja_co2e_completa_t_por_t_soja -> 0
intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja -> 0
intensidade_seeg_residuos_soja_co2e_completa_t_por_ha_colhido -> 0
intensidade_seeg_residuos_soja_co2e_disponivel_t_por_ha_colhido -> 0

REGISTROS SEEG PARCIAIS


,codigo_ibge,municipio,ano,quantidade_produzida_t,area_colhida_ha,emissao_total_co2e_gwp_ar6_soma_disponivel_t,emissao_total_co2e_gwp_ar6_t,intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja,intensidade_seeg_residuos_soja_co2e_completa_t_por_t_soja
5340,4305132,Cerro Branco,2019,3650.0,1170.0,346.61,NaN,0.094962,NaN
5341,4305132,Cerro Branco,2020,2160.0,1200.0,205.12,NaN,0.094963,NaN
5342,4305132,Cerro Branco,2021,4290.0,1300.0,407.39,NaN,0.094963,NaN
6513,4314803,Portão,2019,480.0,160.0,45.58,NaN,0.094958,NaN
6514,4314803,Portão,2020,486.0,450.0,46.16,NaN,0.094979,NaN
6515,4314803,Portão,2021,720.0,450.0,68.37,NaN,0.094958,NaN
6516,4314803,Portão,2022,1485.0,450.0,141.02,NaN,0.094963,NaN
6517,4314803,Portão,2023,990.0,450.0,94.02,NaN,0.094970,NaN
6518,4314803,Portão,2024,1118.0,447.0,106.17,NaN,0.094964,NaN
7315,4322608,Venâncio Aires,2019,13500.0,3750.0,1282.00,NaN,0.094963,NaN


In [13]:
# ============================================================
# DIAGNÓSTICO DAS INTENSIDADES SEEG
# ============================================================

INDICADORES_INTENSIDADE_SEEG = [
    "intensidade_seeg_residuos_soja_co2e_completa_t_por_t_soja",
    "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja",

    "intensidade_seeg_residuos_soja_co2e_completa_t_por_ha_colhido",
    "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_ha_colhido"
]


resultados_distribuicao = []


for coluna in INDICADORES_INTENSIDADE_SEEG:

    serie = (
        base_indicadores[
            coluna
        ]
        .dropna()
    )

    media = (
        serie
        .mean()
    )

    desvio = (
        serie
        .std()
    )

    cv = (
        desvio
        /
        media
        *
        100
        if media != 0
        else np.nan
    )

    resultados_distribuicao.append(
        {
            "variavel":
                coluna,

            "n":
                len(
                    serie
                ),

            "media":
                media,

            "desvio_padrao":
                desvio,

            "cv_pct":
                cv,

            "minimo":
                serie.min(),

            "p25":
                serie.quantile(
                    0.25
                ),

            "mediana":
                serie.median(),

            "p75":
                serie.quantile(
                    0.75
                ),

            "maximo":
                serie.max(),

            "valores_distintos_6_decimais":
                serie
                .round(6)
                .nunique()
        }
    )


diagnostico_intensidades = pd.DataFrame(
    resultados_distribuicao
)


print("=" * 70)
print("DISTRIBUIÇÃO DAS INTENSIDADES SEEG")
print("=" * 70)


display(
    diagnostico_intensidades
)

DISTRIBUIÇÃO DAS INTENSIDADES SEEG


,variavel,n,media,desvio_padrao,cv_pct,minimo,p25,mediana,p75,maximo,valores_distintos_6_decimais
0,intensidade_seeg_residuos_soja_co2e_completa_t...,8654,0.094967,0.000365,0.384807,0.093333,0.094963,0.094963,0.094963,0.128838,84
1,intensidade_seeg_residuos_soja_co2e_disponivel...,8669,0.094967,0.000365,0.384474,0.093333,0.094963,0.094963,0.094963,0.128838,84
2,intensidade_seeg_residuos_soja_co2e_completa_t...,8654,0.285415,0.078942,27.658643,0.010256,0.246904,0.306256,0.341866,0.475000,3775
3,intensidade_seeg_residuos_soja_co2e_disponivel...,8669,0.285338,0.078950,27.669052,0.010256,0.246904,0.306072,0.341866,0.475000,3780


In [14]:
# ============================================================
# RELAÇÃO SEEG × PRODUÇÃO DE SOJA
# ============================================================

controle_seeg_producao = (
    base_indicadores[
        [
            "quantidade_produzida_t",
            "area_colhida_ha",
            "rendimento_medio_kg_ha",
            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
            "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja",
            "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_ha_colhido"
        ]
    ]
    .dropna()
)


print("=" * 70)
print("RELAÇÃO SEEG × PRODUÇÃO")
print("=" * 70)


print(
    "\nCorrelação emissão disponível × produção:"
)

print(
    controle_seeg_producao[
        [
            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
            "quantidade_produzida_t"
        ]
    ]
    .corr()
    .iloc[
        0,
        1
    ]
)


print(
    "\nCorrelação intensidade por tonelada × produção:"
)

print(
    controle_seeg_producao[
        [
            "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja",
            "quantidade_produzida_t"
        ]
    ]
    .corr()
    .iloc[
        0,
        1
    ]
)


print(
    "\nCorrelação intensidade por hectare × rendimento:"
)

print(
    controle_seeg_producao[
        [
            "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_ha_colhido",
            "rendimento_medio_kg_ha"
        ]
    ]
    .corr()
    .iloc[
        0,
        1
    ]
)

RELAÇÃO SEEG × PRODUÇÃO

Correlação emissão disponível × produção:
0.999998714748127

Correlação intensidade por tonelada × produção:
-0.00041905761176290154

Correlação intensidade por hectare × rendimento:
0.9999382747625652


In [15]:
# ============================================================
# AUDITORIA DE REDUNDÂNCIA DOS INDICADORES
# ============================================================

# ------------------------------------------------------------
# 1. Produtividade calculada × rendimento oficial PAM
# ------------------------------------------------------------

diferenca_produtividade = (
    base_indicadores[
        "produtividade_calculada_t_ha"
    ]
    -
    (
        base_indicadores[
            "rendimento_medio_kg_ha"
        ]
        /
        1000
    )
).abs()


# ------------------------------------------------------------
# 2. Intensidade por hectare
#
# intensidade CO2e / ha
# =
# intensidade CO2e / t
# ×
# produtividade t / ha
# ------------------------------------------------------------

intensidade_ha_reconstruida = (
    base_indicadores[
        "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja"
    ]
    *
    base_indicadores[
        "produtividade_calculada_t_ha"
    ]
)


diferenca_intensidade_ha = (
    base_indicadores[
        "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_ha_colhido"
    ]
    -
    intensidade_ha_reconstruida
).abs()


print("=" * 70)
print("AUDITORIA DE REDUNDÂNCIA")
print("=" * 70)


print(
    "\nProdutividade calculada × rendimento PAM"
)


print(
    "Maior diferença em t/ha:"
)

print(
    diferenca_produtividade
    .max()
)


print(
    "Diferenças > 0.001 t/ha:"
)

print(
    (
        diferenca_produtividade
        > 0.001
    )
    .sum()
)


print(
    "\nIntensidade por hectare reconstruída"
)


print(
    "Maior diferença:"
)

print(
    diferenca_intensidade_ha
    .max()
)


print(
    "Diferenças > 0.000001:"
)

print(
    (
        diferenca_intensidade_ha
        > 0.000001
    )
    .sum()
)

AUDITORIA DE REDUNDÂNCIA

Produtividade calculada × rendimento PAM
Maior diferença em t/ha:
0.0005000000000006111
Diferenças > 0.001 t/ha:
0

Intensidade por hectare reconstruída
Maior diferença:
1.1102230246251565e-16
Diferenças > 0.000001:
0


In [16]:
# ============================================================
# EXTREMOS DA INTENSIDADE SEEG POR TONELADA
# ============================================================

COLUNA_INTENSIDADE_T = (
    "intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja"
)


colunas_diagnostico = [
    "codigo_ibge",
    "municipio",
    "uf",
    "ano",
    "quantidade_produzida_t",
    "area_colhida_ha",
    "rendimento_medio_kg_ha",
    "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
    "status_dado_seeg",
    "quantidade_biomas_seeg",
    "biomas_presentes_seeg",
    COLUNA_INTENSIDADE_T
]


print("=" * 70)
print("MENORES INTENSIDADES")
print("=" * 70)


display(
    base_indicadores[
        colunas_diagnostico
    ]
    .dropna(
        subset=[
            COLUNA_INTENSIDADE_T
        ]
    )
    .sort_values(
        by=COLUNA_INTENSIDADE_T,
        ascending=True
    )
    .head(15)
)


print(
    "\n" + "=" * 70
)

print(
    "MAIORES INTENSIDADES"
)

print("=" * 70)


display(
    base_indicadores[
        colunas_diagnostico
    ]
    .dropna(
        subset=[
            COLUNA_INTENSIDADE_T
        ]
    )
    .sort_values(
        by=COLUNA_INTENSIDADE_T,
        ascending=False
    )
    .head(15)
)

MENORES INTENSIDADES


,codigo_ibge,municipio,uf,ano,quantidade_produzida_t,area_colhida_ha,rendimento_medio_kg_ha,emissao_total_co2e_gwp_ar6_soma_disponivel_t,status_dado_seeg,quantidade_biomas_seeg,biomas_presentes_seeg,intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja
7200,4321709,Três Coroas,RS,2020,3.0,1.0,3000.0,0.28,completo,1,Mata Atlântica,0.093333
4740,4127882,Tunas do Paraná,PR,2023,7.0,2.0,3500.0,0.66,completo,1,Mata Atlântica,0.094286
7057,4320453,Sério,RS,2020,11.0,6.0,1833.0,1.04,completo,1,Mata Atlântica,0.094545
5284,4304697,Capitão,RS,2022,11.0,5.0,2200.0,1.04,completo,1,Mata Atlântica,0.094545
7007,4320008,Sapucaia do Sul,RS,2024,26.0,15.0,1733.0,2.46,completo,2,Mata Atlântica | Pampa,0.094615
5888,4310108,Igrejinha,RS,2021,19.0,7.0,2714.0,1.80,completo,1,Mata Atlântica,0.094737
5886,4310108,Igrejinha,RS,2019,19.0,7.0,2714.0,1.80,completo,1,Mata Atlântica,0.094737
7005,4320008,Sapucaia do Sul,RS,2022,54.0,30.0,1800.0,5.12,completo,2,Mata Atlântica | Pampa,0.094815
8544,4218103,Timbé do Sul,SC,2024,27.0,15.0,1800.0,2.56,completo,1,Mata Atlântica,0.094815
954,5216908,Pilar de Goiás,GO,2022,31.0,12.0,2583.0,2.94,completo,1,Cerrado,0.094839



MAIORES INTENSIDADES


,codigo_ibge,municipio,uf,ano,quantidade_produzida_t,area_colhida_ha,rendimento_medio_kg_ha,emissao_total_co2e_gwp_ar6_soma_disponivel_t,status_dado_seeg,quantidade_biomas_seeg,biomas_presentes_seeg,intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja
5646,4307401,Esmeralda,RS,2020,68400.0,28500.0,2400.0,8812.54,completo,1,Mata Atlântica,0.128838
6654,4316006,Rolante,RS,2024,5.0,2.0,2500.0,0.48,completo,1,Mata Atlântica,0.096000
6651,4316006,Rolante,RS,2021,5.0,2.0,2500.0,0.48,completo,1,Mata Atlântica,0.096000
7059,4320453,Sério,RS,2024,5.0,2.0,2500.0,0.48,completo,1,Mata Atlântica,0.096000
5891,4310108,Igrejinha,RS,2024,13.0,16.0,813.0,1.24,completo,1,Mata Atlântica,0.095385
7126,4321204,Taquara,RS,2019,13.0,5.0,2600.0,1.24,completo,1,Mata Atlântica,0.095385
7056,4320453,Sério,RS,2019,21.0,6.0,3500.0,2.00,completo,1,Mata Atlântica,0.095238
7058,4320453,Sério,RS,2021,21.0,6.0,3500.0,2.00,completo,1,Mata Atlântica,0.095238
5394,4305447,Chuvisca,RS,2020,25.0,20.0,1250.0,2.38,completo,1,Pampa,0.095200
6189,4312443,Morrinhos do Sul,RS,2024,25.0,10.0,2500.0,2.38,completo,1,Mata Atlântica,0.095200


In [17]:
# ============================================================
# INTENSIDADE SEEG × ESCALA DE PRODUÇÃO
# ============================================================

FAIXAS_PRODUCAO = [
    ("Todos", 0),
    (">= 100 t", 100),
    (">= 1.000 t", 1000),
    (">= 10.000 t", 10000),
    (">= 100.000 t", 100000)
]


resultados_escala = []


for nome_faixa, limite in FAIXAS_PRODUCAO:

    mascara = (
        base_indicadores[
            "quantidade_produzida_t"
        ]
        .ge(
            limite
        )
        &
        base_indicadores[
            COLUNA_INTENSIDADE_T
        ]
        .notna()
    )


    serie = (
        base_indicadores
        .loc[
            mascara,
            COLUNA_INTENSIDADE_T
        ]
    )


    media = (
        serie.mean()
    )


    desvio = (
        serie.std()
    )


    cv = (
        desvio
        /
        media
        *
        100
        if media != 0
        else np.nan
    )


    resultados_escala.append(
        {
            "faixa":
                nome_faixa,

            "n":
                len(
                    serie
                ),

            "media":
                media,

            "desvio_padrao":
                desvio,

            "cv_pct":
                cv,

            "minimo":
                serie.min(),

            "mediana":
                serie.median(),

            "maximo":
                serie.max()
        }
    )


diagnostico_intensidade_por_escala = pd.DataFrame(
    resultados_escala
)


print("=" * 70)
print("INTENSIDADE SEEG POR ESCALA DE PRODUÇÃO")
print("=" * 70)


display(
    diagnostico_intensidade_por_escala
)

INTENSIDADE SEEG POR ESCALA DE PRODUÇÃO


,faixa,n,media,desvio_padrao,cv_pct,minimo,mediana,maximo
0,Todos,8669,0.094967,3.651236e-04,0.384474,0.093333,0.094963,0.128838
1,>= 100 t,8530,0.094967,3.668102e-04,0.386252,0.094900,0.094963,0.128838
2,>= 1.000 t,7787,0.094967,3.838863e-04,0.404231,0.094954,0.094963,0.128838
3,>= 10.000 t,5470,0.094969,4.580286e-04,0.482293,0.094962,0.094963,0.128838
4,>= 100.000 t,1366,0.094963,2.755481e-08,0.000029,0.094963,0.094963,0.094963


In [18]:
# ============================================================
# DESVIO RELATIVO DA INTENSIDADE SEEG
# ============================================================

mediana_intensidade_seeg = (
    base_indicadores[
        COLUNA_INTENSIDADE_T
    ]
    .median()
)


base_indicadores[
    "desvio_relativo_intensidade_seeg_pct"
] = (
    (
        base_indicadores[
            COLUNA_INTENSIDADE_T
        ]
        -
        mediana_intensidade_seeg
    )
    .abs()
    /
    mediana_intensidade_seeg
    *
    100
)


print("=" * 70)
print("DESVIO DA INTENSIDADE EM RELAÇÃO À MEDIANA")
print("=" * 70)


print(
    "\nMediana:"
)

print(
    mediana_intensidade_seeg
)


for limite in [
    0.01,
    0.1,
    1,
    5,
    10
]:

    quantidade = (
        base_indicadores[
            "desvio_relativo_intensidade_seeg_pct"
        ]
        >
        limite
    ).sum()


    percentual = (
        quantidade
        /
        base_indicadores[
            COLUNA_INTENSIDADE_T
        ]
        .notna()
        .sum()
        *
        100
    )


    print(
        f"\nDesvio > {limite}%:"
    )

    print(
        "Registros:",
        quantidade
    )

    print(
        "Percentual:",
        percentual
    )

DESVIO DA INTENSIDADE EM RELAÇÃO À MEDIANA

Mediana:
0.09496275805119736

Desvio > 0.01%:
Registros: 348
Percentual: 4.014303841273503

Desvio > 0.1%:
Registros: 29
Percentual: 0.33452532010612523

Desvio > 1%:
Registros: 5
Percentual: 0.05767677932864229

Desvio > 5%:
Registros: 1
Percentual: 0.011535355865728458

Desvio > 10%:
Registros: 1
Percentual: 0.011535355865728458


In [19]:
# ============================================================
# OUTLIERS DA INTENSIDADE SEEG
# ============================================================

outliers_intensidade_seeg = (
    base_indicadores[
        base_indicadores[
            "desvio_relativo_intensidade_seeg_pct"
        ]
        .gt(
            1
        )
    ]
    [
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "ano",

            "quantidade_produzida_t",
            "area_colhida_ha",
            "rendimento_medio_kg_ha",

            "emissao_direta_co2e_gwp_ar6_soma_disponivel_t",
            "emissao_indireta_co2e_gwp_ar6_soma_disponivel_t",
            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",

            "emissao_direta_n2o_soma_disponivel_t",
            "emissao_indireta_n2o_soma_disponivel_t",
            "emissao_total_n2o_soma_disponivel_t",

            "status_dado_seeg",
            "quantidade_biomas_seeg",
            "biomas_presentes_seeg",

            COLUNA_INTENSIDADE_T,
            "desvio_relativo_intensidade_seeg_pct"
        ]
    ]
    .sort_values(
        by="desvio_relativo_intensidade_seeg_pct",
        ascending=False
    )
)


print("=" * 70)
print("OUTLIERS SEEG — DESVIO > 1%")
print("=" * 70)


print(
    "\nQuantidade:",
    len(
        outliers_intensidade_seeg
    )
)


display(
    outliers_intensidade_seeg
)

OUTLIERS SEEG — DESVIO > 1%

Quantidade: 5


,codigo_ibge,municipio,uf,ano,quantidade_produzida_t,area_colhida_ha,rendimento_medio_kg_ha,emissao_direta_co2e_gwp_ar6_soma_disponivel_t,emissao_indireta_co2e_gwp_ar6_soma_disponivel_t,emissao_total_co2e_gwp_ar6_soma_disponivel_t,emissao_direta_n2o_soma_disponivel_t,emissao_indireta_n2o_soma_disponivel_t,emissao_total_n2o_soma_disponivel_t,status_dado_seeg,quantidade_biomas_seeg,biomas_presentes_seeg,intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja,desvio_relativo_intensidade_seeg_pct
5646,4307401,Esmeralda,RS,2020,68400.0,28500.0,2400.0,7193.91,1618.63,8812.54,26.35,5.93,32.28,completo,1,Mata Atlântica,0.128838,35.672454
7200,4321709,Três Coroas,RS,2020,3.0,1.0,3000.0,0.23,0.05,0.28,0.00,0.00,0.00,completo,1,Mata Atlântica,0.093333,1.715857
6651,4316006,Rolante,RS,2021,5.0,2.0,2500.0,0.39,0.09,0.48,0.00,0.00,0.00,completo,1,Mata Atlântica,0.096000,1.092262
6654,4316006,Rolante,RS,2024,5.0,2.0,2500.0,0.39,0.09,0.48,0.00,0.00,0.00,completo,1,Mata Atlântica,0.096000,1.092262
7059,4320453,Sério,RS,2024,5.0,2.0,2500.0,0.39,0.09,0.48,0.00,0.00,0.00,completo,1,Mata Atlântica,0.096000,1.092262


In [20]:
# ============================================================
# HISTÓRICO DOS MUNICÍPIOS COM OUTLIERS SEEG
# ============================================================

CODIGOS_OUTLIERS_SEEG = (
    outliers_intensidade_seeg[
        "codigo_ibge"
    ]
    .unique()
)


historico_outliers_seeg = (
    base_indicadores[
        base_indicadores[
            "codigo_ibge"
        ]
        .isin(
            CODIGOS_OUTLIERS_SEEG
        )
    ]
    [
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "ano",

            "quantidade_produzida_t",

            "emissao_direta_co2e_gwp_ar6_soma_disponivel_t",
            "emissao_indireta_co2e_gwp_ar6_soma_disponivel_t",
            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",

            "status_dado_seeg",

            COLUNA_INTENSIDADE_T,
            "desvio_relativo_intensidade_seeg_pct"
        ]
    ]
    .sort_values(
        by=[
            "uf",
            "municipio",
            "ano"
        ]
    )
)


print("=" * 70)
print("HISTÓRICO DOS MUNICÍPIOS COM OUTLIERS")
print("=" * 70)


display(
    historico_outliers_seeg
)

HISTÓRICO DOS MUNICÍPIOS COM OUTLIERS


,codigo_ibge,municipio,uf,ano,quantidade_produzida_t,emissao_direta_co2e_gwp_ar6_soma_disponivel_t,emissao_indireta_co2e_gwp_ar6_soma_disponivel_t,emissao_total_co2e_gwp_ar6_soma_disponivel_t,status_dado_seeg,intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja,desvio_relativo_intensidade_seeg_pct
5645,4307401,Esmeralda,RS,2019,102600.0,7953.62,1789.56,9743.18,completo,0.094963,0.000011
5646,4307401,Esmeralda,RS,2020,68400.0,7193.91,1618.63,8812.54,completo,0.128838,35.672454
5647,4307401,Esmeralda,RS,2021,95700.0,7418.72,1669.21,9087.93,completo,0.094963,0.000065
5648,4307401,Esmeralda,RS,2022,54840.0,4251.23,956.53,5207.76,completo,0.094963,0.000045
5649,4307401,Esmeralda,RS,2023,108000.0,8372.23,1883.75,10255.98,completo,0.094963,0.000021
5650,4307401,Esmeralda,RS,2024,115200.0,8930.38,2009.33,10939.71,completo,0.094963,0.000002
6651,4316006,Rolante,RS,2021,5.0,0.39,0.09,0.48,completo,0.096000,1.092262
6652,4316006,Rolante,RS,2022,10.0,0.78,0.17,0.95,completo,0.095000,0.039217
6653,4316006,Rolante,RS,2023,10.0,0.78,0.17,0.95,completo,0.095000,0.039217
6654,4316006,Rolante,RS,2024,5.0,0.39,0.09,0.48,completo,0.096000,1.092262


In [21]:
# ============================================================
# RESÍDUO EM RELAÇÃO AO PADRÃO SEEG DA BASE
# ============================================================

FATOR_MEDIANO_SEEG = (
    mediana_intensidade_seeg
)


base_indicadores[
    "emissao_seeg_esperada_pelo_fator_mediano_t"
] = (
    base_indicadores[
        "quantidade_produzida_t"
    ]
    *
    FATOR_MEDIANO_SEEG
)


base_indicadores[
    "residuo_emissao_seeg_vs_fator_mediano_t"
] = (
    base_indicadores[
        "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
    ]
    -
    base_indicadores[
        "emissao_seeg_esperada_pelo_fator_mediano_t"
    ]
)


base_indicadores[
    "residuo_emissao_seeg_vs_fator_mediano_pct"
] = (
    base_indicadores[
        "residuo_emissao_seeg_vs_fator_mediano_t"
    ]
    /
    base_indicadores[
        "emissao_seeg_esperada_pelo_fator_mediano_t"
    ]
    *
    100
)


print("=" * 70)
print("RESÍDUOS SEEG × PADRÃO MEDIANO")
print("=" * 70)


display(
    base_indicadores.loc[
        base_indicadores[
            "desvio_relativo_intensidade_seeg_pct"
        ]
        .gt(
            1
        ),
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "ano",

            "quantidade_produzida_t",

            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
            "emissao_seeg_esperada_pelo_fator_mediano_t",

            "residuo_emissao_seeg_vs_fator_mediano_t",
            "residuo_emissao_seeg_vs_fator_mediano_pct"
        ]
    ]
    .sort_values(
        by="residuo_emissao_seeg_vs_fator_mediano_pct",
        key=lambda serie: serie.abs(),
        ascending=False
    )
)

RESÍDUOS SEEG × PADRÃO MEDIANO


,codigo_ibge,municipio,uf,ano,quantidade_produzida_t,emissao_total_co2e_gwp_ar6_soma_disponivel_t,emissao_seeg_esperada_pelo_fator_mediano_t,residuo_emissao_seeg_vs_fator_mediano_t,residuo_emissao_seeg_vs_fator_mediano_pct
5646,4307401,Esmeralda,RS,2020,68400.0,8812.54,6495.452651,2317.087349,35.672454
7200,4321709,Três Coroas,RS,2020,3.0,0.28,0.284888,-0.004888,-1.715857
6651,4316006,Rolante,RS,2021,5.0,0.48,0.474814,0.005186,1.092262
6654,4316006,Rolante,RS,2024,5.0,0.48,0.474814,0.005186,1.092262
7059,4320453,Sério,RS,2024,5.0,0.48,0.474814,0.005186,1.092262


### Diagnóstico da intensidade de emissões SEEG

A emissão de CO₂e utilizada neste estudo apresenta relação praticamente
proporcional com a quantidade de soja produzida no recorte analisado.

A correlação entre emissão disponível e produção foi de aproximadamente
0,999999.

A intensidade de emissão por tonelada produzida apresentou mediana de
aproximadamente 0,094963 t CO₂e por tonelada de soja e baixa
variabilidade geral.

Por essa razão, a intensidade por tonelada é mantida como indicador
diagnóstico, mas não é considerada uma dimensão independente para a
construção do índice de priorização, evitando redundância com produção
e emissão absoluta.

A intensidade por hectare também não é considerada variável
independente, pois pode ser reconstruída matematicamente pela
intensidade por tonelada multiplicada pela produtividade.

Valores extremos são auditados separadamente e não são substituídos
automaticamente.

In [22]:
# ============================================================
# FLAG DE DIAGNÓSTICO — INTENSIDADE SEEG
# ============================================================

base_indicadores[
    "flag_outlier_intensidade_seeg"
] = (
    base_indicadores[
        "desvio_relativo_intensidade_seeg_pct"
    ]
    .gt(1)
)


base_indicadores[
    "flag_outlier_intensidade_seeg_relevante"
] = (
    base_indicadores[
        "desvio_relativo_intensidade_seeg_pct"
    ]
    .gt(5)
)


print("=" * 70)
print("FLAGS DE DIAGNÓSTICO SEEG")
print("=" * 70)


print(
    "\nDesvio > 1%:"
)

print(
    base_indicadores[
        "flag_outlier_intensidade_seeg"
    ]
    .sum()
)


print(
    "\nDesvio > 5%:"
)

print(
    base_indicadores[
        "flag_outlier_intensidade_seeg_relevante"
    ]
    .sum()
)


print(
    "\nRegistro com desvio > 5%:"
)


display(
    base_indicadores.loc[
        base_indicadores[
            "flag_outlier_intensidade_seeg_relevante"
        ],
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "ano",
            "quantidade_produzida_t",
            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
            COLUNA_INTENSIDADE_T,
            "desvio_relativo_intensidade_seeg_pct"
        ]
    ]
)

FLAGS DE DIAGNÓSTICO SEEG

Desvio > 1%:
5

Desvio > 5%:
1

Registro com desvio > 5%:


,codigo_ibge,municipio,uf,ano,quantidade_produzida_t,emissao_total_co2e_gwp_ar6_soma_disponivel_t,intensidade_seeg_residuos_soja_co2e_disponivel_t_por_t_soja,desvio_relativo_intensidade_seeg_pct
5646,4307401,Esmeralda,RS,2020,68400.0,8812.54,0.128838,35.672454


### Diagnóstico das intensidades de emissão SEEG

A emissão de CO₂e utilizada neste estudo apresenta relação praticamente
proporcional com a produção de soja no recorte analisado.

A correlação entre a emissão disponível e a quantidade produzida foi de
aproximadamente 0,999999.

A intensidade de emissão por tonelada apresentou mediana de
aproximadamente 0,094963 t CO₂e por tonelada de soja e baixa
variabilidade geral.

Apenas 0,33% dos registros apresentaram desvio superior a 0,1% em
relação à mediana, e cinco registros apresentaram desvio superior a 1%.

Quatro desses registros estão associados a volumes extremamente baixos
de produção, nos quais o arredondamento dos valores de emissão aumenta
a variação relativa da razão calculada.

Foi identificado um registro particularmente discrepante para
Esmeralda/RS em 2020, com intensidade aproximadamente 35,67% superior
à mediana do conjunto. Os demais anos do mesmo município permanecem
próximos ao padrão geral.

O valor original foi preservado, sem substituição ou correção
automática, e foi criada uma flag de diagnóstico para identificação
desse comportamento.

Dada a baixa variabilidade da intensidade de emissão por tonelada, essa
métrica é mantida para fins diagnósticos e descritivos, mas não será
utilizada como dimensão independente no índice de priorização.

A intensidade por hectare também não será utilizada como dimensão
independente, uma vez que é matematicamente determinada pela intensidade
por tonelada e pela produtividade agrícola.

As emissões SEEG deste estudo permanecem restritas às emissões de N₂O
associadas aos resíduos agrícolas da soja em solos manejados, diretas e
indiretas, representadas em CO₂e pelo GWP-AR6.

In [23]:
# ============================================================
# COBERTURA TEMPORAL POR MUNICÍPIO
# ============================================================

cobertura_temporal_municipal = (
    base_indicadores
    .groupby(
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao"
        ],
        as_index=False
    )
    .agg(
        quantidade_anos=(
            "ano",
            "nunique"
        ),

        primeiro_ano_disponivel=(
            "ano",
            "min"
        ),

        ultimo_ano_disponivel=(
            "ano",
            "max"
        )
    )
)


print("=" * 70)
print("COBERTURA TEMPORAL MUNICIPAL")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    cobertura_temporal_municipal.shape
)


print(
    "\nMunicípios:"
)

print(
    cobertura_temporal_municipal[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDistribuição da quantidade de anos:"
)

display(
    cobertura_temporal_municipal[
        "quantidade_anos"
    ]
    .value_counts()
    .sort_index()
)

COBERTURA TEMPORAL MUNICIPAL

Dimensão:
(1505, 7)

Municípios:
1505

Distribuição da quantidade de anos:


quantidade_anos
1      24
2      27
3      25
4      15
5      23
6    1391
Name: count, dtype: int64

In [24]:
# ============================================================
# ELEGIBILIDADE PARA INDICADORES TEMPORAIS
# ============================================================

cobertura_temporal_municipal[
    "elegivel_media"
] = (
    cobertura_temporal_municipal[
        "quantidade_anos"
    ]
    .ge(1)
)


cobertura_temporal_municipal[
    "elegivel_volatilidade"
] = (
    cobertura_temporal_municipal[
        "quantidade_anos"
    ]
    .ge(2)
)


cobertura_temporal_municipal[
    "elegivel_tendencia"
] = (
    cobertura_temporal_municipal[
        "quantidade_anos"
    ]
    .ge(3)
)


codigos_com_2019 = set(
    base_indicadores.loc[
        base_indicadores[
            "ano"
        ]
        .eq(2019),
        "codigo_ibge"
    ]
)


codigos_com_2024 = set(
    base_indicadores.loc[
        base_indicadores[
            "ano"
        ]
        .eq(2024),
        "codigo_ibge"
    ]
)


cobertura_temporal_municipal[
    "elegivel_variacao_2019_2024"
] = (
    cobertura_temporal_municipal[
        "codigo_ibge"
    ]
    .isin(
        codigos_com_2019
        &
        codigos_com_2024
    )
)


print("=" * 70)
print("ELEGIBILIDADE TEMPORAL")
print("=" * 70)


for coluna in [
    "elegivel_media",
    "elegivel_volatilidade",
    "elegivel_tendencia",
    "elegivel_variacao_2019_2024"
]:

    print(
        f"\n{coluna}:"
    )

    display(
        cobertura_temporal_municipal[
            coluna
        ]
        .value_counts()
    )

ELEGIBILIDADE TEMPORAL

elegivel_media:


elegivel_media
True    1505
Name: count, dtype: int64


elegivel_volatilidade:


elegivel_volatilidade
True     1481
False      24
Name: count, dtype: int64


elegivel_tendencia:


elegivel_tendencia
True     1454
False      51
Name: count, dtype: int64


elegivel_variacao_2019_2024:


elegivel_variacao_2019_2024
True     1406
False      99
Name: count, dtype: int64

In [25]:
# ============================================================
# FUNÇÃO DE RESUMO TEMPORAL MUNICIPAL
# ============================================================

def resumir_variavel_municipal(
    dataframe,
    coluna,
    prefixo
):

    resultados = []


    for codigo_ibge, grupo in dataframe.groupby(
        "codigo_ibge",
        sort=False
    ):

        dados = (
            grupo[
                [
                    "ano",
                    coluna
                ]
            ]
            .dropna()
            .sort_values(
                "ano"
            )
        )


        n_obs = len(
            dados
        )


        # ----------------------------------------------------
        # Média
        # ----------------------------------------------------

        media = (
            dados[
                coluna
            ]
            .mean()
            if n_obs >= 1
            else np.nan
        )


        # ----------------------------------------------------
        # Desvio-padrão
        # ----------------------------------------------------

        desvio = (
            dados[
                coluna
            ]
            .std(
                ddof=1
            )
            if n_obs >= 2
            else np.nan
        )


        # ----------------------------------------------------
        # Coeficiente de variação
        # ----------------------------------------------------

        if (
            n_obs >= 2
            and
            pd.notna(media)
            and
            media != 0
        ):

            cv_pct = (
                desvio
                /
                abs(media)
                *
                100
            )

        else:

            cv_pct = np.nan


        # ----------------------------------------------------
        # Tendência linear
        #
        # Unidade:
        # unidade original da variável / ano
        # ----------------------------------------------------

        if (
            n_obs >= 3
            and
            dados[
                "ano"
            ]
            .nunique()
            >= 3
        ):

            tendencia = np.polyfit(
                dados[
                    "ano"
                ]
                .astype(
                    float
                ),
                dados[
                    coluna
                ]
                .astype(
                    float
                ),
                1
            )[0]

        else:

            tendencia = np.nan


        # ----------------------------------------------------
        # Valores estritos em 2019 e 2024
        # ----------------------------------------------------

        valor_2019 = (
            dados.loc[
                dados[
                    "ano"
                ]
                .eq(2019),
                coluna
            ]
        )


        valor_2024 = (
            dados.loc[
                dados[
                    "ano"
                ]
                .eq(2024),
                coluna
            ]
        )


        valor_2019 = (
            valor_2019.iloc[0]
            if len(valor_2019) == 1
            else np.nan
        )


        valor_2024 = (
            valor_2024.iloc[0]
            if len(valor_2024) == 1
            else np.nan
        )


        # ----------------------------------------------------
        # Variação 2019 → 2024
        # ----------------------------------------------------

        if (
            pd.notna(
                valor_2019
            )
            and
            pd.notna(
                valor_2024
            )
        ):

            variacao_abs = (
                valor_2024
                -
                valor_2019
            )

        else:

            variacao_abs = np.nan


        if (
            pd.notna(
                variacao_abs
            )
            and
            valor_2019 != 0
        ):

            variacao_pct = (
                variacao_abs
                /
                abs(
                    valor_2019
                )
                *
                100
            )

        else:

            variacao_pct = np.nan


        resultados.append(
            {
                "codigo_ibge":
                    codigo_ibge,

                f"{prefixo}_n_obs":
                    n_obs,

                f"{prefixo}_media_periodo_disponivel":
                    media,

                f"{prefixo}_desvio_padrao":
                    desvio,

                f"{prefixo}_cv_pct":
                    cv_pct,

                f"{prefixo}_tendencia_por_ano":
                    tendencia,

                f"{prefixo}_valor_2019":
                    valor_2019,

                f"{prefixo}_valor_2024":
                    valor_2024,

                f"{prefixo}_variacao_abs_2019_2024":
                    variacao_abs,

                f"{prefixo}_variacao_pct_2019_2024":
                    variacao_pct
            }
        )


    return pd.DataFrame(
        resultados
    )

In [26]:
# ============================================================
# VARIÁVEIS PARA RESUMO TEMPORAL MUNICIPAL
# ============================================================

VARIAVEIS_RESUMO_TEMPORAL = {
    # --------------------------------------------------------
    # Produção
    # --------------------------------------------------------
    "area_colhida_ha":
        "area_colhida",

    "quantidade_produzida_t":
        "producao_soja",

    "rendimento_medio_kg_ha":
        "rendimento_soja",

    # --------------------------------------------------------
    # Clima
    # --------------------------------------------------------
    "precipitacao_anual_mm":
        "precipitacao",

    "temperatura_media_anual_c":
        "temperatura",

    "umidade_media_anual_pct":
        "umidade",

    # --------------------------------------------------------
    # Carbono do solo
    # --------------------------------------------------------
    "carbono_solo_t_ha":
        "carbono_solo",

    # --------------------------------------------------------
    # Uso e cobertura
    # --------------------------------------------------------
    "pct_cobertura_natural":
        "cobertura_natural",

    "pct_agricultura":
        "agricultura",

    "pct_soja_mapbiomas":
        "soja_mapbiomas",

    # --------------------------------------------------------
    # SEEG
    #
    # Mantido para análise descritiva.
    # Não será automaticamente usado no score.
    # --------------------------------------------------------
    "emissao_total_co2e_gwp_ar6_t":
        "seeg_co2e_completo"
}


print("=" * 70)
print("VARIÁVEIS DO RESUMO TEMPORAL")
print("=" * 70)


print(
    "\nQuantidade:",
    len(
        VARIAVEIS_RESUMO_TEMPORAL
    )
)


for coluna, prefixo in VARIAVEIS_RESUMO_TEMPORAL.items():

    print(
        f"{prefixo:<25} <- {coluna}"
    )

VARIÁVEIS DO RESUMO TEMPORAL

Quantidade: 11
area_colhida              <- area_colhida_ha
producao_soja             <- quantidade_produzida_t
rendimento_soja           <- rendimento_medio_kg_ha
precipitacao              <- precipitacao_anual_mm
temperatura               <- temperatura_media_anual_c
umidade                   <- umidade_media_anual_pct
carbono_solo              <- carbono_solo_t_ha
cobertura_natural         <- pct_cobertura_natural
agricultura               <- pct_agricultura
soja_mapbiomas            <- pct_soja_mapbiomas
seeg_co2e_completo        <- emissao_total_co2e_gwp_ar6_t


In [27]:
# ============================================================
# CONSTRUÇÃO DA BASE MUNICIPAL TEMPORAL
# ============================================================

base_municipal_temporal = (
    cobertura_temporal_municipal
    .copy()
)


resumos_temporais = {}


for coluna, prefixo in VARIAVEIS_RESUMO_TEMPORAL.items():

    resumo = resumir_variavel_municipal(
        dataframe=base_indicadores,
        coluna=coluna,
        prefixo=prefixo
    )


    resumos_temporais[
        prefixo
    ] = resumo


    base_municipal_temporal = (
        base_municipal_temporal
        .merge(
            resumo,
            on="codigo_ibge",
            how="left",
            validate="one_to_one"
        )
    )


print("=" * 70)
print("BASE MUNICIPAL TEMPORAL")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    base_municipal_temporal.shape
)


print(
    "\nMunicípios:"
)

print(
    base_municipal_temporal[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas codigo_ibge:"
)

print(
    base_municipal_temporal
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nNULL em codigo_ibge:"
)

print(
    base_municipal_temporal[
        "codigo_ibge"
    ]
    .isna()
    .sum()
)

BASE MUNICIPAL TEMPORAL

Dimensão:
(1505, 110)

Municípios:
1505

Duplicatas codigo_ibge:
0

NULL em codigo_ibge:
0


In [28]:
# ============================================================
# ELEGIBILIDADE REAL POR VARIÁVEL
# ============================================================

auditoria_elegibilidade_variavel = []


for coluna, prefixo in VARIAVEIS_RESUMO_TEMPORAL.items():

    coluna_n = (
        f"{prefixo}_n_obs"
    )

    coluna_variacao = (
        f"{prefixo}_variacao_abs_2019_2024"
    )


    auditoria_elegibilidade_variavel.append(
        {
            "variavel_original":
                coluna,

            "prefixo":
                prefixo,

            "municipios_com_dado":
                (
                    base_municipal_temporal[
                        coluna_n
                    ]
                    >= 1
                )
                .sum(),

            "elegiveis_volatilidade_n2":
                (
                    base_municipal_temporal[
                        coluna_n
                    ]
                    >= 2
                )
                .sum(),

            "elegiveis_tendencia_n3":
                (
                    base_municipal_temporal[
                        coluna_n
                    ]
                    >= 3
                )
                .sum(),

            "elegiveis_variacao_2019_2024":
                base_municipal_temporal[
                    coluna_variacao
                ]
                .notna()
                .sum()
        }
    )


auditoria_elegibilidade_variavel = (
    pd.DataFrame(
        auditoria_elegibilidade_variavel
    )
)


print("=" * 70)
print("ELEGIBILIDADE POR VARIÁVEL")
print("=" * 70)


display(
    auditoria_elegibilidade_variavel
)

ELEGIBILIDADE POR VARIÁVEL


,variavel_original,prefixo,municipios_com_dado,elegiveis_volatilidade_n2,elegiveis_tendencia_n3,elegiveis_variacao_2019_2024
0,area_colhida_ha,area_colhida,1504,1481,1454,1404
1,quantidade_produzida_t,producao_soja,1504,1481,1454,1404
2,rendimento_medio_kg_ha,rendimento_soja,1504,1481,1454,1404
3,precipitacao_anual_mm,precipitacao,1505,1481,1454,1406
4,temperatura_media_anual_c,temperatura,1505,1481,1454,1406
5,umidade_media_anual_pct,umidade,1505,1481,1454,1406
6,carbono_solo_t_ha,carbono_solo,1505,1481,1454,1406
7,pct_cobertura_natural,cobertura_natural,1505,1481,1454,1406
8,pct_agricultura,agricultura,1505,1481,1454,1406
9,pct_soja_mapbiomas,soja_mapbiomas,1505,1481,1454,1406


In [29]:
# ============================================================
# DIAGNÓSTICO DAS LACUNAS TEMPORAIS
# ============================================================

print("=" * 70)
print("LACUNAS PAM")
print("=" * 70)


# ------------------------------------------------------------
# Registros anuais PAM com valores ausentes
# ------------------------------------------------------------

registros_pam_ausentes = (
    base_indicadores.loc[
        (
            base_indicadores[
                "area_colhida_ha"
            ]
            .isna()
        )
        |
        (
            base_indicadores[
                "quantidade_produzida_t"
            ]
            .isna()
        )
        |
        (
            base_indicadores[
                "rendimento_medio_kg_ha"
            ]
            .isna()
        ),
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "ano",
            "area_plantada_ha",
            "area_colhida_ha",
            "quantidade_produzida_t",
            "rendimento_medio_kg_ha"
        ]
    ]
    .sort_values(
        by=[
            "uf",
            "municipio",
            "ano"
        ]
    )
)


print(
    "\nRegistros PAM com ausência:"
)

print(
    len(
        registros_pam_ausentes
    )
)


display(
    registros_pam_ausentes
)


# ------------------------------------------------------------
# Municípios sem nenhum valor PAM válido no período
# ------------------------------------------------------------

municipios_sem_producao_valida = (
    base_municipal_temporal.loc[
        base_municipal_temporal[
            "producao_soja_n_obs"
        ]
        .eq(0),
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao",
            "quantidade_anos",
            "primeiro_ano_disponivel",
            "ultimo_ano_disponivel"
        ]
    ]
)


print(
    "\nMunicípios sem nenhuma produção válida:"
)

display(
    municipios_sem_producao_valida
)


# ------------------------------------------------------------
# Municípios territorialmente elegíveis para 2019→2024,
# mas sem valores PAM válidos nos dois extremos
# ------------------------------------------------------------

pam_sem_variacao_2019_2024 = (
    base_municipal_temporal.loc[
        (
            base_municipal_temporal[
                "elegivel_variacao_2019_2024"
            ]
        )
        &
        (
            base_municipal_temporal[
                "producao_soja_variacao_abs_2019_2024"
            ]
            .isna()
        ),
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "producao_soja_n_obs",
            "producao_soja_valor_2019",
            "producao_soja_valor_2024"
        ]
    ]
)


print(
    "\nPossuem 2019 e 2024 na base, mas não produção válida nos dois anos:"
)

display(
    pam_sem_variacao_2019_2024
)


print(
    "\n" + "=" * 70
)

print(
    "LACUNAS SEEG COMPLETO"
)

print("=" * 70)


municipios_sem_seeg_completo = (
    base_municipal_temporal.loc[
        base_municipal_temporal[
            "seeg_co2e_completo_n_obs"
        ]
        .eq(0),
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "quantidade_anos",
            "seeg_co2e_completo_n_obs"
        ]
    ]
)


display(
    municipios_sem_seeg_completo
)

LACUNAS PAM

Registros PAM com ausência:
5


,codigo_ibge,municipio,uf,ano,area_plantada_ha,area_colhida_ha,quantidade_produzida_t,rendimento_medio_kg_ha
4145,4120200,Porto Rico,PR,2024,150.0,NaN,NaN,NaN
5254,4304606,Canoas,RS,2024,150.0,NaN,NaN,NaN
5404,4305454,Cidreira,RS,2024,500.0,NaN,NaN,NaN
5904,4310330,Imbé,RS,2024,150.0,NaN,NaN,NaN
6394,4314050,Parobé,RS,2020,17.0,NaN,NaN,NaN



Municípios sem nenhuma produção válida:


,codigo_ibge,municipio,uf,regiao,quantidade_anos,primeiro_ano_disponivel,ultimo_ano_disponivel
794,4310330,Imbé,RS,Sul,1,2024,2024



Possuem 2019 e 2024 na base, mas não produção válida nos dois anos:


,codigo_ibge,municipio,uf,producao_soja_n_obs,producao_soja_valor_2019,producao_soja_valor_2024
683,4304606,Canoas,RS,5,216.0,NaN
709,4305454,Cidreira,RS,5,780.0,NaN



LACUNAS SEEG COMPLETO


,codigo_ibge,municipio,uf,quantidade_anos,seeg_co2e_completo_n_obs
900,4314803,Portão,RS,6,0
1037,4322608,Venâncio Aires,RS,6,0


In [30]:
# ============================================================
# VALIDAÇÃO AUTOMÁTICA DOS RESUMOS TEMPORAIS
# ============================================================

validacao_resumos = []


for coluna, prefixo in VARIAVEIS_RESUMO_TEMPORAL.items():

    # --------------------------------------------------------
    # Contagem esperada diretamente da base anual
    # --------------------------------------------------------

    contagem_esperada = (
        base_indicadores
        .groupby(
            "codigo_ibge"
        )[
            coluna
        ]
        .count()
        .reindex(
            base_municipal_temporal[
                "codigo_ibge"
            ]
        )
        .fillna(0)
        .astype(int)
        .to_numpy()
    )


    contagem_observada = (
        base_municipal_temporal[
            f"{prefixo}_n_obs"
        ]
        .astype(int)
        .to_numpy()
    )


    falhas_n_obs = (
        contagem_esperada
        !=
        contagem_observada
    ).sum()


    # --------------------------------------------------------
    # Inclinação:
    # n < 3 precisa ficar NULL
    # n >= 3 precisa existir
    # --------------------------------------------------------

    n_obs = (
        base_municipal_temporal[
            f"{prefixo}_n_obs"
        ]
    )


    inclinacao = (
        base_municipal_temporal[
            f"{prefixo}_tendencia_por_ano"
        ]
    )


    falhas_inclinacao_n_menor_3 = (
        (
            n_obs
            < 3
        )
        &
        inclinacao.notna()
    ).sum()


    falhas_inclinacao_n_maior_igual_3 = (
        (
            n_obs
            >= 3
        )
        &
        inclinacao.isna()
    ).sum()


    # --------------------------------------------------------
    # Elegibilidade real 2019 → 2024
    # --------------------------------------------------------

    pivot = (
        base_indicadores
        .pivot(
            index="codigo_ibge",
            columns="ano",
            values=coluna
        )
    )


    possui_2019_2024 = (
        pivot[
            2019
        ]
        .notna()
        &
        pivot[
            2024
        ]
        .notna()
    )


    possui_2019_2024 = (
        possui_2019_2024
        .reindex(
            base_municipal_temporal[
                "codigo_ibge"
            ]
        )
        .fillna(False)
        .to_numpy()
    )


    variacao_observada = (
        base_municipal_temporal[
            f"{prefixo}_variacao_abs_2019_2024"
        ]
        .notna()
        .to_numpy()
    )


    falhas_variacao = (
        possui_2019_2024
        !=
        variacao_observada
    ).sum()


    validacao_resumos.append(
        {
            "variavel":
                coluna,

            "falhas_n_obs":
                falhas_n_obs,

            "falhas_inclinacao_n_menor_3":
                falhas_inclinacao_n_menor_3,

            "falhas_inclinacao_n_maior_igual_3":
                falhas_inclinacao_n_maior_igual_3,

            "falhas_elegibilidade_2019_2024":
                falhas_variacao
        }
    )


validacao_resumos = pd.DataFrame(
    validacao_resumos
)


print("=" * 70)
print("VALIDAÇÃO DOS RESUMOS TEMPORAIS")
print("=" * 70)


display(
    validacao_resumos
)


print(
    "\nTotal de falhas:"
)

print(
    validacao_resumos[
        [
            "falhas_n_obs",
            "falhas_inclinacao_n_menor_3",
            "falhas_inclinacao_n_maior_igual_3",
            "falhas_elegibilidade_2019_2024"
        ]
    ]
    .sum()
    .sum()
)

VALIDAÇÃO DOS RESUMOS TEMPORAIS


,variavel,falhas_n_obs,falhas_inclinacao_n_menor_3,falhas_inclinacao_n_maior_igual_3,falhas_elegibilidade_2019_2024
0,area_colhida_ha,0,0,0,0
1,quantidade_produzida_t,0,0,0,0
2,rendimento_medio_kg_ha,0,0,0,0
3,precipitacao_anual_mm,0,0,0,0
4,temperatura_media_anual_c,0,0,0,0
5,umidade_media_anual_pct,0,0,0,0
6,carbono_solo_t_ha,0,0,0,0
7,pct_cobertura_natural,0,0,0,0
8,pct_agricultura,0,0,0,0
9,pct_soja_mapbiomas,0,0,0,0



Total de falhas:
0


In [31]:
# ============================================================
# INTEGRIDADE TÉCNICA — BASE MUNICIPAL TEMPORAL
# ============================================================

print("=" * 70)
print("INTEGRIDADE — BASE MUNICIPAL TEMPORAL")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    base_municipal_temporal.shape
)


print(
    "\nDuplicatas codigo_ibge:"
)

print(
    base_municipal_temporal
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nNULL codigo_ibge:"
)

print(
    base_municipal_temporal[
        "codigo_ibge"
    ]
    .isna()
    .sum()
)


print(
    "\nQuantidade de municípios:"
)

print(
    base_municipal_temporal[
        "codigo_ibge"
    ]
    .nunique()
)


# ------------------------------------------------------------
# Infinitos
# ------------------------------------------------------------

colunas_numericas_municipais = (
    base_municipal_temporal
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)


quantidade_inf_municipal = (
    np.isinf(
        base_municipal_temporal[
            colunas_numericas_municipais
        ]
    )
    .sum()
)


colunas_com_inf_municipal = (
    quantidade_inf_municipal[
        quantidade_inf_municipal
        > 0
    ]
)


print(
    "\nColunas com infinitos:"
)


if len(
    colunas_com_inf_municipal
) == 0:

    print(
        "Nenhuma"
    )

else:

    display(
        colunas_com_inf_municipal
    )


# ------------------------------------------------------------
# Sufixos acidentais
# ------------------------------------------------------------

colunas_merge_indesejadas = [
    coluna
    for coluna in base_municipal_temporal.columns
    if (
        coluna.endswith("_x")
        or
        coluna.endswith("_y")
    )
]


print(
    "\nColunas _x/_y:"
)

print(
    colunas_merge_indesejadas
)

INTEGRIDADE — BASE MUNICIPAL TEMPORAL

Dimensão:
(1505, 110)

Duplicatas codigo_ibge:
0

NULL codigo_ibge:
0

Quantidade de municípios:
1505

Colunas com infinitos:
Nenhuma

Colunas _x/_y:
[]


In [32]:
# ============================================================
# SELEÇÃO DE INDICADORES MUNICIPAIS CANDIDATOS
# ============================================================

COLUNAS_MUNICIPAIS_SELECIONADAS = [
    # --------------------------------------------------------
    # Identificação e cobertura temporal
    # --------------------------------------------------------
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",

    "quantidade_anos",
    "primeiro_ano_disponivel",
    "ultimo_ano_disponivel",

    # --------------------------------------------------------
    # Escala produtiva
    # --------------------------------------------------------
    "producao_soja_n_obs",
    "producao_soja_media_periodo_disponivel",
    "producao_soja_valor_2024",

    "area_colhida_media_periodo_disponivel",
    "area_colhida_valor_2024",

    # --------------------------------------------------------
    # Eficiência produtiva
    # --------------------------------------------------------
    "rendimento_soja_n_obs",
    "rendimento_soja_media_periodo_disponivel",
    "rendimento_soja_cv_pct",
    "rendimento_soja_tendencia_por_ano",
    "rendimento_soja_variacao_abs_2019_2024",

    # --------------------------------------------------------
    # Clima
    # --------------------------------------------------------
    "precipitacao_media_periodo_disponivel",
    "precipitacao_cv_pct",

    "temperatura_media_periodo_disponivel",
    "temperatura_desvio_padrao",

    "umidade_media_periodo_disponivel",
    "umidade_desvio_padrao",

    # --------------------------------------------------------
    # Carbono do solo
    # --------------------------------------------------------
    "carbono_solo_n_obs",
    "carbono_solo_media_periodo_disponivel",
    "carbono_solo_valor_2024",
    "carbono_solo_tendencia_por_ano",
    "carbono_solo_variacao_abs_2019_2024",

    # --------------------------------------------------------
    # Cobertura natural
    # --------------------------------------------------------
    "cobertura_natural_n_obs",
    "cobertura_natural_valor_2024",
    "cobertura_natural_tendencia_por_ano",
    "cobertura_natural_variacao_abs_2019_2024",

    # --------------------------------------------------------
    # Agricultura
    # --------------------------------------------------------
    "agricultura_valor_2024",
    "agricultura_tendencia_por_ano",
    "agricultura_variacao_abs_2019_2024",

    # --------------------------------------------------------
    # Soja — MapBiomas
    # --------------------------------------------------------
    "soja_mapbiomas_valor_2024",
    "soja_mapbiomas_tendencia_por_ano",
    "soja_mapbiomas_variacao_abs_2019_2024",

    # --------------------------------------------------------
    # SEEG
    # Apenas descritivo por enquanto
    # --------------------------------------------------------
    "seeg_co2e_completo_n_obs",
    "seeg_co2e_completo_media_periodo_disponivel",
    "seeg_co2e_completo_valor_2024"
]


base_municipal_candidatos = (
    base_municipal_temporal[
        COLUNAS_MUNICIPAIS_SELECIONADAS
    ]
    .copy()
)


# ------------------------------------------------------------
# Renomear para nomes finais mais legíveis
# ------------------------------------------------------------

base_municipal_candidatos = (
    base_municipal_candidatos
    .rename(
        columns={
            "producao_soja_media_periodo_disponivel":
                "producao_media_t",

            "producao_soja_valor_2024":
                "producao_2024_t",

            "area_colhida_media_periodo_disponivel":
                "area_colhida_media_ha",

            "area_colhida_valor_2024":
                "area_colhida_2024_ha",

            "rendimento_soja_media_periodo_disponivel":
                "rendimento_medio_kg_ha",

            "rendimento_soja_cv_pct":
                "rendimento_cv_pct",

            "rendimento_soja_tendencia_por_ano":
                "rendimento_tendencia_kg_ha_ano",

            "rendimento_soja_variacao_abs_2019_2024":
                "rendimento_variacao_2019_2024_kg_ha",

            "precipitacao_media_periodo_disponivel":
                "precipitacao_media_mm",

            "temperatura_media_periodo_disponivel":
                "temperatura_media_c",

            "temperatura_desvio_padrao":
                "temperatura_desvio_padrao_c",

            "umidade_media_periodo_disponivel":
                "umidade_media_pct",

            "umidade_desvio_padrao":
                "umidade_desvio_padrao_pct",

            "carbono_solo_media_periodo_disponivel":
                "carbono_solo_medio_t_ha",

            "carbono_solo_valor_2024":
                "carbono_solo_2024_t_ha",

            "carbono_solo_tendencia_por_ano":
                "carbono_solo_tendencia_t_ha_ano",

            "carbono_solo_variacao_abs_2019_2024":
                "carbono_solo_variacao_2019_2024_t_ha",

            "cobertura_natural_valor_2024":
                "cobertura_natural_2024_pct",

            "cobertura_natural_tendencia_por_ano":
                "cobertura_natural_tendencia_pp_ano",

            "cobertura_natural_variacao_abs_2019_2024":
                "cobertura_natural_variacao_2019_2024_pp",

            "agricultura_valor_2024":
                "agricultura_2024_pct",

            "agricultura_tendencia_por_ano":
                "agricultura_tendencia_pp_ano",

            "agricultura_variacao_abs_2019_2024":
                "agricultura_variacao_2019_2024_pp",

            "soja_mapbiomas_valor_2024":
                "soja_mapbiomas_2024_pct",

            "soja_mapbiomas_tendencia_por_ano":
                "soja_mapbiomas_tendencia_pp_ano",

            "soja_mapbiomas_variacao_abs_2019_2024":
                "soja_mapbiomas_variacao_2019_2024_pp",

            "seeg_co2e_completo_media_periodo_disponivel":
                "seeg_co2e_medio_t",

            "seeg_co2e_completo_valor_2024":
                "seeg_co2e_2024_t"
        }
    )
)


print("=" * 70)
print("BASE MUNICIPAL — INDICADORES CANDIDATOS")
print("=" * 70)

print(
    "\nDimensão:",
    base_municipal_candidatos.shape
)

print(
    "Municípios:",
    base_municipal_candidatos[
        "codigo_ibge"
    ].nunique()
)

print(
    "Duplicatas:",
    base_municipal_candidatos
    .duplicated(
        subset=["codigo_ibge"]
    )
    .sum()
)

BASE MUNICIPAL — INDICADORES CANDIDATOS

Dimensão: (1505, 41)
Municípios: 1505
Duplicatas: 0


In [33]:
# ============================================================
# AUDITORIA DE NULL — INDICADORES MUNICIPAIS CANDIDATOS
# ============================================================

auditoria_null_candidatos = (
    pd.DataFrame(
        {
            "coluna":
                base_municipal_candidatos.columns,

            "quantidade_null":
                base_municipal_candidatos
                .isna()
                .sum()
                .values
        }
    )
)


auditoria_null_candidatos[
    "percentual_null"
] = (
    auditoria_null_candidatos[
        "quantidade_null"
    ]
    /
    len(
        base_municipal_candidatos
    )
    *
    100
)


auditoria_null_candidatos = (
    auditoria_null_candidatos[
        auditoria_null_candidatos[
            "quantidade_null"
        ]
        > 0
    ]
    .sort_values(
        by=[
            "quantidade_null",
            "coluna"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 70)
print("NULL — INDICADORES MUNICIPAIS CANDIDATOS")
print("=" * 70)


print(
    "\nColunas com NULL:",
    len(
        auditoria_null_candidatos
    )
)


display(
    auditoria_null_candidatos
)

NULL — INDICADORES MUNICIPAIS CANDIDATOS

Colunas com NULL: 25


,coluna,quantidade_null,percentual_null
0,rendimento_variacao_2019_2024_kg_ha,101,6.710963
1,agricultura_variacao_2019_2024_pp,99,6.578073
2,carbono_solo_variacao_2019_2024_t_ha,99,6.578073
3,cobertura_natural_variacao_2019_2024_pp,99,6.578073
4,soja_mapbiomas_variacao_2019_2024_pp,99,6.578073
5,agricultura_tendencia_pp_ano,51,3.388704
6,carbono_solo_tendencia_t_ha_ano,51,3.388704
7,cobertura_natural_tendencia_pp_ano,51,3.388704
8,rendimento_tendencia_kg_ha_ano,51,3.388704
9,soja_mapbiomas_tendencia_pp_ano,51,3.388704


In [34]:
# ============================================================
# CORRELAÇÃO ENTRE INDICADORES MUNICIPAIS CANDIDATOS
# ============================================================

COLUNAS_CORRELACAO = [
    # Escala
    "producao_media_t",
    "producao_2024_t",
    "area_colhida_media_ha",
    "area_colhida_2024_ha",

    # Eficiência
    "rendimento_medio_kg_ha",
    "rendimento_cv_pct",
    "rendimento_tendencia_kg_ha_ano",

    # Clima
    "precipitacao_media_mm",
    "precipitacao_cv_pct",
    "temperatura_media_c",
    "temperatura_desvio_padrao_c",
    "umidade_media_pct",
    "umidade_desvio_padrao_pct",

    # Carbono
    "carbono_solo_medio_t_ha",
    "carbono_solo_2024_t_ha",
    "carbono_solo_variacao_2019_2024_t_ha",

    # Cobertura
    "cobertura_natural_2024_pct",
    "cobertura_natural_variacao_2019_2024_pp",

    "agricultura_2024_pct",
    "agricultura_variacao_2019_2024_pp",

    "soja_mapbiomas_2024_pct",
    "soja_mapbiomas_variacao_2019_2024_pp",

    # SEEG — propositalmente incluído para testar redundância
    "seeg_co2e_medio_t",
    "seeg_co2e_2024_t"
]


matriz_correlacao = (
    base_municipal_candidatos[
        COLUNAS_CORRELACAO
    ]
    .corr(
        method="pearson",
        min_periods=30
    )
)


pares_correlacionados = []


for i, coluna_a in enumerate(
    COLUNAS_CORRELACAO
):

    for coluna_b in COLUNAS_CORRELACAO[
        i + 1:
    ]:

        correlacao = (
            matriz_correlacao
            .loc[
                coluna_a,
                coluna_b
            ]
        )


        if (
            pd.notna(
                correlacao
            )
            and
            abs(
                correlacao
            )
            >= 0.85
        ):

            pares_correlacionados.append(
                {
                    "variavel_a":
                        coluna_a,

                    "variavel_b":
                        coluna_b,

                    "correlacao":
                        correlacao,

                    "correlacao_abs":
                        abs(
                            correlacao
                        )
                }
            )


pares_correlacionados = (
    pd.DataFrame(
        pares_correlacionados
    )
)


if not pares_correlacionados.empty:

    pares_correlacionados = (
        pares_correlacionados
        .sort_values(
            by="correlacao_abs",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


print("=" * 70)
print("PARES COM |CORRELAÇÃO| >= 0.85")
print("=" * 70)


if pares_correlacionados.empty:

    print(
        "Nenhum par encontrado."
    )

else:

    display(
        pares_correlacionados
    )

PARES COM |CORRELAÇÃO| >= 0.85


,variavel_a,variavel_b,correlacao,correlacao_abs
0,producao_2024_t,seeg_co2e_2024_t,1.000000,1.000000
1,producao_media_t,seeg_co2e_medio_t,1.000000,1.000000
2,carbono_solo_medio_t_ha,carbono_solo_2024_t_ha,0.999892,0.999892
3,area_colhida_media_ha,area_colhida_2024_ha,0.993751,0.993751
4,producao_media_t,area_colhida_media_ha,0.992633,0.992633
5,area_colhida_media_ha,seeg_co2e_medio_t,0.992632,0.992632
6,producao_media_t,area_colhida_2024_ha,0.987059,0.987059
7,area_colhida_2024_ha,seeg_co2e_medio_t,0.987058,0.987058
8,seeg_co2e_medio_t,seeg_co2e_2024_t,0.986589,0.986589
9,producao_media_t,seeg_co2e_2024_t,0.986586,0.986586


In [35]:
# ============================================================
# DECISÕES DE REDUNDÂNCIA ENTRE INDICADORES
# ============================================================

decisoes_redundancia = pd.DataFrame(
    [
        {
            "dimensao": "Escala produtiva",
            "variavel": "producao_media_t",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa escala produtiva com maior cobertura temporal "
                "que o valor isolado de 2024."
            )
        },

        {
            "dimensao": "Escala produtiva",
            "variavel": "area_colhida_media_ha",
            "decisao": "descritiva",
            "uso": "contexto",
            "justificativa": (
                "Altamente correlacionada com produção média; evitar peso duplo."
            )
        },

        {
            "dimensao": "Emissões SEEG",
            "variavel": "seeg_co2e_medio_t",
            "decisao": "descritiva",
            "uso": "contexto_emissoes",
            "justificativa": (
                "Correlação praticamente perfeita com produção média; "
                "não representa dimensão independente."
            )
        },

        {
            "dimensao": "Eficiência produtiva",
            "variavel": "rendimento_medio_kg_ha",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa produtividade e não apenas escala."
            )
        },

        {
            "dimensao": "Eficiência produtiva",
            "variavel": "rendimento_cv_pct",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa variabilidade interanual do rendimento."
            )
        },

        {
            "dimensao": "Eficiência produtiva",
            "variavel": "rendimento_tendencia_kg_ha_ano",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Inclinação linear descritiva do rendimento no período observado."
            )
        },

        {
            "dimensao": "Carbono do solo",
            "variavel": "carbono_solo_medio_t_ha",
            "decisao": "descritiva",
            "uso": "contexto",
            "justificativa": (
                "Praticamente redundante com carbono do solo em 2024."
            )
        },

        {
            "dimensao": "Carbono do solo",
            "variavel": "carbono_solo_2024_t_ha",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa o estado mais recente disponível."
            )
        },

        {
            "dimensao": "Carbono do solo",
            "variavel": "carbono_solo_variacao_2019_2024_t_ha",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa mudança observada entre os extremos disponíveis."
            )
        },

        {
            "dimensao": "Uso da terra",
            "variavel": "agricultura_2024_pct",
            "decisao": "descritiva",
            "uso": "contexto",
            "justificativa": (
                "Fortemente correlacionada com percentual de soja MapBiomas."
            )
        },

        {
            "dimensao": "Uso da terra",
            "variavel": "soja_mapbiomas_2024_pct",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Mais específica ao objeto do projeto: soja."
            )
        },

        {
            "dimensao": "Conservação",
            "variavel": "cobertura_natural_2024_pct",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa o estado recente da cobertura natural."
            )
        },

        {
            "dimensao": "Conservação",
            "variavel": "cobertura_natural_variacao_2019_2024_pp",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa mudança em pontos percentuais."
            )
        }
    ]
)


print("=" * 70)
print("DECISÕES SOBRE REDUNDÂNCIA")
print("=" * 70)

display(
    decisoes_redundancia
)

DECISÕES SOBRE REDUNDÂNCIA


,dimensao,variavel,decisao,uso,justificativa
0,Escala produtiva,producao_media_t,manter,candidata_priorizacao,Representa escala produtiva com maior cobertur...
1,Escala produtiva,area_colhida_media_ha,descritiva,contexto,Altamente correlacionada com produção média; e...
2,Emissões SEEG,seeg_co2e_medio_t,descritiva,contexto_emissoes,Correlação praticamente perfeita com produção ...
3,Eficiência produtiva,rendimento_medio_kg_ha,manter,candidata_priorizacao,Representa produtividade e não apenas escala.
4,Eficiência produtiva,rendimento_cv_pct,manter,candidata_priorizacao,Representa variabilidade interanual do rendime...
5,Eficiência produtiva,rendimento_tendencia_kg_ha_ano,manter,candidata_priorizacao,Inclinação linear descritiva do rendimento no ...
6,Carbono do solo,carbono_solo_medio_t_ha,descritiva,contexto,Praticamente redundante com carbono do solo em...
7,Carbono do solo,carbono_solo_2024_t_ha,manter,candidata_priorizacao,Representa o estado mais recente disponível.
8,Carbono do solo,carbono_solo_variacao_2019_2024_t_ha,manter,candidata_priorizacao,Representa mudança observada entre os extremos...
9,Uso da terra,agricultura_2024_pct,descritiva,contexto,Fortemente correlacionada com percentual de so...


In [36]:
# ============================================================
# BASE MUNICIPAL REDUZIDA — PRÉ-BRLUC
# ============================================================

COLUNAS_BASE_REDUZIDA = [
    # --------------------------------------------------------
    # Identificação / qualidade temporal
    # --------------------------------------------------------
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",

    "quantidade_anos",
    "primeiro_ano_disponivel",
    "ultimo_ano_disponivel",

    # --------------------------------------------------------
    # Escala produtiva
    # --------------------------------------------------------
    "producao_soja_n_obs",
    "producao_media_t",

    # Contexto, sem peso independente
    "area_colhida_media_ha",

    # --------------------------------------------------------
    # Eficiência produtiva
    # --------------------------------------------------------
    "rendimento_soja_n_obs",
    "rendimento_medio_kg_ha",
    "rendimento_cv_pct",
    "rendimento_tendencia_kg_ha_ano",
    "rendimento_variacao_2019_2024_kg_ha",

    # --------------------------------------------------------
    # Clima
    #
    # Médias = contexto climático.
    # Variabilidade = candidata analítica.
    # --------------------------------------------------------
    "precipitacao_media_mm",
    "precipitacao_cv_pct",

    "temperatura_media_c",
    "temperatura_desvio_padrao_c",

    "umidade_media_pct",
    "umidade_desvio_padrao_pct",

    # --------------------------------------------------------
    # Carbono do solo
    # --------------------------------------------------------
    "carbono_solo_n_obs",
    "carbono_solo_2024_t_ha",
    "carbono_solo_tendencia_t_ha_ano",
    "carbono_solo_variacao_2019_2024_t_ha",

    # --------------------------------------------------------
    # Cobertura natural
    # --------------------------------------------------------
    "cobertura_natural_n_obs",
    "cobertura_natural_2024_pct",
    "cobertura_natural_tendencia_pp_ano",
    "cobertura_natural_variacao_2019_2024_pp",

    # --------------------------------------------------------
    # Soja no território
    # --------------------------------------------------------
    "soja_mapbiomas_2024_pct",
    "soja_mapbiomas_tendencia_pp_ano",
    "soja_mapbiomas_variacao_2019_2024_pp",

    # Agricultura total fica como contexto
    "agricultura_2024_pct",

    # --------------------------------------------------------
    # SEEG
    # Descritivo — sem peso independente
    # --------------------------------------------------------
    "seeg_co2e_completo_n_obs",
    "seeg_co2e_medio_t"
]


base_municipal_reduzida = (
    base_municipal_candidatos[
        COLUNAS_BASE_REDUZIDA
    ]
    .copy()
)


print("=" * 70)
print("BASE MUNICIPAL REDUZIDA — PRÉ-BRLUC")
print("=" * 70)


print(
    "\nDimensão:",
    base_municipal_reduzida.shape
)

print(
    "Municípios:",
    base_municipal_reduzida[
        "codigo_ibge"
    ]
    .nunique()
)

print(
    "Duplicatas:",
    base_municipal_reduzida
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)

BASE MUNICIPAL REDUZIDA — PRÉ-BRLUC

Dimensão: (1505, 35)
Municípios: 1505
Duplicatas: 0


In [37]:
# ============================================================
# EXTRAÇÃO DA CAMADA BRLUC — UMA LINHA POR MUNICÍPIO
# ============================================================

COLUNAS_BRLUC_CANDIDATAS = [
    "codigo_ibge",

    "periodo_inicio_brluc",
    "periodo_fim_brluc",

    # --------------------------------------------------------
    # Transição para soja
    # --------------------------------------------------------
    "area_soja_t1_brluc_ha",
    "area_soja_persistente_2000_2019_ha",
    "area_conversao_para_soja_2000_2019_ha",

    "percentual_persistencia_soja_pct",
    "percentual_conversao_para_soja_pct",

    "area_origem_natural_ha",

    # --------------------------------------------------------
    # Emissões BRLUC
    # --------------------------------------------------------
    "emissao_absoluta_co2_t_ano",
    "taxa_emissao_co2_t_ha_ano",

    "ic95_emissao_absoluta_disponivel",
    "ic95_taxa_emissao_disponivel",

    # --------------------------------------------------------
    # Estoques de carbono BRLUC
    # --------------------------------------------------------
    "soc_t0_t_c_ha",
    "soc_t1_t_c_ha",

    "ctotal_t0_t_c_ha",
    "ctotal_t1_t_c_ha",

    "delta_soc_classes_brluc_t1_t0_t_c_ha",
    "delta_ctotal_classes_brluc_t1_t0_t_c_ha",

    # --------------------------------------------------------
    # Incerteza
    # --------------------------------------------------------
    "soc_t1_incerteza_pct",
    "ctotal_t1_incerteza_pct"
]


# ------------------------------------------------------------
# Verificar novamente se essas variáveis são realmente
# constantes dentro de cada município
# ------------------------------------------------------------

auditoria_estatica_brluc = []


for coluna in COLUNAS_BRLUC_CANDIDATAS[1:]:

    max_distintos = (
        base
        .groupby(
            "codigo_ibge"
        )[
            coluna
        ]
        .nunique(
            dropna=False
        )
        .max()
    )

    auditoria_estatica_brluc.append(
        {
            "coluna":
                coluna,

            "max_valores_distintos_por_municipio":
                max_distintos
        }
    )


auditoria_estatica_brluc = pd.DataFrame(
    auditoria_estatica_brluc
)


print("=" * 70)
print("AUDITORIA BRLUC ESTÁTICO")
print("=" * 70)


display(
    auditoria_estatica_brluc
)


print(
    "\nTodas constantes por município:"
)

print(
    (
        auditoria_estatica_brluc[
            "max_valores_distintos_por_municipio"
        ]
        <= 1
    )
    .all()
)


# ------------------------------------------------------------
# Extrair uma linha por município
# ------------------------------------------------------------

brluc_municipal_candidatos = (
    base[
        COLUNAS_BRLUC_CANDIDATAS
    ]
    .drop_duplicates(
        subset=[
            "codigo_ibge"
        ]
    )
    .copy()
)


print(
    "\nDimensão BRLUC municipal:"
)

print(
    brluc_municipal_candidatos.shape
)


print(
    "\nDuplicatas codigo_ibge:"
)

print(
    brluc_municipal_candidatos
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)

AUDITORIA BRLUC ESTÁTICO


,coluna,max_valores_distintos_por_municipio
0,periodo_inicio_brluc,1
1,periodo_fim_brluc,1
2,area_soja_t1_brluc_ha,1
3,area_soja_persistente_2000_2019_ha,1
4,area_conversao_para_soja_2000_2019_ha,1
5,percentual_persistencia_soja_pct,1
6,percentual_conversao_para_soja_pct,1
7,area_origem_natural_ha,1
8,emissao_absoluta_co2_t_ano,1
9,taxa_emissao_co2_t_ha_ano,1



Todas constantes por município:
True

Dimensão BRLUC municipal:
(1505, 21)

Duplicatas codigo_ibge:
0


In [38]:
# ============================================================
# PERFIL DAS VARIÁVEIS BRLUC MUNICIPAIS
# ============================================================

COLUNAS_BRLUC_NUMERICAS_DIAGNOSTICO = [
    "area_soja_t1_brluc_ha",
    "area_soja_persistente_2000_2019_ha",
    "area_conversao_para_soja_2000_2019_ha",

    "percentual_persistencia_soja_pct",
    "percentual_conversao_para_soja_pct",

    "area_origem_natural_ha",

    "emissao_absoluta_co2_t_ano",
    "taxa_emissao_co2_t_ha_ano",

    "soc_t0_t_c_ha",
    "soc_t1_t_c_ha",

    "ctotal_t0_t_c_ha",
    "ctotal_t1_t_c_ha",

    "delta_soc_classes_brluc_t1_t0_t_c_ha",
    "delta_ctotal_classes_brluc_t1_t0_t_c_ha",

    "soc_t1_incerteza_pct",
    "ctotal_t1_incerteza_pct"
]


perfil_brluc = []


for coluna in COLUNAS_BRLUC_NUMERICAS_DIAGNOSTICO:

    serie = (
        brluc_municipal_candidatos[
            coluna
        ]
    )


    perfil_brluc.append(
        {
            "variavel":
                coluna,

            "n":
                serie
                .notna()
                .sum(),

            "null":
                serie
                .isna()
                .sum(),

            "negativos":
                (
                    serie
                    < 0
                )
                .sum(),

            "zeros":
                (
                    serie
                    == 0
                )
                .sum(),

            "minimo":
                serie.min(),

            "mediana":
                serie.median(),

            "media":
                serie.mean(),

            "maximo":
                serie.max()
        }
    )


perfil_brluc = pd.DataFrame(
    perfil_brluc
)


print("=" * 70)
print("PERFIL DAS VARIÁVEIS BRLUC")
print("=" * 70)


display(
    perfil_brluc
)

PERFIL DAS VARIÁVEIS BRLUC


,variavel,n,null,negativos,zeros,minimo,mediana,media,maximo
0,area_soja_t1_brluc_ha,1505,0,0,27,0.000000,5027.818342,18448.595028,5.834695e+05
1,area_soja_persistente_2000_2019_ha,1505,0,0,178,0.000000,305.146161,5917.791182,3.599788e+05
2,area_conversao_para_soja_2000_2019_ha,1505,0,0,30,0.000000,3399.248465,12530.803845,3.550016e+05
3,percentual_persistencia_soja_pct,1478,27,0,151,0.000000,9.046333,20.527106,1.000000e+02
4,percentual_conversao_para_soja_pct,1478,27,0,3,0.000000,90.953667,79.472894,1.000000e+02
5,area_origem_natural_ha,1505,0,0,131,0.000000,62.965760,3052.826166,2.399102e+05
6,emissao_absoluta_co2_t_ano,1505,0,17,31,-3370.005124,2870.996508,38896.835522,3.463402e+06
7,taxa_emissao_co2_t_ha_ano,1505,0,17,31,-0.248041,0.710774,1.566974,2.662434e+01
8,soc_t0_t_c_ha,1505,0,0,0,16.060909,38.686206,46.268173,9.339134e+01
9,soc_t1_t_c_ha,1505,0,0,0,16.049937,39.293259,47.507748,9.377226e+01


In [39]:
# ============================================================
# REDUNDÂNCIAS MATEMÁTICAS — BRLUC MUNICIPAL
# ============================================================

# ------------------------------------------------------------
# 1. Persistência + conversão = área soja t1
# ------------------------------------------------------------

dif_area_soja = (
    brluc_municipal_candidatos[
        "area_soja_t1_brluc_ha"
    ]
    -
    (
        brluc_municipal_candidatos[
            "area_soja_persistente_2000_2019_ha"
        ]
        +
        brluc_municipal_candidatos[
            "area_conversao_para_soja_2000_2019_ha"
        ]
    )
).abs()


# ------------------------------------------------------------
# 2. Percentual persistência + conversão = 100
# apenas quando ambos estão definidos
# ------------------------------------------------------------

soma_percentuais = (
    brluc_municipal_candidatos[
        "percentual_persistencia_soja_pct"
    ]
    +
    brluc_municipal_candidatos[
        "percentual_conversao_para_soja_pct"
    ]
)


dif_percentuais = (
    soma_percentuais
    -
    100
).abs()


# ------------------------------------------------------------
# 3. Delta Ctotal ≈ Delta SOC
# ------------------------------------------------------------

dif_delta_carbono = (
    brluc_municipal_candidatos[
        "delta_ctotal_classes_brluc_t1_t0_t_c_ha"
    ]
    -
    brluc_municipal_candidatos[
        "delta_soc_classes_brluc_t1_t0_t_c_ha"
    ]
).abs()


# ------------------------------------------------------------
# 4. Ctotal t1 - SOC t1
# Deve refletir Cveg da classe
# ------------------------------------------------------------

diferenca_ctotal_soc_t1 = (
    brluc_municipal_candidatos[
        "ctotal_t1_t_c_ha"
    ]
    -
    brluc_municipal_candidatos[
        "soc_t1_t_c_ha"
    ]
)


print("=" * 70)
print("REDUNDÂNCIAS MATEMÁTICAS — BRLUC")
print("=" * 70)


print(
    "\nÁrea soja t1 = persistente + conversão"
)

print(
    "Maior diferença:",
    dif_area_soja.max()
)

print(
    "Falhas > 0.000001:",
    (
        dif_area_soja
        > 0.000001
    ).sum()
)


print(
    "\nPersistência % + conversão % = 100"
)

print(
    "Registros comparáveis:",
    dif_percentuais
    .notna()
    .sum()
)

print(
    "Maior diferença:",
    dif_percentuais.max()
)

print(
    "Falhas > 0.000001:",
    (
        dif_percentuais
        > 0.000001
    ).sum()
)


print(
    "\nDelta Ctotal = Delta SOC"
)

print(
    "Maior diferença:",
    dif_delta_carbono.max()
)

print(
    "Falhas > 0.000001:",
    (
        dif_delta_carbono
        > 0.000001
    ).sum()
)


print(
    "\nCtotal t1 - SOC t1"
)

print(
    "Valores distintos arredondados a 6 casas:",
    diferenca_ctotal_soc_t1
    .round(6)
    .nunique()
)

print(
    "Mínimo:",
    diferenca_ctotal_soc_t1.min()
)

print(
    "Máximo:",
    diferenca_ctotal_soc_t1.max()
)

REDUNDÂNCIAS MATEMÁTICAS — BRLUC

Área soja t1 = persistente + conversão
Maior diferença: 5.820766091346741e-11
Falhas > 0.000001: 0

Persistência % + conversão % = 100
Registros comparáveis: 1478
Maior diferença: 2.842170943040401e-14
Falhas > 0.000001: 0

Delta Ctotal = Delta SOC
Maior diferença: 1.141309269314661e-13
Falhas > 0.000001: 0

Ctotal t1 - SOC t1
Valores distintos arredondados a 6 casas: 1
Mínimo: 4.7058750524922885
Máximo: 4.705875052492409


In [40]:
# ============================================================
# CORRELAÇÃO ENTRE CANDIDATOS BRLUC
# ============================================================

COLUNAS_BRLUC_CORRELACAO = [
    "area_soja_t1_brluc_ha",
    "area_soja_persistente_2000_2019_ha",
    "area_conversao_para_soja_2000_2019_ha",

    "percentual_conversao_para_soja_pct",
    "area_origem_natural_ha",

    "emissao_absoluta_co2_t_ano",
    "taxa_emissao_co2_t_ha_ano",

    "soc_t1_t_c_ha",
    "ctotal_t1_t_c_ha",

    "delta_soc_classes_brluc_t1_t0_t_c_ha",

    "soc_t1_incerteza_pct",
    "ctotal_t1_incerteza_pct"
]


matriz_correlacao_brluc = (
    brluc_municipal_candidatos[
        COLUNAS_BRLUC_CORRELACAO
    ]
    .corr(
        method="pearson",
        min_periods=30
    )
)


pares_correlacionados_brluc = []


for i, coluna_a in enumerate(
    COLUNAS_BRLUC_CORRELACAO
):

    for coluna_b in COLUNAS_BRLUC_CORRELACAO[
        i + 1:
    ]:

        correlacao = (
            matriz_correlacao_brluc.loc[
                coluna_a,
                coluna_b
            ]
        )


        if (
            pd.notna(
                correlacao
            )
            and
            abs(
                correlacao
            )
            >= 0.85
        ):

            pares_correlacionados_brluc.append(
                {
                    "variavel_a":
                        coluna_a,

                    "variavel_b":
                        coluna_b,

                    "correlacao":
                        correlacao,

                    "correlacao_abs":
                        abs(
                            correlacao
                        )
                }
            )


pares_correlacionados_brluc = pd.DataFrame(
    pares_correlacionados_brluc
)


if not pares_correlacionados_brluc.empty:

    pares_correlacionados_brluc = (
        pares_correlacionados_brluc
        .sort_values(
            by="correlacao_abs",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


print("=" * 70)
print("BRLUC — PARES COM |CORRELAÇÃO| >= 0.85")
print("=" * 70)


if pares_correlacionados_brluc.empty:

    print(
        "Nenhum par encontrado."
    )

else:

    display(
        pares_correlacionados_brluc
    )

BRLUC — PARES COM |CORRELAÇÃO| >= 0.85


,variavel_a,variavel_b,correlacao,correlacao_abs
0,soc_t1_t_c_ha,ctotal_t1_t_c_ha,1.000000,1.000000
1,soc_t1_incerteza_pct,ctotal_t1_incerteza_pct,0.995096,0.995096
2,area_origem_natural_ha,emissao_absoluta_co2_t_ano,0.950528,0.950528
3,area_soja_t1_brluc_ha,area_conversao_para_soja_2000_2019_ha,0.932497,0.932497


In [41]:
# ============================================================
# DECISÕES DE REDUNDÂNCIA — BRLUC
# ============================================================

decisoes_brluc = pd.DataFrame(
    [
        {
            "variavel": "area_soja_t1_brluc_ha",
            "decisao": "descritiva",
            "uso": "contexto",
            "justificativa": (
                "É decomposta matematicamente em área persistente "
                "+ área de conversão."
            )
        },

        {
            "variavel": "area_soja_persistente_2000_2019_ha",
            "decisao": "descritiva",
            "uso": "contexto",
            "justificativa": (
                "Complementa a área convertida dentro da área soja t1."
            )
        },

        {
            "variavel": "area_conversao_para_soja_2000_2019_ha",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa a escala histórica de conversão "
                "associada à soja."
            )
        },

        {
            "variavel": "percentual_persistencia_soja_pct",
            "decisao": "remover_do_modelo",
            "uso": "redundante",
            "justificativa": (
                "É complementar ao percentual de conversão; "
                "a soma dos dois é 100% quando definida."
            )
        },

        {
            "variavel": "percentual_conversao_para_soja_pct",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa a proporção relativa de conversão "
                "associada à soja."
            )
        },

        {
            "variavel": "area_origem_natural_ha",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa a escala da área de origem natural "
                "associada à transição para soja."
            )
        },

        {
            "variavel": "emissao_absoluta_co2_t_ano",
            "decisao": "descritiva",
            "uso": "contexto_emissoes",
            "justificativa": (
                "Fortemente correlacionada com área de origem natural; "
                "evitar peso duplo."
            )
        },

        {
            "variavel": "taxa_emissao_co2_t_ha_ano",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa intensidade do balanço de CO2 BRLUC "
                "por área de referência."
            )
        },

        {
            "variavel": "soc_t1_t_c_ha",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa SOC da classe BRLUC associada a t1."
            )
        },

        {
            "variavel": "ctotal_t1_t_c_ha",
            "decisao": "remover_do_modelo",
            "uso": "redundante",
            "justificativa": (
                "Correlação perfeita com SOC t1 porque Cveg "
                "é constante no recorte."
            )
        },

        {
            "variavel": "delta_soc_classes_brluc_t1_t0_t_c_ha",
            "decisao": "manter",
            "uso": "candidata_priorizacao",
            "justificativa": (
                "Representa diferença entre as estimativas SOC "
                "das classes associadas a t1 e t0."
            )
        },

        {
            "variavel": "delta_ctotal_classes_brluc_t1_t0_t_c_ha",
            "decisao": "remover_do_modelo",
            "uso": "redundante",
            "justificativa": (
                "É matematicamente equivalente ao Delta SOC "
                "neste recorte."
            )
        },

        {
            "variavel": "soc_t1_incerteza_pct",
            "decisao": "manter",
            "uso": "qualidade",
            "justificativa": (
                "Indicador de incerteza do dado; não representa "
                "desempenho ambiental."
            )
        },

        {
            "variavel": "ctotal_t1_incerteza_pct",
            "decisao": "remover_do_modelo",
            "uso": "redundante_qualidade",
            "justificativa": (
                "Altamente correlacionada com a incerteza SOC t1."
            )
        }
    ]
)


print("=" * 70)
print("DECISÕES FINAIS — BRLUC")
print("=" * 70)

display(
    decisoes_brluc
)

DECISÕES FINAIS — BRLUC


,variavel,decisao,uso,justificativa
0,area_soja_t1_brluc_ha,descritiva,contexto,É decomposta matematicamente em área persisten...
1,area_soja_persistente_2000_2019_ha,descritiva,contexto,Complementa a área convertida dentro da área s...
2,area_conversao_para_soja_2000_2019_ha,manter,candidata_priorizacao,Representa a escala histórica de conversão ass...
3,percentual_persistencia_soja_pct,remover_do_modelo,redundante,É complementar ao percentual de conversão; a s...
4,percentual_conversao_para_soja_pct,manter,candidata_priorizacao,Representa a proporção relativa de conversão a...
5,area_origem_natural_ha,manter,candidata_priorizacao,Representa a escala da área de origem natural ...
6,emissao_absoluta_co2_t_ano,descritiva,contexto_emissoes,Fortemente correlacionada com área de origem n...
7,taxa_emissao_co2_t_ha_ano,manter,candidata_priorizacao,Representa intensidade do balanço de CO2 BRLUC...
8,soc_t1_t_c_ha,manter,candidata_priorizacao,Representa SOC da classe BRLUC associada a t1.
9,ctotal_t1_t_c_ha,remover_do_modelo,redundante,Correlação perfeita com SOC t1 porque Cveg é c...


In [42]:
# ============================================================
# BRLUC MUNICIPAL REDUZIDO
# ============================================================

COLUNAS_BRLUC_REDUZIDAS = [
    "codigo_ibge",

    # --------------------------------------------------------
    # Referência temporal
    # --------------------------------------------------------
    "periodo_inicio_brluc",
    "periodo_fim_brluc",

    # --------------------------------------------------------
    # Conversão histórica
    # --------------------------------------------------------
    "area_conversao_para_soja_2000_2019_ha",
    "percentual_conversao_para_soja_pct",
    "area_origem_natural_ha",

    # --------------------------------------------------------
    # Emissões BRLUC
    # --------------------------------------------------------
    # Absoluta = contexto
    "emissao_absoluta_co2_t_ano",

    # Taxa = candidata analítica
    "taxa_emissao_co2_t_ha_ano",

    # --------------------------------------------------------
    # Carbono BRLUC
    # --------------------------------------------------------
    "soc_t1_t_c_ha",
    "delta_soc_classes_brluc_t1_t0_t_c_ha",

    # --------------------------------------------------------
    # Qualidade / incerteza
    # --------------------------------------------------------
    "soc_t1_incerteza_pct",

    "ic95_emissao_absoluta_disponivel",
    "ic95_taxa_emissao_disponivel"
]


brluc_municipal_reduzido = (
    brluc_municipal_candidatos[
        COLUNAS_BRLUC_REDUZIDAS
    ]
    .copy()
)


print("=" * 70)
print("BRLUC MUNICIPAL REDUZIDO")
print("=" * 70)


print(
    "\nDimensão:",
    brluc_municipal_reduzido.shape
)


print(
    "Municípios:",
    brluc_municipal_reduzido[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "Duplicatas:",
    brluc_municipal_reduzido
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nNULL por coluna:"
)


display(
    brluc_municipal_reduzido
    .isna()
    .sum()
    .loc[
        lambda serie:
            serie > 0
    ]
    .sort_values(
        ascending=False
    )
)

BRLUC MUNICIPAL REDUZIDO

Dimensão: (1505, 13)
Municípios: 1505
Duplicatas: 0

NULL por coluna:


percentual_conversao_para_soja_pct    27
dtype: int64

In [43]:
# ============================================================
# MERGE MUNICIPAL DEFINITIVO — TEMPORAL + BRLUC
# ============================================================

base_municipal_integrada = (
    base_municipal_reduzida
    .merge(
        brluc_municipal_reduzido,
        on="codigo_ibge",
        how="left",
        validate="one_to_one",
        indicator=True
    )
)


print("=" * 70)
print("MERGE MUNICIPAL — BASE TEMPORAL + BRLUC")
print("=" * 70)


print(
    "\nDimensão antes de remover _merge:"
)

print(
    base_municipal_integrada.shape
)


print(
    "\nStatus do merge:"
)

display(
    base_municipal_integrada[
        "_merge"
    ]
    .value_counts()
)


print(
    "\nMunicípios:"
)

print(
    base_municipal_integrada[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas codigo_ibge:"
)

print(
    base_municipal_integrada
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


# ------------------------------------------------------------
# Remover indicador após validar
# ------------------------------------------------------------

base_municipal_integrada = (
    base_municipal_integrada
    .drop(
        columns=[
            "_merge"
        ]
    )
)


print(
    "\nDimensão final:"
)

print(
    base_municipal_integrada.shape
)

MERGE MUNICIPAL — BASE TEMPORAL + BRLUC

Dimensão antes de remover _merge:
(1505, 48)

Status do merge:


_merge
both          1505
left_only        0
right_only       0
Name: count, dtype: int64


Municípios:
1505

Duplicatas codigo_ibge:
0

Dimensão final:
(1505, 47)


In [44]:
# ============================================================
# AUDITORIA CRUZADA ENTRE FONTES
# ============================================================

VARIAVEIS_AUDITORIA_CRUZADA = [
    # --------------------------------------------------------
    # Produção / eficiência
    # --------------------------------------------------------
    "producao_media_t",
    "rendimento_medio_kg_ha",
    "rendimento_cv_pct",

    # --------------------------------------------------------
    # Clima
    # --------------------------------------------------------
    "precipitacao_media_mm",
    "precipitacao_cv_pct",
    "temperatura_media_c",
    "temperatura_desvio_padrao_c",
    "umidade_media_pct",
    "umidade_desvio_padrao_pct",

    # --------------------------------------------------------
    # MapBiomas Solo
    # --------------------------------------------------------
    "carbono_solo_2024_t_ha",
    "carbono_solo_variacao_2019_2024_t_ha",

    # --------------------------------------------------------
    # MapBiomas Cobertura
    # --------------------------------------------------------
    "cobertura_natural_2024_pct",
    "cobertura_natural_variacao_2019_2024_pp",
    "soja_mapbiomas_2024_pct",

    # --------------------------------------------------------
    # BRLUC
    # --------------------------------------------------------
    "area_conversao_para_soja_2000_2019_ha",
    "percentual_conversao_para_soja_pct",
    "area_origem_natural_ha",

    "taxa_emissao_co2_t_ha_ano",

    "soc_t1_t_c_ha",
    "delta_soc_classes_brluc_t1_t0_t_c_ha",

    "soc_t1_incerteza_pct"
]


correlacao_pearson_cruzada = (
    base_municipal_integrada[
        VARIAVEIS_AUDITORIA_CRUZADA
    ]
    .corr(
        method="pearson",
        min_periods=30
    )
)


correlacao_spearman_cruzada = (
    base_municipal_integrada[
        VARIAVEIS_AUDITORIA_CRUZADA
    ]
    .corr(
        method="spearman",
        min_periods=30
    )
)


print("=" * 70)
print("AUDITORIA CRUZADA ENTRE FONTES")
print("=" * 70)


print(
    "\nVariáveis analisadas:",
    len(
        VARIAVEIS_AUDITORIA_CRUZADA
    )
)

AUDITORIA CRUZADA ENTRE FONTES

Variáveis analisadas: 21


In [45]:
# ============================================================
# PARES FORTEMENTE ASSOCIADOS ENTRE FONTES
# ============================================================

pares_cruzados = []


for i, coluna_a in enumerate(
    VARIAVEIS_AUDITORIA_CRUZADA
):

    for coluna_b in VARIAVEIS_AUDITORIA_CRUZADA[
        i + 1:
    ]:

        pearson = (
            correlacao_pearson_cruzada
            .loc[
                coluna_a,
                coluna_b
            ]
        )


        spearman = (
            correlacao_spearman_cruzada
            .loc[
                coluna_a,
                coluna_b
            ]
        )


        maior_associacao = max(
            abs(pearson)
            if pd.notna(pearson)
            else 0,

            abs(spearman)
            if pd.notna(spearman)
            else 0
        )


        if maior_associacao >= 0.85:

            pares_cruzados.append(
                {
                    "variavel_a":
                        coluna_a,

                    "variavel_b":
                        coluna_b,

                    "pearson":
                        pearson,

                    "spearman":
                        spearman,

                    "maior_correlacao_abs":
                        maior_associacao
                }
            )


pares_cruzados = pd.DataFrame(
    pares_cruzados
)


if not pares_cruzados.empty:

    pares_cruzados = (
        pares_cruzados
        .sort_values(
            by="maior_correlacao_abs",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


print("=" * 70)
print("PARES CRUZADOS COM |CORRELAÇÃO| >= 0.85")
print("=" * 70)


if pares_cruzados.empty:

    print(
        "Nenhuma redundância forte encontrada."
    )

else:

    display(
        pares_cruzados
    )

PARES CRUZADOS COM |CORRELAÇÃO| >= 0.85


,variavel_a,variavel_b,pearson,spearman,maior_correlacao_abs
0,producao_media_t,area_conversao_para_soja_2000_2019_ha,0.930257,0.951221,0.951221


In [46]:
# ============================================================
# COBERTURA DOS CANDIDATOS AO MODELO
# ============================================================

auditoria_candidatos_finais = []


for coluna in VARIAVEIS_AUDITORIA_CRUZADA:

    serie = (
        base_municipal_integrada[
            coluna
        ]
    )


    auditoria_candidatos_finais.append(
        {
            "variavel":
                coluna,

            "municipios_com_dado":
                serie
                .notna()
                .sum(),

            "municipios_sem_dado":
                serie
                .isna()
                .sum(),

            "cobertura_pct":
                (
                    serie
                    .notna()
                    .mean()
                    *
                    100
                ),

            "negativos":
                (
                    serie
                    < 0
                )
                .sum(),

            "zeros":
                (
                    serie
                    == 0
                )
                .sum()
        }
    )


auditoria_candidatos_finais = (
    pd.DataFrame(
        auditoria_candidatos_finais
    )
    .sort_values(
        by=[
            "cobertura_pct",
            "variavel"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 70)
print("COBERTURA DOS CANDIDATOS AO MODELO")
print("=" * 70)


display(
    auditoria_candidatos_finais
)

COBERTURA DOS CANDIDATOS AO MODELO


,variavel,municipios_com_dado,municipios_sem_dado,cobertura_pct,negativos,zeros
0,carbono_solo_variacao_2019_2024_t_ha,1406,99,93.421927,1343,0
1,cobertura_natural_variacao_2019_2024_pp,1406,99,93.421927,980,0
2,percentual_conversao_para_soja_pct,1478,27,98.205980,0,3
3,precipitacao_cv_pct,1481,24,98.405316,0,0
4,rendimento_cv_pct,1481,24,98.405316,0,8
5,temperatura_desvio_padrao_c,1481,24,98.405316,0,0
6,umidade_desvio_padrao_pct,1481,24,98.405316,0,0
7,carbono_solo_2024_t_ha,1482,23,98.471761,0,0
8,cobertura_natural_2024_pct,1482,23,98.471761,0,0
9,soja_mapbiomas_2024_pct,1482,23,98.471761,0,9


In [47]:
# ============================================================
# CATÁLOGO FINAL DOS INDICADORES MUNICIPAIS
# ============================================================

catalogo_indicadores = pd.DataFrame(
    [
        # ====================================================
        # 1. ESCALA PRODUTIVA
        # ====================================================
        {
            "dimensao": "Escala produtiva",
            "variavel": "producao_media_t",
            "papel": "score",
            "interpretacao": "Escala média da produção de soja no período disponível",
            "direcao_preliminar": "maior = maior escala/oportunidade"
        },

        # ====================================================
        # 2. EFICIÊNCIA / ESTABILIDADE PRODUTIVA
        # ====================================================
        {
            "dimensao": "Eficiência produtiva",
            "variavel": "rendimento_medio_kg_ha",
            "papel": "score",
            "interpretacao": "Produtividade média da soja",
            "direcao_preliminar": "maior = maior eficiência produtiva"
        },

        {
            "dimensao": "Estabilidade produtiva",
            "variavel": "rendimento_cv_pct",
            "papel": "score",
            "interpretacao": "Variabilidade interanual do rendimento",
            "direcao_preliminar": "maior = maior instabilidade"
        },

        # ====================================================
        # 3. VARIABILIDADE CLIMÁTICA
        # ====================================================
        {
            "dimensao": "Variabilidade climática",
            "variavel": "precipitacao_cv_pct",
            "papel": "score",
            "interpretacao": "Variabilidade interanual da precipitação",
            "direcao_preliminar": "maior = maior variabilidade"
        },

        {
            "dimensao": "Variabilidade climática",
            "variavel": "temperatura_desvio_padrao_c",
            "papel": "score",
            "interpretacao": "Dispersão interanual da temperatura média",
            "direcao_preliminar": "maior = maior variabilidade"
        },

        {
            "dimensao": "Variabilidade climática",
            "variavel": "umidade_desvio_padrao_pct",
            "papel": "contexto",
            "interpretacao": "Dispersão interanual da umidade média",
            "direcao_preliminar": "contextual"
        },

        # ====================================================
        # 4. CARBONO DO SOLO
        # ====================================================
        {
            "dimensao": "Carbono do solo",
            "variavel": "carbono_solo_2024_t_ha",
            "papel": "score",
            "interpretacao": "Estoque de carbono do solo no dado mais recente",
            "direcao_preliminar": "a definir conforme objetivo do índice"
        },

        {
            "dimensao": "Carbono do solo",
            "variavel": "carbono_solo_variacao_2019_2024_t_ha",
            "papel": "score",
            "interpretacao": "Mudança observada entre 2019 e 2024",
            "direcao_preliminar": "mais negativo = maior perda observada"
        },

        # ====================================================
        # 5. COBERTURA / USO DA TERRA
        # ====================================================
        {
            "dimensao": "Cobertura natural",
            "variavel": "cobertura_natural_2024_pct",
            "papel": "score",
            "interpretacao": "Percentual atual de cobertura natural",
            "direcao_preliminar": "a definir conforme objetivo do índice"
        },

        {
            "dimensao": "Cobertura natural",
            "variavel": "cobertura_natural_variacao_2019_2024_pp",
            "papel": "score",
            "interpretacao": "Mudança da cobertura natural em pontos percentuais",
            "direcao_preliminar": "mais negativo = maior perda observada"
        },

        {
            "dimensao": "Presença da soja",
            "variavel": "soja_mapbiomas_2024_pct",
            "papel": "score",
            "interpretacao": "Participação da soja na área mapeada do município",
            "direcao_preliminar": "maior = maior presença relativa da soja"
        },

        # ====================================================
        # 6. HISTÓRICO BRLUC
        # ====================================================
        {
            "dimensao": "Histórico de conversão",
            "variavel": "percentual_conversao_para_soja_pct",
            "papel": "score",
            "interpretacao": "Participação relativa da conversão associada à soja",
            "direcao_preliminar": "maior = maior intensidade histórica de conversão"
        },

        {
            "dimensao": "Histórico de conversão",
            "variavel": "area_conversao_para_soja_2000_2019_ha",
            "papel": "contexto",
            "interpretacao": "Escala absoluta da conversão histórica",
            "direcao_preliminar": "contexto; redundante com escala produtiva"
        },

        {
            "dimensao": "Origem natural BRLUC",
            "variavel": "area_origem_natural_ha",
            "papel": "contexto",
            "interpretacao": "Área de origem natural associada à transição",
            "direcao_preliminar": "contexto"
        },

        {
            "dimensao": "Balanço BRLUC",
            "variavel": "taxa_emissao_co2_t_ha_ano",
            "papel": "score",
            "interpretacao": "Taxa de emissão/balanço de CO2 do BRLUC",
            "direcao_preliminar": "maior positivo = maior emissão estimada"
        },

        {
            "dimensao": "Carbono BRLUC",
            "variavel": "soc_t1_t_c_ha",
            "papel": "contexto",
            "interpretacao": "SOC da classe BRLUC associada a t1",
            "direcao_preliminar": "contextual"
        },

        {
            "dimensao": "Carbono BRLUC",
            "variavel": "delta_soc_classes_brluc_t1_t0_t_c_ha",
            "papel": "contexto",
            "interpretacao": "Diferença entre estimativas SOC das classes t1 e t0",
            "direcao_preliminar": "não interpretar como sequestro/crédito"
        },

        # ====================================================
        # QUALIDADE
        # ====================================================
        {
            "dimensao": "Qualidade do dado",
            "variavel": "quantidade_anos",
            "papel": "qualidade",
            "interpretacao": "Quantidade de anos disponíveis na base analítica",
            "direcao_preliminar": "maior = maior cobertura temporal"
        },

        {
            "dimensao": "Qualidade do dado",
            "variavel": "soc_t1_incerteza_pct",
            "papel": "qualidade",
            "interpretacao": "Incerteza relativa do SOC BRLUC",
            "direcao_preliminar": "menor = maior precisão"
        }
    ]
)


print("=" * 70)
print("CATÁLOGO FINAL DOS INDICADORES")
print("=" * 70)


display(
    catalogo_indicadores
)


print(
    "\nQuantidade por papel:"
)

display(
    catalogo_indicadores[
        "papel"
    ]
    .value_counts()
)

CATÁLOGO FINAL DOS INDICADORES


,dimensao,variavel,papel,interpretacao,direcao_preliminar
0,Escala produtiva,producao_media_t,score,Escala média da produção de soja no período di...,maior = maior escala/oportunidade
1,Eficiência produtiva,rendimento_medio_kg_ha,score,Produtividade média da soja,maior = maior eficiência produtiva
2,Estabilidade produtiva,rendimento_cv_pct,score,Variabilidade interanual do rendimento,maior = maior instabilidade
3,Variabilidade climática,precipitacao_cv_pct,score,Variabilidade interanual da precipitação,maior = maior variabilidade
4,Variabilidade climática,temperatura_desvio_padrao_c,score,Dispersão interanual da temperatura média,maior = maior variabilidade
5,Variabilidade climática,umidade_desvio_padrao_pct,contexto,Dispersão interanual da umidade média,contextual
6,Carbono do solo,carbono_solo_2024_t_ha,score,Estoque de carbono do solo no dado mais recente,a definir conforme objetivo do índice
7,Carbono do solo,carbono_solo_variacao_2019_2024_t_ha,score,Mudança observada entre 2019 e 2024,mais negativo = maior perda observada
8,Cobertura natural,cobertura_natural_2024_pct,score,Percentual atual de cobertura natural,a definir conforme objetivo do índice
9,Cobertura natural,cobertura_natural_variacao_2019_2024_pp,score,Mudança da cobertura natural em pontos percent...,mais negativo = maior perda observada



Quantidade por papel:


papel
score        12
contexto      5
qualidade     2
Name: count, dtype: int64

In [48]:
# ============================================================
# DISTRIBUIÇÃO DOS INDICADORES CANDIDATOS AO SCORE
# ============================================================

INDICADORES_SCORE = (
    catalogo_indicadores.loc[
        catalogo_indicadores[
            "papel"
        ]
        .eq("score"),
        "variavel"
    ]
    .tolist()
)


distribuicao_score = []


for coluna in INDICADORES_SCORE:

    serie = (
        base_municipal_integrada[
            coluna
        ]
        .dropna()
    )


    distribuicao_score.append(
        {
            "variavel":
                coluna,

            "n":
                len(serie),

            "min":
                serie.min(),

            "p01":
                serie.quantile(0.01),

            "p05":
                serie.quantile(0.05),

            "p25":
                serie.quantile(0.25),

            "mediana":
                serie.median(),

            "p75":
                serie.quantile(0.75),

            "p95":
                serie.quantile(0.95),

            "p99":
                serie.quantile(0.99),

            "max":
                serie.max(),

            "assimetria":
                serie.skew()
        }
    )


distribuicao_score = pd.DataFrame(
    distribuicao_score
)


print("=" * 70)
print("DISTRIBUIÇÃO DOS CANDIDATOS AO SCORE")
print("=" * 70)


display(
    distribuicao_score
)

DISTRIBUIÇÃO DOS CANDIDATOS AO SCORE


,variavel,n,min,p01,p05,p25,mediana,p75,p95,p99,max,assimetria
0,producao_media_t,1504,2.000000,38.135000,359.916667,4044.750000,18770.500000,59464.583333,261030.833333,912718.540000,2.147258e+06,5.971181
1,rendimento_medio_kg_ha,1504,1440.666667,1733.005000,2053.008333,2631.791667,3073.000000,3425.000000,3705.283333,3895.405000,4.200000e+03,-0.479513
2,rendimento_cv_pct,1481,0.000000,2.170323,4.662138,9.047782,20.512074,33.585587,50.429979,62.648572,8.008034e+01,0.665723
3,precipitacao_cv_pct,1481,0.403406,7.051144,12.923546,18.445716,22.919226,27.189328,32.013519,35.775330,4.873824e+01,-0.136560
4,temperatura_desvio_padrao_c,1481,0.009409,0.224738,0.305041,0.435899,0.560502,0.786243,1.261242,1.749011,2.380419e+00,1.535334
5,carbono_solo_2024_t_ha,1482,17.103862,32.292041,34.635313,43.788533,53.614047,58.445874,73.322436,93.525966,1.043971e+02,0.735208
6,carbono_solo_variacao_2019_2024_t_ha,1406,-4.114098,-2.415493,-1.584115,-0.795680,-0.497744,-0.295249,-0.019644,0.203510,3.612375e-01,-1.909859
7,cobertura_natural_2024_pct,1482,3.347426,4.833010,7.516771,16.976448,27.157471,40.511947,64.526483,83.251524,9.212206e+01,0.881711
8,cobertura_natural_variacao_2019_2024_pp,1406,-12.655143,-8.280899,-4.752714,-1.456360,-0.440380,0.116894,0.799022,1.349975,4.362832e+00,-2.226249
9,soja_mapbiomas_2024_pct,1482,0.000000,0.002259,0.269480,3.724532,14.057700,38.101336,70.188396,79.124588,8.553260e+01,0.899524


In [49]:
# ============================================================
# DISTRIBUIÇÃO DOS SINAIS — MUDANÇAS AMBIENTAIS
# ============================================================

VARIAVEIS_MUDANCA_AMBIENTAL = [
    "carbono_solo_variacao_2019_2024_t_ha",
    "cobertura_natural_variacao_2019_2024_pp"
]


resultado_sinais = []


for coluna in VARIAVEIS_MUDANCA_AMBIENTAL:

    serie = (
        base_municipal_integrada[
            coluna
        ]
        .dropna()
    )


    negativos = (
        serie < 0
    ).sum()

    zeros = (
        serie == 0
    ).sum()

    positivos = (
        serie > 0
    ).sum()


    resultado_sinais.append(
        {
            "variavel":
                coluna,

            "n":
                len(serie),

            "negativos":
                negativos,

            "negativos_pct":
                negativos
                /
                len(serie)
                *
                100,

            "zeros":
                zeros,

            "zeros_pct":
                zeros
                /
                len(serie)
                *
                100,

            "positivos":
                positivos,

            "positivos_pct":
                positivos
                /
                len(serie)
                *
                100,

            "mediana":
                serie.median(),

            "media":
                serie.mean(),

            "minimo":
                serie.min(),

            "maximo":
                serie.max()
        }
    )


resultado_sinais = pd.DataFrame(
    resultado_sinais
)


print("=" * 70)
print("SINAIS DAS MUDANÇAS AMBIENTAIS")
print("=" * 70)


display(
    resultado_sinais
)

SINAIS DAS MUDANÇAS AMBIENTAIS


,variavel,n,negativos,negativos_pct,zeros,zeros_pct,positivos,positivos_pct,mediana,media,minimo,maximo
0,carbono_solo_variacao_2019_2024_t_ha,1406,1343,95.519203,0,0.0,63,4.480797,-0.497744,-0.605826,-4.114098,0.361238
1,cobertura_natural_variacao_2019_2024_pp,1406,980,69.701280,0,0.0,426,30.298720,-0.440380,-0.988107,-12.655143,4.362832


In [50]:
# ============================================================
# ARQUITETURA DO MODELO DE PRIORIZAÇÃO
# ============================================================

INDICADORES_RELEVANCIA_PRODUTIVA = {
    # maior = maior relevância
    "producao_media_t": "maior",
    "rendimento_medio_kg_ha": "maior",
    "soja_mapbiomas_2024_pct": "maior"
}


INDICADORES_PRESSAO_AGROAMBIENTAL = {
    # --------------------------------------------------------
    # Instabilidade produtiva
    # --------------------------------------------------------
    "rendimento_cv_pct": "maior",

    # --------------------------------------------------------
    # Variabilidade climática
    # --------------------------------------------------------
    "precipitacao_cv_pct": "maior",
    "temperatura_desvio_padrao_c": "maior",

    # --------------------------------------------------------
    # Mudanças ambientais
    #
    # Quanto MAIS NEGATIVO, maior a pressão/perda observada.
    # --------------------------------------------------------
    "carbono_solo_variacao_2019_2024_t_ha": "menor",
    "cobertura_natural_variacao_2019_2024_pp": "menor",

    # --------------------------------------------------------
    # Histórico BRLUC
    # --------------------------------------------------------
    "percentual_conversao_para_soja_pct": "maior",
    "taxa_emissao_co2_t_ha_ano": "maior"
}


INDICADORES_ATIVOS_AMBIENTAIS = [
    "carbono_solo_2024_t_ha",
    "cobertura_natural_2024_pct",

    # contexto BRLUC
    "soc_t1_t_c_ha",
    "delta_soc_classes_brluc_t1_t0_t_c_ha"
]


INDICADORES_QUALIDADE = [
    "quantidade_anos",
    "soc_t1_incerteza_pct"
]


print("=" * 70)
print("ARQUITETURA FINAL DO MODELO")
print("=" * 70)


print(
    "\nRelevância produtiva:",
    len(
        INDICADORES_RELEVANCIA_PRODUTIVA
    )
)


for variavel, direcao in INDICADORES_RELEVANCIA_PRODUTIVA.items():

    print(
        "->",
        variavel,
        "| direção:",
        direcao
    )


print(
    "\nPressão / vulnerabilidade agroambiental:",
    len(
        INDICADORES_PRESSAO_AGROAMBIENTAL
    )
)


for variavel, direcao in INDICADORES_PRESSAO_AGROAMBIENTAL.items():

    print(
        "->",
        variavel,
        "| direção:",
        direcao
    )


print(
    "\nAtivos ambientais / contexto:",
    len(
        INDICADORES_ATIVOS_AMBIENTAIS
    )
)


for variavel in INDICADORES_ATIVOS_AMBIENTAIS:

    print(
        "->",
        variavel
    )


print(
    "\nQualidade:",
    len(
        INDICADORES_QUALIDADE
    )
)


for variavel in INDICADORES_QUALIDADE:

    print(
        "->",
        variavel
    )

ARQUITETURA FINAL DO MODELO

Relevância produtiva: 3
-> producao_media_t | direção: maior
-> rendimento_medio_kg_ha | direção: maior
-> soja_mapbiomas_2024_pct | direção: maior

Pressão / vulnerabilidade agroambiental: 7
-> rendimento_cv_pct | direção: maior
-> precipitacao_cv_pct | direção: maior
-> temperatura_desvio_padrao_c | direção: maior
-> carbono_solo_variacao_2019_2024_t_ha | direção: menor
-> cobertura_natural_variacao_2019_2024_pp | direção: menor
-> percentual_conversao_para_soja_pct | direção: maior
-> taxa_emissao_co2_t_ha_ano | direção: maior

Ativos ambientais / contexto: 4
-> carbono_solo_2024_t_ha
-> cobertura_natural_2024_pct
-> soc_t1_t_c_ha
-> delta_soc_classes_brluc_t1_t0_t_c_ha

Qualidade: 2
-> quantidade_anos
-> soc_t1_incerteza_pct


In [51]:
# ============================================================
# ELEGIBILIDADE DOS EIXOS
# ============================================================

COLUNAS_RELEVANCIA = list(
    INDICADORES_RELEVANCIA_PRODUTIVA.keys()
)


COLUNAS_PRESSAO = list(
    INDICADORES_PRESSAO_AGROAMBIENTAL.keys()
)


base_municipal_integrada[
    "elegivel_relevancia_produtiva"
] = (
    base_municipal_integrada[
        COLUNAS_RELEVANCIA
    ]
    .notna()
    .all(
        axis=1
    )
)


base_municipal_integrada[
    "elegivel_pressao_agroambiental"
] = (
    base_municipal_integrada[
        COLUNAS_PRESSAO
    ]
    .notna()
    .all(
        axis=1
    )
)


base_municipal_integrada[
    "elegivel_cruzamento_priorizacao"
] = (
    base_municipal_integrada[
        "elegivel_relevancia_produtiva"
    ]
    &
    base_municipal_integrada[
        "elegivel_pressao_agroambiental"
    ]
)


print("=" * 70)
print("ELEGIBILIDADE DOS EIXOS")
print("=" * 70)


for coluna in [
    "elegivel_relevancia_produtiva",
    "elegivel_pressao_agroambiental",
    "elegivel_cruzamento_priorizacao"
]:

    print(
        f"\n{coluna}"
    )

    display(
        base_municipal_integrada[
            coluna
        ]
        .value_counts()
    )


print(
    "\nPercentual elegível para o cruzamento:"
)


print(
    (
        base_municipal_integrada[
            "elegivel_cruzamento_priorizacao"
        ]
        .mean()
        *
        100
    )
)

ELEGIBILIDADE DOS EIXOS

elegivel_relevancia_produtiva


elegivel_relevancia_produtiva
True     1481
False      24
Name: count, dtype: int64


elegivel_pressao_agroambiental


elegivel_pressao_agroambiental
True     1400
False     105
Name: count, dtype: int64


elegivel_cruzamento_priorizacao


elegivel_cruzamento_priorizacao
True     1400
False     105
Name: count, dtype: int64


Percentual elegível para o cruzamento:
93.02325581395348


In [52]:
# ============================================================
# NORMALIZAÇÃO POR RANK PERCENTIL
# ============================================================

base_modelo = (
    base_municipal_integrada
    .copy()
)


def normalizar_percentil(
    serie,
    direcao
):

    if direcao == "maior":

        # Maior valor original
        # -> maior score normalizado
        resultado = (
            serie
            .rank(
                method="average",
                pct=True,
                ascending=True
            )
            *
            100
        )

    elif direcao == "menor":

        # Menor valor original
        # -> maior score normalizado
        resultado = (
            serie
            .rank(
                method="average",
                pct=True,
                ascending=False
            )
            *
            100
        )

    else:

        raise ValueError(
            "Direção deve ser 'maior' ou 'menor'."
        )


    return resultado


# ============================================================
# RELEVÂNCIA PRODUTIVA
# ============================================================

for variavel, direcao in INDICADORES_RELEVANCIA_PRODUTIVA.items():

    nome_normalizado = (
        f"norm_relevancia__{variavel}"
    )


    base_modelo[
        nome_normalizado
    ] = normalizar_percentil(
        base_modelo[
            variavel
        ],
        direcao
    )


# ============================================================
# PRESSÃO AGROAMBIENTAL
# ============================================================

for variavel, direcao in INDICADORES_PRESSAO_AGROAMBIENTAL.items():

    nome_normalizado = (
        f"norm_pressao__{variavel}"
    )


    base_modelo[
        nome_normalizado
    ] = normalizar_percentil(
        base_modelo[
            variavel
        ],
        direcao
    )


print("=" * 70)
print("NORMALIZAÇÃO POR PERCENTIL")
print("=" * 70)


COLUNAS_NORMALIZADAS = [
    coluna
    for coluna in base_modelo.columns
    if coluna.startswith(
        "norm_"
    )
]


print(
    "\nIndicadores normalizados:",
    len(
        COLUNAS_NORMALIZADAS
    )
)


controle_normalizacao = []


for coluna in COLUNAS_NORMALIZADAS:

    serie = (
        base_modelo[
            coluna
        ]
        .dropna()
    )


    controle_normalizacao.append(
        {
            "variavel":
                coluna,

            "n":
                len(
                    serie
                ),

            "min":
                serie.min(),

            "mediana":
                serie.median(),

            "max":
                serie.max()
        }
    )


controle_normalizacao = pd.DataFrame(
    controle_normalizacao
)


display(
    controle_normalizacao
)

NORMALIZAÇÃO POR PERCENTIL

Indicadores normalizados: 10


,variavel,n,min,mediana,max
0,norm_relevancia__producao_media_t,1504,0.066489,50.033245,100.000000
1,norm_relevancia__rendimento_medio_kg_ha,1504,0.066489,50.033245,100.000000
2,norm_relevancia__soja_mapbiomas_2024_pct,1482,0.337382,50.033738,100.000000
3,norm_pressao__rendimento_cv_pct,1481,0.303849,50.033761,100.000000
4,norm_pressao__precipitacao_cv_pct,1481,0.067522,50.033761,100.000000
5,norm_pressao__temperatura_desvio_padrao_c,1481,0.067522,50.033761,100.000000
6,norm_pressao__carbono_solo_variacao_2019_2024_...,1406,0.071124,50.035562,100.000000
7,norm_pressao__cobertura_natural_variacao_2019_...,1406,0.071124,50.035562,100.000000
8,norm_pressao__percentual_conversao_para_soja_pct,1478,0.135318,50.033829,94.925575
9,norm_pressao__taxa_emissao_co2_t_ha_ano,1505,0.066445,50.033223,100.000000


In [53]:
# ============================================================
# NORMALIZAÇÃO DEFINITIVA — UNIVERSO COMUM DO MODELO
# ============================================================

base_modelo = (
    base_municipal_integrada
    .copy()
)


mascara_modelo = (
    base_modelo[
        "elegivel_cruzamento_priorizacao"
    ]
)


print("=" * 70)
print("UNIVERSO DO MODELO")
print("=" * 70)

print(
    "\nMunicípios elegíveis:",
    mascara_modelo.sum()
)


# ------------------------------------------------------------
# Normalização robusta por posição relativa
# dentro do MESMO universo de 1.400 municípios
# ------------------------------------------------------------

def normalizar_rank_0_100(
    serie,
    direcao
):

    resultado = pd.Series(
        np.nan,
        index=serie.index,
        dtype="float64"
    )


    serie_valida = (
        serie
        .dropna()
    )


    if direcao == "maior":

        ranks = (
            serie_valida
            .rank(
                method="average",
                ascending=True
            )
        )

    elif direcao == "menor":

        ranks = (
            serie_valida
            .rank(
                method="average",
                ascending=False
            )
        )

    else:

        raise ValueError(
            "Direção deve ser 'maior' ou 'menor'."
        )


    rank_min = (
        ranks.min()
    )

    rank_max = (
        ranks.max()
    )


    if rank_max != rank_min:

        normalizado = (
            (
                ranks
                -
                rank_min
            )
            /
            (
                rank_max
                -
                rank_min
            )
            *
            100
        )

    else:

        normalizado = pd.Series(
            50.0,
            index=ranks.index
        )


    resultado.loc[
        normalizado.index
    ] = (
        normalizado
    )


    return resultado

UNIVERSO DO MODELO

Municípios elegíveis: 1400


In [54]:
# ============================================================
# APLICAR NORMALIZAÇÃO NO UNIVERSO COMUM
# ============================================================

for variavel, direcao in (
    INDICADORES_RELEVANCIA_PRODUTIVA.items()
):

    coluna_saida = (
        f"score_norm_relevancia__{variavel}"
    )


    serie_modelo = (
        base_modelo
        .loc[
            mascara_modelo,
            variavel
        ]
    )


    normalizado = (
        normalizar_rank_0_100(
            serie_modelo,
            direcao
        )
    )


    base_modelo.loc[
        mascara_modelo,
        coluna_saida
    ] = normalizado


for variavel, direcao in (
    INDICADORES_PRESSAO_AGROAMBIENTAL.items()
):

    coluna_saida = (
        f"score_norm_pressao__{variavel}"
    )


    serie_modelo = (
        base_modelo
        .loc[
            mascara_modelo,
            variavel
        ]
    )


    normalizado = (
        normalizar_rank_0_100(
            serie_modelo,
            direcao
        )
    )


    base_modelo.loc[
        mascara_modelo,
        coluna_saida
    ] = normalizado

In [55]:
# ============================================================
# CONTROLE DA NORMALIZAÇÃO DEFINITIVA
# ============================================================

COLUNAS_SCORE_NORMALIZADAS = [
    coluna
    for coluna in base_modelo.columns
    if coluna.startswith(
        "score_norm_"
    )
]


controle_normalizacao_final = []


for coluna in COLUNAS_SCORE_NORMALIZADAS:

    serie = (
        base_modelo.loc[
            mascara_modelo,
            coluna
        ]
        .dropna()
    )


    controle_normalizacao_final.append(
        {
            "variavel":
                coluna,

            "n":
                len(
                    serie
                ),

            "min":
                serie.min(),

            "mediana":
                serie.median(),

            "max":
                serie.max(),

            "null_no_universo":
                (
                    base_modelo
                    .loc[
                        mascara_modelo,
                        coluna
                    ]
                    .isna()
                    .sum()
                )
        }
    )


controle_normalizacao_final = (
    pd.DataFrame(
        controle_normalizacao_final
    )
)


print("=" * 70)
print("NORMALIZAÇÃO DEFINITIVA")
print("=" * 70)


display(
    controle_normalizacao_final
)

NORMALIZAÇÃO DEFINITIVA


,variavel,n,min,mediana,max,null_no_universo
0,score_norm_relevancia__producao_media_t,1400,0.0,50.00000,100.0,0
1,score_norm_relevancia__rendimento_medio_kg_ha,1400,0.0,50.00000,100.0,0
2,score_norm_relevancia__soja_mapbiomas_2024_pct,1400,0.0,50.00000,100.0,0
3,score_norm_pressao__rendimento_cv_pct,1400,0.0,50.00000,100.0,0
4,score_norm_pressao__precipitacao_cv_pct,1400,0.0,50.00000,100.0,0
5,score_norm_pressao__temperatura_desvio_padrao_c,1400,0.0,50.00000,100.0,0
6,score_norm_pressao__carbono_solo_variacao_2019...,1400,0.0,50.00000,100.0,0
7,score_norm_pressao__cobertura_natural_variacao...,1400,0.0,50.00000,100.0,0
8,score_norm_pressao__percentual_conversao_para_...,1400,0.0,52.12528,100.0,0
9,score_norm_pressao__taxa_emissao_co2_t_ha_ano,1400,0.0,50.00000,100.0,0


In [56]:
# ============================================================
# SUBSCORES DAS DIMENSÕES
# ============================================================

# ------------------------------------------------------------
# RELEVÂNCIA PRODUTIVA
# ------------------------------------------------------------

base_modelo[
    "subscore_escala_produtiva"
] = (
    base_modelo[
        "score_norm_relevancia__producao_media_t"
    ]
)


base_modelo[
    "subscore_eficiencia_produtiva"
] = (
    base_modelo[
        "score_norm_relevancia__rendimento_medio_kg_ha"
    ]
)


base_modelo[
    "subscore_presenca_soja"
] = (
    base_modelo[
        "score_norm_relevancia__soja_mapbiomas_2024_pct"
    ]
)


# ============================================================
# PRESSÃO AGROAMBIENTAL
# ============================================================

# ------------------------------------------------------------
# 1. Instabilidade produtiva
# ------------------------------------------------------------

base_modelo[
    "subscore_instabilidade_produtiva"
] = (
    base_modelo[
        "score_norm_pressao__rendimento_cv_pct"
    ]
)


# ------------------------------------------------------------
# 2. Variabilidade climática
# ------------------------------------------------------------

base_modelo[
    "subscore_variabilidade_climatica"
] = (
    base_modelo[
        [
            "score_norm_pressao__precipitacao_cv_pct",
            "score_norm_pressao__temperatura_desvio_padrao_c"
        ]
    ]
    .mean(
        axis=1
    )
)


# ------------------------------------------------------------
# 3. Mudanças ambientais recentes
# ------------------------------------------------------------

base_modelo[
    "subscore_mudancas_ambientais"
] = (
    base_modelo[
        [
            "score_norm_pressao__carbono_solo_variacao_2019_2024_t_ha",
            "score_norm_pressao__cobertura_natural_variacao_2019_2024_pp"
        ]
    ]
    .mean(
        axis=1
    )
)


# ------------------------------------------------------------
# 4. Histórico BRLUC
# ------------------------------------------------------------

base_modelo[
    "subscore_historico_brluc"
] = (
    base_modelo[
        [
            "score_norm_pressao__percentual_conversao_para_soja_pct",
            "score_norm_pressao__taxa_emissao_co2_t_ha_ano"
        ]
    ]
    .mean(
        axis=1
    )
)

In [57]:
# ============================================================
# EIXOS PRINCIPAIS DO MODELO
# ============================================================

# ------------------------------------------------------------
# EIXO 1 — RELEVÂNCIA PRODUTIVA
#
# 1/3 escala
# 1/3 eficiência
# 1/3 presença relativa da soja
# ------------------------------------------------------------

base_modelo[
    "score_relevancia_produtiva"
] = (
    base_modelo[
        [
            "subscore_escala_produtiva",
            "subscore_eficiencia_produtiva",
            "subscore_presenca_soja"
        ]
    ]
    .mean(
        axis=1
    )
)


# ------------------------------------------------------------
# EIXO 2 — PRESSÃO AGROAMBIENTAL
#
# 25% instabilidade produtiva
# 25% variabilidade climática
# 25% mudanças ambientais
# 25% histórico BRLUC
# ------------------------------------------------------------

base_modelo[
    "score_pressao_agroambiental"
] = (
    base_modelo[
        [
            "subscore_instabilidade_produtiva",
            "subscore_variabilidade_climatica",
            "subscore_mudancas_ambientais",
            "subscore_historico_brluc"
        ]
    ]
    .mean(
        axis=1
    )
)


# ------------------------------------------------------------
# Municípios fora do universo elegível
# não recebem score.
# ------------------------------------------------------------

base_modelo.loc[
    ~mascara_modelo,
    [
        "score_relevancia_produtiva",
        "score_pressao_agroambiental"
    ]
] = np.nan


print("=" * 70)
print("EIXOS DO MODELO")
print("=" * 70)


for coluna in [
    "score_relevancia_produtiva",
    "score_pressao_agroambiental"
]:

    serie = (
        base_modelo[
            coluna
        ]
        .dropna()
    )


    print(
        f"\n{coluna}"
    )

    print(
        "N:",
        len(
            serie
        )
    )

    print(
        "Mínimo:",
        serie.min()
    )

    print(
        "Mediana:",
        serie.median()
    )

    print(
        "Média:",
        serie.mean()
    )

    print(
        "Máximo:",
        serie.max()
    )


print(
    "\nCorrelação entre os dois eixos:"
)


print(
    base_modelo[
        [
            "score_relevancia_produtiva",
            "score_pressao_agroambiental"
        ]
    ]
    .corr()
    .iloc[
        0,
        1
    ]
)

EIXOS DO MODELO

score_relevancia_produtiva
N: 1400
Mínimo: 0.5480104836788183
Mediana: 50.428877769835594
Média: 50.0
Máximo: 93.62639980938765

score_pressao_agroambiental
N: 1400
Mínimo: 19.126787190594754
Mediana: 49.34261462459336
Média: 50.26565995525727
Máximo: 83.62223016440315

Correlação entre os dois eixos:
-0.5064545982140427


In [58]:
# ============================================================
# QUADRANTES DE PRIORIZAÇÃO
# ============================================================

LIMIAR_RELEVANCIA = (
    base_modelo.loc[
        mascara_modelo,
        "score_relevancia_produtiva"
    ]
    .median()
)


LIMIAR_PRESSAO = (
    base_modelo.loc[
        mascara_modelo,
        "score_pressao_agroambiental"
    ]
    .median()
)


print("=" * 70)
print("LIMIARES DOS QUADRANTES")
print("=" * 70)

print(
    "\nRelevância produtiva:",
    LIMIAR_RELEVANCIA
)

print(
    "Pressão agroambiental:",
    LIMIAR_PRESSAO
)


def classificar_quadrante(
    linha
):

    if not linha[
        "elegivel_cruzamento_priorizacao"
    ]:

        return "Dados insuficientes"


    relevancia_alta = (
        linha[
            "score_relevancia_produtiva"
        ]
        >=
        LIMIAR_RELEVANCIA
    )


    pressao_alta = (
        linha[
            "score_pressao_agroambiental"
        ]
        >=
        LIMIAR_PRESSAO
    )


    if (
        relevancia_alta
        and
        pressao_alta
    ):

        return (
            "Alta relevância + Alta pressão"
        )


    elif (
        relevancia_alta
        and
        not pressao_alta
    ):

        return (
            "Alta relevância + Baixa pressão"
        )


    elif (
        not relevancia_alta
        and
        pressao_alta
    ):

        return (
            "Baixa relevância + Alta pressão"
        )


    else:

        return (
            "Baixa relevância + Baixa pressão"
        )


base_modelo[
    "quadrante_priorizacao"
] = (
    base_modelo
    .apply(
        classificar_quadrante,
        axis=1
    )
)


print(
    "\nDistribuição dos quadrantes:"
)


display(
    base_modelo[
        "quadrante_priorizacao"
    ]
    .value_counts()
)

LIMIARES DOS QUADRANTES

Relevância produtiva: 50.428877769835594
Pressão agroambiental: 49.34261462459336

Distribuição dos quadrantes:


quadrante_priorizacao
Alta relevância + Baixa pressão     475
Baixa relevância + Alta pressão     475
Alta relevância + Alta pressão      225
Baixa relevância + Baixa pressão    225
Dados insuficientes                 105
Name: count, dtype: int64

In [59]:
# ============================================================
# DISTRIBUIÇÃO DOS QUADRANTES POR REGIÃO E UF
# ============================================================

print("=" * 70)
print("QUADRANTES POR REGIÃO")
print("=" * 70)


tabela_regiao_quadrante = (
    pd.crosstab(
        base_modelo[
            "regiao"
        ],
        base_modelo[
            "quadrante_priorizacao"
        ]
    )
)


display(
    tabela_regiao_quadrante
)


print(
    "\n" + "=" * 70
)

print(
    "QUADRANTES POR UF"
)

print("=" * 70)


tabela_uf_quadrante = (
    pd.crosstab(
        base_modelo[
            "uf"
        ],
        base_modelo[
            "quadrante_priorizacao"
        ]
    )
)


display(
    tabela_uf_quadrante
)

QUADRANTES POR REGIÃO


quadrante_priorizacao,Alta relevância + Alta pressão,Alta relevância + Baixa pressão,Baixa relevância + Alta pressão,Baixa relevância + Baixa pressão,Dados insuficientes
regiao,,,,,
Centro-Oeste,84,145,96,81,44
Sul,141,330,379,144,61



QUADRANTES POR UF


quadrante_priorizacao,Alta relevância + Alta pressão,Alta relevância + Baixa pressão,Baixa relevância + Alta pressão,Baixa relevância + Baixa pressão,Dados insuficientes
uf,,,,,
DF,0,1,0,0,0
GO,12,85,49,61,30
MS,13,26,22,16,1
MT,59,33,25,4,13
PR,54,217,68,41,12
RS,73,89,231,25,23
SC,14,24,80,78,26


In [60]:
# ============================================================
# MUNICÍPIOS — ALTA RELEVÂNCIA + ALTA PRESSÃO
# ============================================================

quadrante_estrategico = (
    base_modelo.loc[
        base_modelo[
            "quadrante_priorizacao"
        ]
        .eq(
            "Alta relevância + Alta pressão"
        )
    ]
    .copy()
)


print("=" * 70)
print("QUADRANTE ESTRATÉGICO")
print("=" * 70)


print(
    "\nMunicípios:",
    len(
        quadrante_estrategico
    )
)


# ------------------------------------------------------------
# Ordenação apenas exploratória:
# média dos dois eixos.
#
# Ainda NÃO é o ranking final.
# ------------------------------------------------------------

quadrante_estrategico[
    "indice_exploratorio_priorizacao"
] = (
    quadrante_estrategico[
        [
            "score_relevancia_produtiva",
            "score_pressao_agroambiental"
        ]
    ]
    .mean(
        axis=1
    )
)


colunas_inspecao = [
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",

    "score_relevancia_produtiva",
    "score_pressao_agroambiental",
    "indice_exploratorio_priorizacao",

    # Produção
    "producao_media_t",
    "rendimento_medio_kg_ha",

    # Pressão produtiva/climática
    "rendimento_cv_pct",
    "precipitacao_cv_pct",

    # Mudanças ambientais
    "carbono_solo_variacao_2019_2024_t_ha",
    "cobertura_natural_variacao_2019_2024_pp",

    # BRLUC
    "percentual_conversao_para_soja_pct",
    "taxa_emissao_co2_t_ha_ano",

    # Qualidade temporal
    "quantidade_anos"
]


display(
    quadrante_estrategico[
        colunas_inspecao
    ]
    .sort_values(
        by="indice_exploratorio_priorizacao",
        ascending=False
    )
    .head(25)
)

QUADRANTE ESTRATÉGICO

Municípios: 225


,codigo_ibge,municipio,uf,regiao,score_relevancia_produtiva,score_pressao_agroambiental,indice_exploratorio_priorizacao,producao_media_t,rendimento_medio_kg_ha,rendimento_cv_pct,precipitacao_cv_pct,carbono_solo_variacao_2019_2024_t_ha,cobertura_natural_variacao_2019_2024_pp,percentual_conversao_para_soja_pct,taxa_emissao_co2_t_ha_ano,quantidade_anos
1261,5108501,Vera,MT,Centro-Oeste,88.170122,63.150256,75.660189,5.283867e+05,3503.333333,6.056942,27.842307,-0.945546,-1.327264,93.391544,5.427332,6
1160,5103056,Cláudia,MT,Centro-Oeste,81.736955,67.532028,74.634491,3.758143e+05,3573.333333,8.032578,23.704517,-2.026339,-4.683852,99.198247,4.056883,6
1207,5106240,Nova Ubiratã,MT,Centro-Oeste,84.846319,62.807670,73.826994,1.399125e+06,3570.000000,3.485116,29.455238,-1.769811,-2.229060,83.349884,4.782200,6
1251,5107909,Sinop,MT,Centro-Oeste,86.907315,60.118109,73.512712,5.861090e+05,3555.000000,4.276223,28.205827,-1.033442,-2.678318,87.306118,3.319606,6
1233,5107248,Santa Carmem,MT,Centro-Oeste,78.961163,67.707282,73.334222,4.169467e+05,3401.666667,7.278245,25.878279,-2.159841,-2.779709,95.888201,5.627869,6
1180,5104542,Itanhangá,MT,Centro-Oeste,71.217536,74.133519,72.675528,3.450025e+05,3050.500000,14.561690,22.530649,-1.171663,-3.594956,99.908544,7.665721,6
1255,5108006,Tapurah,MT,Centro-Oeste,78.961163,63.427846,71.194504,6.251863e+05,3221.666667,12.394835,26.285414,-0.687341,-1.930283,76.843022,3.365911,6
1148,5101852,Bom Jesus do Araguaia,MT,Centro-Oeste,84.798666,57.432346,71.115506,4.506958e+05,3691.333333,7.209768,16.820390,-0.785938,-2.060593,99.438193,1.943528,6
848,4312617,Muitos Capões,RS,Sul,87.800810,52.931927,70.366369,1.795650e+05,3600.000000,12.909944,17.133560,-0.580278,-2.530545,99.183777,0.971872,6
1248,5107859,São Félix do Araguaia,MT,Centro-Oeste,81.010245,59.413833,70.212039,9.530055e+05,3731.333333,7.281864,14.414111,-2.149038,-2.089898,99.777516,1.587585,6


In [61]:
# ============================================================
# QUADRANTES — PROPORÇÃO DENTRO DE CADA UF
# ============================================================

base_elegivel_modelo = (
    base_modelo.loc[
        base_modelo[
            "elegivel_cruzamento_priorizacao"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Quantidade absoluta
# ------------------------------------------------------------

quadrantes_uf_contagem = (
    pd.crosstab(
        base_elegivel_modelo[
            "uf"
        ],
        base_elegivel_modelo[
            "quadrante_priorizacao"
        ]
    )
)


# ------------------------------------------------------------
# Percentual dentro da UF
# ------------------------------------------------------------

quadrantes_uf_pct = (
    pd.crosstab(
        base_elegivel_modelo[
            "uf"
        ],
        base_elegivel_modelo[
            "quadrante_priorizacao"
        ],
        normalize="index"
    )
    *
    100
)


# ------------------------------------------------------------
# Resumo estratégico
# ------------------------------------------------------------

resumo_estrategico_uf = pd.DataFrame(
    {
        "municipios_elegiveis":
            base_elegivel_modelo
            .groupby(
                "uf"
            )
            .size(),

        "municipios_alta_relevancia_alta_pressao":
            (
                base_elegivel_modelo
                .loc[
                    base_elegivel_modelo[
                        "quadrante_priorizacao"
                    ]
                    .eq(
                        "Alta relevância + Alta pressão"
                    )
                ]
                .groupby(
                    "uf"
                )
                .size()
            )
    }
)


resumo_estrategico_uf[
    "municipios_alta_relevancia_alta_pressao"
] = (
    resumo_estrategico_uf[
        "municipios_alta_relevancia_alta_pressao"
    ]
    .fillna(0)
    .astype(int)
)


resumo_estrategico_uf[
    "pct_estrategico_na_uf"
] = (
    resumo_estrategico_uf[
        "municipios_alta_relevancia_alta_pressao"
    ]
    /
    resumo_estrategico_uf[
        "municipios_elegiveis"
    ]
    *
    100
)


resumo_estrategico_uf = (
    resumo_estrategico_uf
    .sort_values(
        by="pct_estrategico_na_uf",
        ascending=False
    )
)


print("=" * 70)
print("QUADRANTE ESTRATÉGICO — PROPORÇÃO POR UF")
print("=" * 70)


display(
    resumo_estrategico_uf
)


print(
    "\nPercentual de todos os quadrantes dentro de cada UF:"
)


display(
    quadrantes_uf_pct
    .round(2)
)

QUADRANTE ESTRATÉGICO — PROPORÇÃO POR UF


,municipios_elegiveis,municipios_alta_relevancia_alta_pressao,pct_estrategico_na_uf
uf,,,
MT,121,59,48.760331
RS,418,73,17.464115
MS,77,13,16.883117
PR,380,54,14.210526
SC,196,14,7.142857
GO,207,12,5.797101
DF,1,0,0.000000



Percentual de todos os quadrantes dentro de cada UF:


quadrante_priorizacao,Alta relevância + Alta pressão,Alta relevância + Baixa pressão,Baixa relevância + Alta pressão,Baixa relevância + Baixa pressão
uf,,,,
DF,0.00,100.00,0.00,0.00
GO,5.80,41.06,23.67,29.47
MS,16.88,33.77,28.57,20.78
MT,48.76,27.27,20.66,3.31
PR,14.21,57.11,17.89,10.79
RS,17.46,21.29,55.26,5.98
SC,7.14,12.24,40.82,39.80


In [62]:
# ============================================================
# PERFIL DOS SUBSCORES POR QUADRANTE
# ============================================================

COLUNAS_SUBSCORES = [
    # Relevância
    "subscore_escala_produtiva",
    "subscore_eficiencia_produtiva",
    "subscore_presenca_soja",

    # Pressão
    "subscore_instabilidade_produtiva",
    "subscore_variabilidade_climatica",
    "subscore_mudancas_ambientais",
    "subscore_historico_brluc",

    # Scores finais
    "score_relevancia_produtiva",
    "score_pressao_agroambiental"
]


ORDEM_QUADRANTES = [
    "Alta relevância + Alta pressão",
    "Alta relevância + Baixa pressão",
    "Baixa relevância + Alta pressão",
    "Baixa relevância + Baixa pressão"
]


perfil_medio_quadrantes = (
    base_elegivel_modelo
    .groupby(
        "quadrante_priorizacao"
    )[
        COLUNAS_SUBSCORES
    ]
    .mean()
    .reindex(
        ORDEM_QUADRANTES
    )
    .round(2)
)


print("=" * 70)
print("PERFIL MÉDIO DOS QUADRANTES")
print("=" * 70)


display(
    perfil_medio_quadrantes
)

PERFIL MÉDIO DOS QUADRANTES


,subscore_escala_produtiva,subscore_eficiencia_produtiva,subscore_presenca_soja,subscore_instabilidade_produtiva,subscore_variabilidade_climatica,subscore_mudancas_ambientais,subscore_historico_brluc,score_relevancia_produtiva,score_pressao_agroambiental
quadrante_priorizacao,,,,,,,,,
Alta relevância + Alta pressão,74.18,55.65,61.60,48.94,62.44,63.29,55.88,63.81,57.64
Alta relevância + Baixa pressão,69.90,65.85,71.73,41.89,46.78,35.88,29.86,69.16,38.60
Baixa relevância + Alta pressão,30.20,31.26,32.25,62.66,53.31,62.33,68.84,31.23,61.79
Baixa relevância + Baixa pressão,25.62,50.46,30.01,41.46,37.37,40.49,53.48,35.36,43.20


In [63]:
# ============================================================
# AUDITORIA DE QUALIDADE / CONFIANÇA
# ============================================================

print("=" * 70)
print("COBERTURA TEMPORAL DOS MUNICÍPIOS ELEGÍVEIS")
print("=" * 70)


display(
    base_elegivel_modelo[
        "quantidade_anos"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\n" + "=" * 70
)

print(
    "COBERTURA TEMPORAL — QUADRANTE ESTRATÉGICO"
)

print("=" * 70)


display(
    quadrante_estrategico[
        "quantidade_anos"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\n" + "=" * 70
)

print(
    "INCERTEZA SOC BRLUC POR QUADRANTE"
)

print("=" * 70)


qualidade_brluc_quadrante = (
    base_elegivel_modelo
    .groupby(
        "quadrante_priorizacao"
    )[
        "soc_t1_incerteza_pct"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max"
        ]
    )
    .reindex(
        ORDEM_QUADRANTES
    )
    .round(2)
)


display(
    qualidade_brluc_quadrante
)


print(
    "\nDistribuição geral da incerteza SOC BRLUC:"
)


display(
    base_elegivel_modelo[
        "soc_t1_incerteza_pct"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

COBERTURA TEMPORAL DOS MUNICÍPIOS ELEGÍVEIS


quantidade_anos
3       1
4       3
5      10
6    1386
Name: count, dtype: int64


COBERTURA TEMPORAL — QUADRANTE ESTRATÉGICO


quantidade_anos
6    225
Name: count, dtype: int64


INCERTEZA SOC BRLUC POR QUADRANTE


,count,mean,median,min,max
quadrante_priorizacao,,,,,
Alta relevância + Alta pressão,225,27.18,23.71,14.46,60.23
Alta relevância + Baixa pressão,475,25.32,23.11,14.21,57.67
Baixa relevância + Alta pressão,475,31.28,31.52,13.39,74.09
Baixa relevância + Baixa pressão,225,25.09,22.55,14.24,58.84



Distribuição geral da incerteza SOC BRLUC:


count    1400.000000
mean       27.602318
std         9.815011
min        13.385147
10%        16.095294
25%        20.309963
50%        24.978798
75%        33.180190
90%        41.092553
max        74.087374
Name: soc_t1_incerteza_pct, dtype: float64

In [64]:
# ============================================================
# INVENTÁRIO DE VARIÁVEIS DE QUALIDADE / INCERTEZA
# ============================================================

PALAVRAS_CHAVE_QUALIDADE = [
    "qualidade",
    "incerteza",
    "ic95",
    "status",
    "estacao",
    "distancia",
    "interpol",
    "n_obs",
    "disponivel"
]


colunas_qualidade_disponiveis = [
    coluna
    for coluna in base.columns
    if any(
        palavra.lower()
        in coluna.lower()
        for palavra in PALAVRAS_CHAVE_QUALIDADE
    )
]


print("=" * 70)
print("VARIÁVEIS POTENCIAIS DE QUALIDADE")
print("=" * 70)


print(
    "\nQuantidade:",
    len(
        colunas_qualidade_disponiveis
    )
)


for coluna in colunas_qualidade_disponiveis:

    print(
        "->",
        coluna
    )

VARIÁVEIS POTENCIAIS DE QUALIDADE

Quantidade: 43
-> status_dado
-> qualidade_espacial_precipitacao
-> qualidade_espacial_temperatura
-> qualidade_espacial_umidade
-> score_qualidade_climatica
-> qualidade_climatica_geral
-> status_dado_seeg
-> emissao_direta_co2e_gwp_ar6_soma_disponivel_t
-> emissao_indireta_co2e_gwp_ar6_soma_disponivel_t
-> emissao_total_co2e_gwp_ar6_soma_disponivel_t
-> emissao_direta_n2o_soma_disponivel_t
-> emissao_indireta_n2o_soma_disponivel_t
-> emissao_total_n2o_soma_disponivel_t
-> emissao_absoluta_co2_ic95_inf
-> emissao_absoluta_co2_ic95_sup
-> ic95_emissao_absoluta_disponivel
-> taxa_emissao_co2_ic95_inf
-> taxa_emissao_co2_ic95_sup
-> ic95_taxa_emissao_disponivel
-> ctotal_t0_ic95_inf
-> ctotal_t0_ic95_sup
-> ctotal_t0_incerteza_relativa
-> ctotal_t1_ic95_inf
-> ctotal_t1_ic95_sup
-> ctotal_t1_incerteza_relativa
-> soc_t0_ic95_inf
-> soc_t0_ic95_sup
-> soc_t0_incerteza_relativa
-> soc_t1_ic95_inf
-> soc_t1_ic95_sup
-> soc_t1_incerteza_relativa
-> cveg_t0_

In [65]:
# ============================================================
# QUALIDADE BRLUC — DISPONIBILIDADE DOS IC95
# ============================================================

COLUNAS_IC95_BRLUC = [
    "ic95_emissao_absoluta_disponivel",
    "ic95_taxa_emissao_disponivel"
]


print("=" * 70)
print("DISPONIBILIDADE DOS IC95 — BRLUC")
print("=" * 70)


for coluna in COLUNAS_IC95_BRLUC:

    print(
        f"\n{coluna}"
    )

    display(
        base_elegivel_modelo[
            coluna
        ]
        .value_counts(
            dropna=False
        )
    )


print(
    "\n" + "=" * 70
)

print(
    "QUADRANTE ESTRATÉGICO"
)

print("=" * 70)


for coluna in COLUNAS_IC95_BRLUC:

    print(
        f"\n{coluna}"
    )

    display(
        quadrante_estrategico[
            coluna
        ]
        .value_counts(
            dropna=False
        )
    )

DISPONIBILIDADE DOS IC95 — BRLUC

ic95_emissao_absoluta_disponivel


ic95_emissao_absoluta_disponivel
True    1400
Name: count, dtype: int64


ic95_taxa_emissao_disponivel


ic95_taxa_emissao_disponivel
True     1397
False       3
Name: count, dtype: int64


QUADRANTE ESTRATÉGICO

ic95_emissao_absoluta_disponivel


ic95_emissao_absoluta_disponivel
True    225
Name: count, dtype: int64


ic95_taxa_emissao_disponivel


ic95_taxa_emissao_disponivel
True    225
Name: count, dtype: int64

In [66]:
# ============================================================
# INCERTEZA BRLUC — QUADRANTE ESTRATÉGICO
# ============================================================

colunas_incerteza_estrategico = [
    "codigo_ibge",
    "municipio",
    "uf",

    "score_relevancia_produtiva",
    "score_pressao_agroambiental",

    "soc_t1_incerteza_pct",

    "ic95_emissao_absoluta_disponivel",
    "ic95_taxa_emissao_disponivel",

    "quantidade_anos"
]


print("=" * 70)
print("MAIORES INCERTEZAS BRLUC — QUADRANTE ESTRATÉGICO")
print("=" * 70)


display(
    quadrante_estrategico[
        colunas_incerteza_estrategico
    ]
    .sort_values(
        by="soc_t1_incerteza_pct",
        ascending=False
    )
    .head(20)
)

MAIORES INCERTEZAS BRLUC — QUADRANTE ESTRATÉGICO


,codigo_ibge,municipio,uf,score_relevancia_produtiva,score_pressao_agroambiental,soc_t1_incerteza_pct,ic95_emissao_absoluta_disponivel,ic95_taxa_emissao_disponivel,quantidade_anos
706,4305405,Chiapetta,RS,71.658327,49.739037,60.227018,True,True,6
773,4309001,Giruá,RS,67.476769,53.913700,56.970647,True,True,6
862,4313334,Nova Ramada,RS,60.495592,51.171475,56.829714,True,True,6
997,4320651,Silveira Martins,RS,55.063140,55.245361,55.551853,True,True,6
796,4310405,Independência,RS,64.343579,51.784475,53.868747,True,True,6
945,4317509,Santo Ângelo,RS,60.233500,57.117623,53.818438,True,True,6
638,4301503,Augusto Pestana,RS,58.017632,52.569872,50.983555,True,True,6
808,4310876,Jacuizinho,RS,54.658089,58.497131,49.727882,True,True,6
1033,4322509,Vacaria,RS,72.373124,51.263410,46.845320,True,True,6
742,4306932,Entre-Ijuís,RS,57.350488,58.951128,46.683033,True,True,6


In [67]:
# ============================================================
# AUDITORIA DETALHADA DAS VARIÁVEIS DE QUALIDADE
# ============================================================

COLUNAS_QUALIDADE_ANALISAR = [
    # MapBiomas Solo / dado ambiental
    "status_dado",

    # INMET / qualidade espacial
    "qualidade_espacial_precipitacao",
    "qualidade_espacial_temperatura",
    "qualidade_espacial_umidade",

    # INMET / qualidade consolidada
    "score_qualidade_climatica",
    "qualidade_climatica_geral"
]


print("=" * 70)
print("AUDITORIA DAS VARIÁVEIS DE QUALIDADE")
print("=" * 70)


for coluna in COLUNAS_QUALIDADE_ANALISAR:

    serie = base[coluna]

    print("\n" + "-" * 70)
    print(coluna)
    print("-" * 70)

    print(
        "dtype:",
        serie.dtype
    )

    print(
        "NULL:",
        serie.isna().sum()
    )

    print(
        "Valores distintos:",
        serie.nunique(dropna=False)
    )


    # --------------------------------------------------------
    # Numérica
    # --------------------------------------------------------

    if pd.api.types.is_numeric_dtype(serie):

        display(
            serie.describe(
                percentiles=[
                    0.01,
                    0.05,
                    0.25,
                    0.50,
                    0.75,
                    0.95,
                    0.99
                ]
            )
        )

    # --------------------------------------------------------
    # Categórica
    # --------------------------------------------------------

    else:

        display(
            serie
            .value_counts(
                dropna=False
            )
            .head(20)
        )

AUDITORIA DAS VARIÁVEIS DE QUALIDADE

----------------------------------------------------------------------
status_dado
----------------------------------------------------------------------
dtype: object
NULL: 0
Valores distintos: 1


status_dado
disponivel    8674
Name: count, dtype: int64


----------------------------------------------------------------------
qualidade_espacial_precipitacao
----------------------------------------------------------------------
dtype: object
NULL: 0
Valores distintos: 5


qualidade_espacial_precipitacao
alta           7008
observado       606
media           461
baixa           452
muito_baixa     147
Name: count, dtype: int64


----------------------------------------------------------------------
qualidade_espacial_temperatura
----------------------------------------------------------------------
dtype: object
NULL: 0
Valores distintos: 5


...

qualidade_espacial_umidade
alta           7189
observado       648
media           387
baixa           304
muito_baixa     146
Name: count, dtype: int64


----------------------------------------------------------------------
score_qualidade_climatica
----------------------------------------------------------------------
dtype: int64
NULL: 0
Valores distintos: 5


count    8674.000000
mean        3.839290
std         0.708692
min         1.000000
1%          1.000000
5%          2.000000
25%         4.000000
50%         4.000000
75%         4.000000
95%         5.000000
99%         5.000000
max         5.000000
Name: score_qualidade_climatica, dtype: float64


----------------------------------------------------------------------
qualidade_climatica_geral
----------------------------------------------------------------------
dtype: object
NULL: 0
Valores distintos: 5


qualidade_climatica_geral
alta           6993
observado       564
baixa           465
media           464
muito_baixa     188
Name: count, dtype: int64

In [68]:
# ============================================================
# COMPORTAMENTO TEMPORAL DA QUALIDADE CLIMÁTICA
# ============================================================

auditoria_temporal_qualidade = (
    base
    .groupby(
        [
            "codigo_ibge",
            "municipio",
            "uf"
        ],
        as_index=False
    )
    .agg(
        anos_com_score_climatico=(
            "score_qualidade_climatica",
            "count"
        ),

        score_climatico_medio=(
            "score_qualidade_climatica",
            "mean"
        ),

        score_climatico_minimo=(
            "score_qualidade_climatica",
            "min"
        ),

        score_climatico_maximo=(
            "score_qualidade_climatica",
            "max"
        ),

        quantidade_scores_climaticos_distintos=(
            "score_qualidade_climatica",
            "nunique"
        ),

        quantidade_classes_climaticas_distintas=(
            "qualidade_climatica_geral",
            "nunique"
        )
    )
)


print("=" * 70)
print("QUALIDADE CLIMÁTICA — COMPORTAMENTO MUNICIPAL")
print("=" * 70)


print(
    "\nDimensão:",
    auditoria_temporal_qualidade.shape
)


print(
    "Municípios:",
    auditoria_temporal_qualidade[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nMunicípios com mais de um score climático:"
)

print(
    (
        auditoria_temporal_qualidade[
            "quantidade_scores_climaticos_distintos"
        ]
        > 1
    )
    .sum()
)


print(
    "\nMunicípios com mais de uma classe climática:"
)

print(
    (
        auditoria_temporal_qualidade[
            "quantidade_classes_climaticas_distintas"
        ]
        > 1
    )
    .sum()
)


print(
    "\nDistribuição do número de anos com score:"
)


display(
    auditoria_temporal_qualidade[
        "anos_com_score_climatico"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nDistribuição do score climático médio:"
)


display(
    auditoria_temporal_qualidade[
        "score_climatico_medio"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

QUALIDADE CLIMÁTICA — COMPORTAMENTO MUNICIPAL

Dimensão: (1505, 9)
Municípios: 1505

Municípios com mais de um score climático:
532

Municípios com mais de uma classe climática:
532

Distribuição do número de anos com score:


anos_com_score_climatico
1      24
2      27
3      25
4      15
5      23
6    1391
Name: count, dtype: int64


Distribuição do score climático médio:


count    1505.000000
mean        3.835759
std         0.542547
min         1.000000
10%         3.333333
25%         3.833333
50%         4.000000
75%         4.000000
90%         4.000000
max         5.000000
Name: score_climatico_medio, dtype: float64

In [69]:
# ============================================================
# COBERTURA DE QUALIDADE — UNIVERSO DO MODELO
# ============================================================

codigos_modelo = set(
    base_elegivel_modelo[
        "codigo_ibge"
    ]
)


qualidade_modelo = (
    auditoria_temporal_qualidade.loc[
        auditoria_temporal_qualidade[
            "codigo_ibge"
        ]
        .isin(
            codigos_modelo
        )
    ]
    .copy()
)


print("=" * 70)
print("QUALIDADE CLIMÁTICA — UNIVERSO ELEGÍVEL")
print("=" * 70)


print(
    "\nMunicípios encontrados:",
    qualidade_modelo[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "Esperado:",
    len(
        codigos_modelo
    )
)


print(
    "\nNULL no score climático médio:",
    qualidade_modelo[
        "score_climatico_medio"
    ]
    .isna()
    .sum()
)


print(
    "NULL no score climático mínimo:",
    qualidade_modelo[
        "score_climatico_minimo"
    ]
    .isna()
    .sum()
)


print(
    "\nResumo do score mínimo:"
)


display(
    qualidade_modelo[
        "score_climatico_minimo"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

QUALIDADE CLIMÁTICA — UNIVERSO ELEGÍVEL

Municípios encontrados: 1400
Esperado: 1400

NULL no score climático médio: 0
NULL no score climático mínimo: 0

Resumo do score mínimo:


count    1400.000000
mean        3.440000
std         0.959122
min         1.000000
10%         2.000000
25%         3.000000
50%         4.000000
75%         4.000000
90%         4.000000
max         5.000000
Name: score_climatico_minimo, dtype: float64

In [70]:
# ============================================================
# MAPEAMENTO ENTRE SCORE E CLASSE DE QUALIDADE CLIMÁTICA
# ============================================================

mapeamento_score_classe_climatica = (
    pd.crosstab(
        base[
            "score_qualidade_climatica"
        ],
        base[
            "qualidade_climatica_geral"
        ]
    )
)


print("=" * 70)
print("MAPEAMENTO SCORE × CLASSE CLIMÁTICA")
print("=" * 70)


display(
    mapeamento_score_classe_climatica
)


# ------------------------------------------------------------
# Distribuição do PIOR score climático no universo elegível
# ------------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "PIOR SCORE CLIMÁTICO — UNIVERSO DO MODELO"
)

print("=" * 70)


display(
    qualidade_modelo[
        "score_climatico_minimo"
    ]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Mesmo diagnóstico apenas no quadrante estratégico
# ------------------------------------------------------------

codigos_estrategicos = set(
    quadrante_estrategico[
        "codigo_ibge"
    ]
)


qualidade_estrategico = (
    auditoria_temporal_qualidade.loc[
        auditoria_temporal_qualidade[
            "codigo_ibge"
        ]
        .isin(
            codigos_estrategicos
        )
    ]
    .copy()
)


print(
    "\n" + "=" * 70
)

print(
    "PIOR SCORE CLIMÁTICO — QUADRANTE ESTRATÉGICO"
)

print("=" * 70)


display(
    qualidade_estrategico[
        "score_climatico_minimo"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nScore climático médio — quadrante estratégico:"
)


display(
    qualidade_estrategico[
        "score_climatico_medio"
    ]
    .describe()
)

MAPEAMENTO SCORE × CLASSE CLIMÁTICA


qualidade_climatica_geral,alta,baixa,media,muito_baixa,observado
score_qualidade_climatica,,,,,
1,0,0,0,188,0
2,0,465,0,0,0
3,0,0,464,0,0
4,6993,0,0,0,0
5,0,0,0,0,564



PIOR SCORE CLIMÁTICO — UNIVERSO DO MODELO


score_climatico_minimo
1     80
2    205
3    160
4    929
5     26
Name: count, dtype: int64


PIOR SCORE CLIMÁTICO — QUADRANTE ESTRATÉGICO


score_climatico_minimo
1     53
2     23
3     15
4    129
5      5
Name: count, dtype: int64


Score climático médio — quadrante estratégico:


count    225.000000
mean       3.595556
std        0.869101
min        1.000000
25%        3.333333
50%        4.000000
75%        4.000000
max        5.000000
Name: score_climatico_medio, dtype: float64

In [71]:
# ============================================================
# IC95 DA TAXA DE EMISSÃO BRLUC — CAMADA MUNICIPAL
# ============================================================

COLUNAS_IC95_TAXA = [
    "taxa_emissao_co2_ic95_inf",
    "taxa_emissao_co2_ic95_sup"
]


auditoria_estatica_ic95_taxa = []


for coluna in COLUNAS_IC95_TAXA:

    max_distintos = (
        base
        .groupby(
            "codigo_ibge"
        )[
            coluna
        ]
        .nunique(
            dropna=False
        )
        .max()
    )


    auditoria_estatica_ic95_taxa.append(
        {
            "coluna":
                coluna,

            "max_valores_distintos_por_municipio":
                max_distintos
        }
    )


auditoria_estatica_ic95_taxa = (
    pd.DataFrame(
        auditoria_estatica_ic95_taxa
    )
)


print("=" * 70)
print("AUDITORIA IC95 TAXA BRLUC")
print("=" * 70)


display(
    auditoria_estatica_ic95_taxa
)


print(
    "\nCampos constantes por município:"
)

print(
    (
        auditoria_estatica_ic95_taxa[
            "max_valores_distintos_por_municipio"
        ]
        <= 1
    )
    .all()
)


# ------------------------------------------------------------
# Uma linha por município
# ------------------------------------------------------------

ic95_taxa_municipal = (
    base[
        [
            "codigo_ibge",
            "taxa_emissao_co2_ic95_inf",
            "taxa_emissao_co2_ic95_sup"
        ]
    ]
    .drop_duplicates(
        subset=[
            "codigo_ibge"
        ]
    )
    .copy()
)


print(
    "\nDimensão:",
    ic95_taxa_municipal.shape
)


print(
    "Duplicatas:",
    ic95_taxa_municipal
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)

AUDITORIA IC95 TAXA BRLUC


,coluna,max_valores_distintos_por_municipio
0,taxa_emissao_co2_ic95_inf,1
1,taxa_emissao_co2_ic95_sup,1



Campos constantes por município:
True

Dimensão: (1505, 3)
Duplicatas: 0


In [72]:
# ============================================================
# ACRESCENTAR CAMPOS DE CONFIANÇA AO MODELO MUNICIPAL
# ============================================================

qualidade_climatica_municipal = (
    auditoria_temporal_qualidade[
        [
            "codigo_ibge",
            "anos_com_score_climatico",
            "score_climatico_medio",
            "score_climatico_minimo",
            "score_climatico_maximo",
            "quantidade_scores_climaticos_distintos"
        ]
    ]
    .copy()
)


base_modelo = (
    base_modelo
    .merge(
        qualidade_climatica_municipal,
        on="codigo_ibge",
        how="left",
        validate="one_to_one"
    )
    .merge(
        ic95_taxa_municipal,
        on="codigo_ibge",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# IC95 da taxa cruza zero?
#
# Isso é relevante porque indica que o intervalo inclui
# tanto valores positivos quanto negativos.
# ------------------------------------------------------------

base_modelo[
    "flag_ic95_taxa_cruza_zero"
] = pd.Series(
    pd.NA,
    index=base_modelo.index,
    dtype="boolean"
)


mascara_ic95_valido = (
    base_modelo[
        "ic95_taxa_emissao_disponivel"
    ]
    &
    base_modelo[
        "taxa_emissao_co2_ic95_inf"
    ]
    .notna()
    &
    base_modelo[
        "taxa_emissao_co2_ic95_sup"
    ]
    .notna()
)


base_modelo.loc[
    mascara_ic95_valido,
    "flag_ic95_taxa_cruza_zero"
] = (
    (
        base_modelo.loc[
            mascara_ic95_valido,
            "taxa_emissao_co2_ic95_inf"
        ]
        <= 0
    )
    &
    (
        base_modelo.loc[
            mascara_ic95_valido,
            "taxa_emissao_co2_ic95_sup"
        ]
        >= 0
    )
)


# ------------------------------------------------------------
# Amplitude absoluta do IC95
# Apenas para diagnóstico.
# ------------------------------------------------------------

base_modelo[
    "amplitude_ic95_taxa_emissao"
] = (
    base_modelo[
        "taxa_emissao_co2_ic95_sup"
    ]
    -
    base_modelo[
        "taxa_emissao_co2_ic95_inf"
    ]
)


print("=" * 70)
print("BASE MODELO + QUALIDADE")
print("=" * 70)


print(
    "\nDimensão:",
    base_modelo.shape
)


print(
    "Municípios:",
    base_modelo[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "Duplicatas:",
    base_modelo
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)

BASE MODELO + QUALIDADE

Dimensão: (1505, 79)
Municípios: 1505
Duplicatas: 0


In [73]:
# ============================================================
# AUDITORIA DOS COMPONENTES DE CONFIANÇA
# ============================================================

modelo_elegivel_qualidade = (
    base_modelo.loc[
        base_modelo[
            "elegivel_cruzamento_priorizacao"
        ]
    ]
    .copy()
)


estrategico_qualidade = (
    base_modelo.loc[
        base_modelo[
            "quadrante_priorizacao"
        ]
        .eq(
            "Alta relevância + Alta pressão"
        )
    ]
    .copy()
)


print("=" * 70)
print("COMPONENTES DE CONFIANÇA — 1.400 MUNICÍPIOS")
print("=" * 70)


print(
    "\nCobertura temporal:"
)

display(
    modelo_elegivel_qualidade[
        "quantidade_anos"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nPior qualidade climática observada:"
)

display(
    modelo_elegivel_qualidade[
        "score_climatico_minimo"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nIC95 da taxa BRLUC disponível:"
)

display(
    modelo_elegivel_qualidade[
        "ic95_taxa_emissao_disponivel"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nIC95 da taxa BRLUC cruza zero:"
)

display(
    modelo_elegivel_qualidade[
        "flag_ic95_taxa_cruza_zero"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nAmplitude do IC95 da taxa:"
)

display(
    modelo_elegivel_qualidade[
        "amplitude_ic95_taxa_emissao"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


print(
    "\n" + "=" * 70
)

print(
    "COMPONENTES DE CONFIANÇA — 225 ESTRATÉGICOS"
)

print("=" * 70)


print(
    "\nPior qualidade climática observada:"
)

display(
    estrategico_qualidade[
        "score_climatico_minimo"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nIC95 da taxa cruza zero:"
)

display(
    estrategico_qualidade[
        "flag_ic95_taxa_cruza_zero"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nAmplitude do IC95:"
)

display(
    estrategico_qualidade[
        "amplitude_ic95_taxa_emissao"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

COMPONENTES DE CONFIANÇA — 1.400 MUNICÍPIOS

Cobertura temporal:


quantidade_anos
3       1
4       3
5      10
6    1386
Name: count, dtype: int64


Pior qualidade climática observada:


score_climatico_minimo
1     80
2    205
3    160
4    929
5     26
Name: count, dtype: int64


IC95 da taxa BRLUC disponível:


...

score_climatico_minimo
1     53
2     23
3     15
4    129
5      5
Name: count, dtype: int64


IC95 da taxa cruza zero:


flag_ic95_taxa_cruza_zero
False    202
True      23
Name: count, dtype: Int64


Amplitude do IC95:


count    225.000000
mean       1.738977
std        1.989588
min        0.100836
10%        0.226023
25%        0.405965
50%        0.838850
75%        2.375617
90%        4.804026
max        9.732386
Name: amplitude_ic95_taxa_emissao, dtype: float64

In [74]:
# ============================================================
# FAIXAS DE CONFIANÇA DO MODELO
# ============================================================

# ------------------------------------------------------------
# Classe correspondente ao PIOR score climático observado
# ------------------------------------------------------------

MAPA_QUALIDADE_CLIMATICA = {
    1: "muito_baixa",
    2: "baixa",
    3: "media",
    4: "alta",
    5: "observado"
}


base_modelo[
    "pior_qualidade_climatica"
] = (
    base_modelo[
        "score_climatico_minimo"
    ]
    .map(
        MAPA_QUALIDADE_CLIMATICA
    )
)


# ------------------------------------------------------------
# Flags de ressalva
# ------------------------------------------------------------

base_modelo[
    "flag_cobertura_temporal_incompleta"
] = (
    base_modelo[
        "quantidade_anos"
    ]
    < 6
)


base_modelo[
    "flag_clima_muito_baixa"
] = (
    base_modelo[
        "score_climatico_minimo"
    ]
    .eq(1)
)


base_modelo[
    "flag_clima_baixa_ou_media"
] = (
    base_modelo[
        "score_climatico_minimo"
    ]
    .isin(
        [
            2,
            3
        ]
    )
)


base_modelo[
    "flag_ic95_taxa_indisponivel"
] = (
    ~base_modelo[
        "ic95_taxa_emissao_disponivel"
    ]
)


# ------------------------------------------------------------
# Classificação
#
# ALTA:
# - modelo completo
# - 6 anos
# - pior qualidade climática alta ou observada
# - IC95 disponível
# - IC95 da taxa não cruza zero
#
# MODERADA:
# - modelo completo
# - sem condição grave
# - mas existe alguma ressalva intermediária
#
# LIMITADA:
# - qualidade climática muito baixa em algum ano
# - ou cobertura temporal <= 4 anos
# - ou IC95 da taxa indisponível
#
# Municípios fora do modelo:
# - não recebem faixa de confiança do ranking
# ------------------------------------------------------------

base_modelo[
    "faixa_confianca_modelo"
] = (
    "Não aplicável — dados insuficientes"
)


mascara_elegivel = (
    base_modelo[
        "elegivel_cruzamento_priorizacao"
    ]
)


mascara_limitada = (
    mascara_elegivel
    &
    (
        (
            base_modelo[
                "quantidade_anos"
            ]
            <= 4
        )
        |
        base_modelo[
            "flag_clima_muito_baixa"
        ]
        |
        base_modelo[
            "flag_ic95_taxa_indisponivel"
        ]
    )
)


mascara_moderada = (
    mascara_elegivel
    &
    ~mascara_limitada
    &
    (
        (
            base_modelo[
                "quantidade_anos"
            ]
            == 5
        )
        |
        base_modelo[
            "flag_clima_baixa_ou_media"
        ]
        |
        base_modelo[
            "flag_ic95_taxa_cruza_zero"
        ]
        .fillna(False)
    )
)


mascara_alta = (
    mascara_elegivel
    &
    ~mascara_limitada
    &
    ~mascara_moderada
)


base_modelo.loc[
    mascara_alta,
    "faixa_confianca_modelo"
] = "Alta"


base_modelo.loc[
    mascara_moderada,
    "faixa_confianca_modelo"
] = "Moderada"


base_modelo.loc[
    mascara_limitada,
    "faixa_confianca_modelo"
] = "Limitada"


print("=" * 70)
print("FAIXAS DE CONFIANÇA")
print("=" * 70)


display(
    base_modelo[
        "faixa_confianca_modelo"
    ]
    .value_counts()
)

FAIXAS DE CONFIANÇA


faixa_confianca_modelo
Alta                                   781
Moderada                               532
Não aplicável — dados insuficientes    105
Limitada                                87
Name: count, dtype: int64

In [75]:
# ============================================================
# CONFIANÇA POR QUADRANTE
# ============================================================

ORDEM_CONFIANCA = [
    "Alta",
    "Moderada",
    "Limitada"
]


confianca_por_quadrante = (
    pd.crosstab(
        base_modelo.loc[
            base_modelo[
                "elegivel_cruzamento_priorizacao"
            ],
            "quadrante_priorizacao"
        ],

        base_modelo.loc[
            base_modelo[
                "elegivel_cruzamento_priorizacao"
            ],
            "faixa_confianca_modelo"
        ]
    )
    .reindex(
        columns=ORDEM_CONFIANCA,
        fill_value=0
    )
)


confianca_por_quadrante_pct = (
    pd.crosstab(
        base_modelo.loc[
            base_modelo[
                "elegivel_cruzamento_priorizacao"
            ],
            "quadrante_priorizacao"
        ],

        base_modelo.loc[
            base_modelo[
                "elegivel_cruzamento_priorizacao"
            ],
            "faixa_confianca_modelo"
        ],

        normalize="index"
    )
    *
    100
)


confianca_por_quadrante_pct = (
    confianca_por_quadrante_pct
    .reindex(
        columns=ORDEM_CONFIANCA,
        fill_value=0
    )
)


print("=" * 70)
print("CONFIANÇA POR QUADRANTE — CONTAGEM")
print("=" * 70)


display(
    confianca_por_quadrante
)


print(
    "\n" + "=" * 70
)

print(
    "CONFIANÇA POR QUADRANTE — PERCENTUAL"
)

print("=" * 70)


display(
    confianca_por_quadrante_pct
    .round(2)
)

CONFIANÇA POR QUADRANTE — CONTAGEM


faixa_confianca_modelo,Alta,Moderada,Limitada
quadrante_priorizacao,,,
Alta relevância + Alta pressão,119,53,53
Alta relevância + Baixa pressão,194,272,9
Baixa relevância + Alta pressão,338,118,19
Baixa relevância + Baixa pressão,130,89,6



CONFIANÇA POR QUADRANTE — PERCENTUAL


faixa_confianca_modelo,Alta,Moderada,Limitada
quadrante_priorizacao,,,
Alta relevância + Alta pressão,52.89,23.56,23.56
Alta relevância + Baixa pressão,40.84,57.26,1.89
Baixa relevância + Alta pressão,71.16,24.84,4.00
Baixa relevância + Baixa pressão,57.78,39.56,2.67


In [78]:
# ============================================================
# QUADRANTE ESTRATÉGICO + CONFIANÇA
# ============================================================

estrategico_final_preliminar = (
    base_modelo.loc[
        base_modelo[
            "quadrante_priorizacao"
        ]
        .eq(
            "Alta relevância + Alta pressão"
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# Índice exploratório continua apenas como ordenação
# ------------------------------------------------------------

estrategico_final_preliminar[
    "indice_exploratorio_priorizacao"
] = (
    estrategico_final_preliminar[
        [
            "score_relevancia_produtiva",
            "score_pressao_agroambiental"
        ]
    ]
    .mean(
        axis=1
    )
)


ORDEM_CONFIANCA_MAPA = {
    "Alta": 1,
    "Moderada": 2,
    "Limitada": 3
}


estrategico_final_preliminar[
    "ordem_confianca"
] = (
    estrategico_final_preliminar[
        "faixa_confianca_modelo"
    ]
    .map(
        ORDEM_CONFIANCA_MAPA
    )
)


COLUNAS_ESTRATEGICO_FINAL = [
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",

    "quadrante_priorizacao",
    "faixa_confianca_modelo",

    "score_relevancia_produtiva",
    "score_pressao_agroambiental",
    "indice_exploratorio_priorizacao",

    # --------------------------------------------------------
    # Por que apareceu?
    # --------------------------------------------------------
    "subscore_escala_produtiva",
    "subscore_eficiencia_produtiva",
    "subscore_presenca_soja",

    "subscore_instabilidade_produtiva",
    "subscore_variabilidade_climatica",
    "subscore_mudancas_ambientais",
    "subscore_historico_brluc",

    # --------------------------------------------------------
    # Evidência / confiança
    # --------------------------------------------------------
    "quantidade_anos",
    "score_climatico_medio",
    "score_climatico_minimo",
    "pior_qualidade_climatica",

    "ic95_taxa_emissao_disponivel",
    "flag_ic95_taxa_cruza_zero"
]


print("=" * 70)
print("QUADRANTE ESTRATÉGICO — PRIORIZAÇÃO + CONFIANÇA")
print("=" * 70)


print(
    "\nMunicípios:",
    len(
        estrategico_final_preliminar
    )
)


# ============================================================
# ORDENAR PRIMEIRO E SELECIONAR COLUNAS DEPOIS
# ============================================================

estrategico_final_ordenado = (
    estrategico_final_preliminar
    .sort_values(
        by=[
            "ordem_confianca",
            "indice_exploratorio_priorizacao"
        ],
        ascending=[
            True,
            False
        ]
    )
)


display(
    estrategico_final_ordenado[
        COLUNAS_ESTRATEGICO_FINAL
    ]
    .head(30)
)

QUADRANTE ESTRATÉGICO — PRIORIZAÇÃO + CONFIANÇA

Municípios: 225


,codigo_ibge,municipio,uf,regiao,quadrante_priorizacao,faixa_confianca_modelo,score_relevancia_produtiva,score_pressao_agroambiental,indice_exploratorio_priorizacao,subscore_escala_produtiva,...,subscore_instabilidade_produtiva,subscore_variabilidade_climatica,subscore_mudancas_ambientais,subscore_historico_brluc,quantidade_anos,score_climatico_medio,score_climatico_minimo,pior_qualidade_climatica,ic95_taxa_emissao_disponivel,flag_ic95_taxa_cruza_zero
848,4312617,Muitos Capões,RS,Sul,Alta relevância + Alta pressão,Alta,87.800810,52.931927,70.366369,91.350965,...,36.954968,30.450322,73.016440,71.305980,6,4.000000,4,alta,True,False
278,4120606,Prudentópolis,PR,Sul,Alta relevância + Alta pressão,Alta,77.888968,54.732437,66.310703,89.063617,...,15.796998,72.480343,71.979986,58.672422,6,4.000000,4,alta,True,False
919,4315701,Rio Pardo,RS,Sul,Alta relevância + Alta pressão,Alta,56.802478,74.852713,65.827596,90.493209,...,84.488921,57.612580,92.172981,65.136371,6,5.000000,5,observado,True,False
1396,5212501,Luziânia,GO,Centro-Oeste,Alta relevância + Alta pressão,Alta,81.224684,49.996092,65.610388,94.210150,...,1.786991,78.305933,85.739814,34.151631,6,4.500000,4,alta,True,False
978,4319604,São Sepé,RS,Sul,Alta relevância + Alta pressão,Alta,52.704313,78.324113,65.514213,88.777698,...,94.996426,56.754825,92.637598,68.907601,6,4.000000,4,alta,True,False
299,4122156,Rio Bonito do Iguaçu,PR,Sul,Alta relevância + Alta pressão,Alta,79.890398,50.882152,65.386275,83.988563,...,14.010007,76.483202,61.043603,51.991795,6,4.000000,4,alta,True,False
283,4120903,Quedas do Iguaçu,PR,Sul,Alta relevância + Alta pressão,Alta,70.836312,59.857423,65.346867,82.916369,...,62.258756,73.981415,71.729807,31.459712,6,4.000000,4,alta,True,False
97,4107207,Dois Vizinhos,PR,Sul,Alta relevância + Alta pressão,Alta,74.910650,55.565906,65.238278,74.267334,...,72.551823,86.919228,27.162259,35.630316,6,4.333333,4,alta,True,False
428,4203808,Canoinhas,SC,Sul,Alta relevância + Alta pressão,Alta,76.840600,53.597292,65.218946,82.344532,...,17.655468,80.057184,50.643317,66.033198,6,4.000000,4,alta,True,False
912,4315321,Quevedos,RS,Sul,Alta relevância + Alta pressão,Alta,58.327377,71.774452,65.050914,76.340243,...,83.845604,45.496783,91.136526,66.618894,6,4.000000,4,alta,True,False


In [79]:
# ============================================================
# TESTE DE SENSIBILIDADE — CENÁRIOS DE PESOS
# ============================================================

CENARIOS_PESOS = {
    # ========================================================
    # CENÁRIO BASE
    # ========================================================
    "base": {
        "relevancia": {
            "subscore_escala_produtiva": 1/3,
            "subscore_eficiencia_produtiva": 1/3,
            "subscore_presenca_soja": 1/3
        },

        "pressao": {
            "subscore_instabilidade_produtiva": 0.25,
            "subscore_variabilidade_climatica": 0.25,
            "subscore_mudancas_ambientais": 0.25,
            "subscore_historico_brluc": 0.25
        }
    },


    # ========================================================
    # CENÁRIO 2 — MAIOR PESO À ESCALA PRODUTIVA
    # ========================================================
    "produtivo": {
        "relevancia": {
            "subscore_escala_produtiva": 0.50,
            "subscore_eficiencia_produtiva": 0.25,
            "subscore_presenca_soja": 0.25
        },

        "pressao": {
            "subscore_instabilidade_produtiva": 0.25,
            "subscore_variabilidade_climatica": 0.25,
            "subscore_mudancas_ambientais": 0.25,
            "subscore_historico_brluc": 0.25
        }
    },


    # ========================================================
    # CENÁRIO 3 — MAIOR PESO ÀS MUDANÇAS AMBIENTAIS
    # ========================================================
    "ambiental": {
        "relevancia": {
            "subscore_escala_produtiva": 1/3,
            "subscore_eficiencia_produtiva": 1/3,
            "subscore_presenca_soja": 1/3
        },

        "pressao": {
            "subscore_instabilidade_produtiva": 0.15,
            "subscore_variabilidade_climatica": 0.20,
            "subscore_mudancas_ambientais": 0.40,
            "subscore_historico_brluc": 0.25
        }
    },


    # ========================================================
    # CENÁRIO 4 — MAIOR PESO À VARIABILIDADE CLIMÁTICA
    # ========================================================
    "climatico": {
        "relevancia": {
            "subscore_escala_produtiva": 1/3,
            "subscore_eficiencia_produtiva": 1/3,
            "subscore_presenca_soja": 1/3
        },

        "pressao": {
            "subscore_instabilidade_produtiva": 0.15,
            "subscore_variabilidade_climatica": 0.40,
            "subscore_mudancas_ambientais": 0.20,
            "subscore_historico_brluc": 0.25
        }
    }
}


# ------------------------------------------------------------
# Auditoria dos pesos
# ------------------------------------------------------------

auditoria_pesos = []


for nome_cenario, configuracao in CENARIOS_PESOS.items():

    auditoria_pesos.append(
        {
            "cenario": nome_cenario,

            "soma_pesos_relevancia":
                sum(
                    configuracao[
                        "relevancia"
                    ].values()
                ),

            "soma_pesos_pressao":
                sum(
                    configuracao[
                        "pressao"
                    ].values()
                )
        }
    )


auditoria_pesos = pd.DataFrame(
    auditoria_pesos
)


print("=" * 70)
print("CENÁRIOS DE SENSIBILIDADE")
print("=" * 70)


display(
    auditoria_pesos
)

CENÁRIOS DE SENSIBILIDADE


,cenario,soma_pesos_relevancia,soma_pesos_pressao
0,base,1.0,1.0
1,produtivo,1.0,1.0
2,ambiental,1.0,1.0
3,climatico,1.0,1.0


In [80]:
# ============================================================
# CALCULAR CENÁRIOS DE SENSIBILIDADE
# ============================================================

resultado_sensibilidade = (
    base_modelo.loc[
        base_modelo[
            "elegivel_cruzamento_priorizacao"
        ],
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao"
        ]
    ]
    .copy()
)


resumo_cenarios = []


for nome_cenario, configuracao in CENARIOS_PESOS.items():

    # --------------------------------------------------------
    # Score de relevância
    # --------------------------------------------------------

    score_relevancia = pd.Series(
        0.0,
        index=resultado_sensibilidade.index
    )


    for coluna, peso in configuracao[
        "relevancia"
    ].items():

        score_relevancia = (
            score_relevancia
            +
            base_modelo.loc[
                resultado_sensibilidade.index,
                coluna
            ]
            *
            peso
        )


    # --------------------------------------------------------
    # Score de pressão
    # --------------------------------------------------------

    score_pressao = pd.Series(
        0.0,
        index=resultado_sensibilidade.index
    )


    for coluna, peso in configuracao[
        "pressao"
    ].items():

        score_pressao = (
            score_pressao
            +
            base_modelo.loc[
                resultado_sensibilidade.index,
                coluna
            ]
            *
            peso
        )


    # --------------------------------------------------------
    # Limiares próprios do cenário
    # --------------------------------------------------------

    limiar_relevancia = (
        score_relevancia.median()
    )

    limiar_pressao = (
        score_pressao.median()
    )


    # --------------------------------------------------------
    # Quadrante estratégico do cenário
    # --------------------------------------------------------

    flag_estrategico = (
        (score_relevancia >= limiar_relevancia)
        &
        (score_pressao >= limiar_pressao)
    )


    # --------------------------------------------------------
    # Salvar
    # --------------------------------------------------------

    resultado_sensibilidade[
        f"score_relevancia__{nome_cenario}"
    ] = score_relevancia


    resultado_sensibilidade[
        f"score_pressao__{nome_cenario}"
    ] = score_pressao


    resultado_sensibilidade[
        f"estrategico__{nome_cenario}"
    ] = flag_estrategico


    resumo_cenarios.append(
        {
            "cenario":
                nome_cenario,

            "limiar_relevancia":
                limiar_relevancia,

            "limiar_pressao":
                limiar_pressao,

            "municipios_estrategicos":
                flag_estrategico.sum()
        }
    )


resumo_cenarios = pd.DataFrame(
    resumo_cenarios
)


print("=" * 70)
print("RESUMO DOS CENÁRIOS")
print("=" * 70)


display(
    resumo_cenarios
)

RESUMO DOS CENÁRIOS


,cenario,limiar_relevancia,limiar_pressao,municipios_estrategicos
0,base,50.428878,49.342615,225
1,produtivo,50.710329,49.342615,250
2,ambiental,50.428878,48.466830,242
3,climatico,50.428878,49.909923,247


In [81]:
# ============================================================
# ESTABILIDADE DO QUADRANTE ESTRATÉGICO
# ============================================================

base_estrategicos = set(
    resultado_sensibilidade.loc[
        resultado_sensibilidade[
            "estrategico__base"
        ],
        "codigo_ibge"
    ]
)


estabilidade_cenarios = []


for nome_cenario in CENARIOS_PESOS.keys():

    estrategicos_cenario = set(
        resultado_sensibilidade.loc[
            resultado_sensibilidade[
                f"estrategico__{nome_cenario}"
            ],
            "codigo_ibge"
        ]
    )


    intersecao = (
        base_estrategicos
        &
        estrategicos_cenario
    )


    uniao = (
        base_estrategicos
        |
        estrategicos_cenario
    )


    # --------------------------------------------------------
    # Correlação dos scores com o cenário base
    # --------------------------------------------------------

    correlacao_relevancia = (
        resultado_sensibilidade[
            [
                "score_relevancia__base",
                f"score_relevancia__{nome_cenario}"
            ]
        ]
        .corr(
            method="spearman"
        )
        .iloc[
            0,
            1
        ]
    )


    correlacao_pressao = (
        resultado_sensibilidade[
            [
                "score_pressao__base",
                f"score_pressao__{nome_cenario}"
            ]
        ]
        .corr(
            method="spearman"
        )
        .iloc[
            0,
            1
        ]
    )


    estabilidade_cenarios.append(
        {
            "cenario":
                nome_cenario,

            "estrategicos_cenario":
                len(
                    estrategicos_cenario
                ),

            "estrategicos_em_comum_com_base":
                len(
                    intersecao
                ),

            "retencao_dos_225_base_pct":
                (
                    len(
                        intersecao
                    )
                    /
                    len(
                        base_estrategicos
                    )
                    *
                    100
                ),

            "jaccard_pct":
                (
                    len(
                        intersecao
                    )
                    /
                    len(
                        uniao
                    )
                    *
                    100
                ),

            "spearman_relevancia_vs_base":
                correlacao_relevancia,

            "spearman_pressao_vs_base":
                correlacao_pressao
        }
    )


estabilidade_cenarios = (
    pd.DataFrame(
        estabilidade_cenarios
    )
)


print("=" * 70)
print("ESTABILIDADE — TESTE DE SENSIBILIDADE")
print("=" * 70)


display(
    estabilidade_cenarios
)

ESTABILIDADE — TESTE DE SENSIBILIDADE


,cenario,estrategicos_cenario,estrategicos_em_comum_com_base,retencao_dos_225_base_pct,jaccard_pct,spearman_relevancia_vs_base,spearman_pressao_vs_base
0,base,225,225,100.000000,100.000000,1.000000,1.000000
1,produtivo,250,216,96.000000,83.397683,0.985597,1.000000
2,ambiental,242,191,84.888889,69.202899,1.000000,0.917803
3,climatico,247,207,92.000000,78.113208,1.000000,0.949328


In [82]:
# ============================================================
# ROBUSTEZ DA PRIORIZAÇÃO POR MUNICÍPIO
# ============================================================

COLUNAS_CENARIOS_ESTRATEGICOS = [
    f"estrategico__{cenario}"
    for cenario in CENARIOS_PESOS.keys()
]


robustez_municipal = (
    resultado_sensibilidade[
        [
            "codigo_ibge",
            *COLUNAS_CENARIOS_ESTRATEGICOS
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Quantidade de cenários em que o município aparece
# no quadrante estratégico
# ------------------------------------------------------------

robustez_municipal[
    "quantidade_cenarios_estrategico"
] = (
    robustez_municipal[
        COLUNAS_CENARIOS_ESTRATEGICOS
    ]
    .sum(
        axis=1
    )
    .astype(int)
)


# ------------------------------------------------------------
# Faixa de robustez
# ------------------------------------------------------------

MAPA_ROBUSTEZ = {
    4: "Muito robusta — 4/4 cenários",
    3: "Robusta — 3/4 cenários",
    2: "Sensível — 2/4 cenários",
    1: "Baixa robustez — 1/4 cenários",
    0: "Fora do estratégico — 0/4 cenários"
}


robustez_municipal[
    "faixa_robustez_priorizacao"
] = (
    robustez_municipal[
        "quantidade_cenarios_estrategico"
    ]
    .map(
        MAPA_ROBUSTEZ
    )
)


print("=" * 70)
print("ROBUSTEZ MUNICIPAL — CENÁRIOS DE SENSIBILIDADE")
print("=" * 70)


display(
    robustez_municipal[
        "faixa_robustez_priorizacao"
    ]
    .value_counts()
)


print(
    "\nDistribuição do número de cenários:"
)


display(
    robustez_municipal[
        "quantidade_cenarios_estrategico"
    ]
    .value_counts()
    .sort_index()
)

ROBUSTEZ MUNICIPAL — CENÁRIOS DE SENSIBILIDADE


faixa_robustez_priorizacao
Fora do estratégico — 0/4 cenários    1067
Muito robusta — 4/4 cenários           171
Baixa robustez — 1/4 cenários           92
Robusta — 3/4 cenários                  48
Sensível — 2/4 cenários                 22
Name: count, dtype: int64


Distribuição do número de cenários:


quantidade_cenarios_estrategico
0    1067
1      92
2      22
3      48
4     171
Name: count, dtype: int64

In [83]:
# ============================================================
# INTEGRAR ROBUSTEZ AO MODELO MUNICIPAL
# ============================================================

base_modelo = (
    base_modelo
    .merge(
        robustez_municipal,
        on="codigo_ibge",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# Municípios não elegíveis não participaram da sensibilidade
# ------------------------------------------------------------

base_modelo.loc[
    ~base_modelo[
        "elegivel_cruzamento_priorizacao"
    ],
    "faixa_robustez_priorizacao"
] = "Não aplicável — dados insuficientes"


print("=" * 70)
print("ROBUSTEZ — QUADRANTE ESTRATÉGICO BASE")
print("=" * 70)


estrategicos_base_final = (
    base_modelo.loc[
        base_modelo[
            "quadrante_priorizacao"
        ]
        .eq(
            "Alta relevância + Alta pressão"
        )
    ]
    .copy()
)


print(
    "\nMunicípios estratégicos:",
    len(
        estrategicos_base_final
    )
)


display(
    estrategicos_base_final[
        "faixa_robustez_priorizacao"
    ]
    .value_counts()
)


print(
    "\nRobustez × Confiança:"
)


tabela_robustez_confianca = (
    pd.crosstab(
        estrategicos_base_final[
            "faixa_robustez_priorizacao"
        ],
        estrategicos_base_final[
            "faixa_confianca_modelo"
        ]
    )
)


display(
    tabela_robustez_confianca
)

ROBUSTEZ — QUADRANTE ESTRATÉGICO BASE

Municípios estratégicos: 225


faixa_robustez_priorizacao
Muito robusta — 4/4 cenários     171
Robusta — 3/4 cenários            48
Sensível — 2/4 cenários            5
Baixa robustez — 1/4 cenários      1
Name: count, dtype: int64


Robustez × Confiança:


faixa_confianca_modelo,Alta,Limitada,Moderada
faixa_robustez_priorizacao,,,
Baixa robustez — 1/4 cenários,0,0,1
Muito robusta — 4/4 cenários,83,53,35
Robusta — 3/4 cenários,33,0,15
Sensível — 2/4 cenários,3,0,2


In [84]:
# ============================================================
# AUDITORIA FINAL — MODELO MUNICIPAL
# ============================================================

print("=" * 70)
print("AUDITORIA FINAL — BASE MODELO")
print("=" * 70)


print(
    "\nDimensão:",
    base_modelo.shape
)


print(
    "Municípios:",
    base_modelo[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "Duplicatas codigo_ibge:",
    base_modelo
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nElegíveis para o modelo:"
)


display(
    base_modelo[
        "elegivel_cruzamento_priorizacao"
    ]
    .value_counts()
)


print(
    "\nQuadrantes:"
)


display(
    base_modelo[
        "quadrante_priorizacao"
    ]
    .value_counts()
)


print(
    "\nConfiança:"
)


display(
    base_modelo[
        "faixa_confianca_modelo"
    ]
    .value_counts()
)


print(
    "\nRobustez:"
)


display(
    base_modelo[
        "faixa_robustez_priorizacao"
    ]
    .value_counts()
)


# ------------------------------------------------------------
# Verificar infinitos nas colunas numéricas
# ------------------------------------------------------------

colunas_numericas_modelo = (
    base_modelo
    .select_dtypes(
        include=np.number
    )
    .columns
)


quantidade_inf = (
    np.isinf(
        base_modelo[
            colunas_numericas_modelo
        ]
    )
    .sum()
    .sum()
)


print(
    "\nValores infinitos:",
    quantidade_inf
)


# ------------------------------------------------------------
# Scores só podem existir para municípios elegíveis
# ------------------------------------------------------------

falhas_score_fora_modelo = (
    base_modelo.loc[
        ~base_modelo[
            "elegivel_cruzamento_priorizacao"
        ],
        [
            "score_relevancia_produtiva",
            "score_pressao_agroambiental"
        ]
    ]
    .notna()
    .any(
        axis=1
    )
    .sum()
)


print(
    "Scores indevidos fora do universo elegível:",
    falhas_score_fora_modelo
)


# ------------------------------------------------------------
# Elegíveis devem possuir os dois scores
# ------------------------------------------------------------

falhas_score_dentro_modelo = (
    base_modelo.loc[
        base_modelo[
            "elegivel_cruzamento_priorizacao"
        ],
        [
            "score_relevancia_produtiva",
            "score_pressao_agroambiental"
        ]
    ]
    .isna()
    .any(
        axis=1
    )
    .sum()
)


print(
    "Elegíveis sem score:",
    falhas_score_dentro_modelo
)

AUDITORIA FINAL — BASE MODELO

Dimensão: (1505, 91)
Municípios: 1505
Duplicatas codigo_ibge: 0

Elegíveis para o modelo:


elegivel_cruzamento_priorizacao
True     1400
False     105
Name: count, dtype: int64


Quadrantes:


quadrante_priorizacao
Alta relevância + Baixa pressão     475
Baixa relevância + Alta pressão     475
Alta relevância + Alta pressão      225
Baixa relevância + Baixa pressão    225
Dados insuficientes                 105
Name: count, dtype: int64


Confiança:


faixa_confianca_modelo
Alta                                   781
Moderada                               532
Não aplicável — dados insuficientes    105
Limitada                                87
Name: count, dtype: int64


Robustez:


faixa_robustez_priorizacao
Fora do estratégico — 0/4 cenários     1067
Muito robusta — 4/4 cenários            171
Não aplicável — dados insuficientes     105
Baixa robustez — 1/4 cenários            92
Robusta — 3/4 cenários                   48
Sensível — 2/4 cenários                  22
Name: count, dtype: int64


Valores infinitos: 0
Scores indevidos fora do universo elegível: 0
Elegíveis sem score: 0


In [85]:
# ============================================================
# PREPARAÇÃO DAS SAÍDAS FINAIS
# ============================================================

# ------------------------------------------------------------
# Flag explícita do quadrante estratégico no cenário-base
# ------------------------------------------------------------

base_modelo[
    "flag_prioridade_estrategica_base"
] = (
    base_modelo[
        "quadrante_priorizacao"
    ]
    .eq(
        "Alta relevância + Alta pressão"
    )
)


# ------------------------------------------------------------
# Robustez forte:
# aparece como estratégico em 3 ou 4 dos 4 cenários
# ------------------------------------------------------------

base_modelo[
    "flag_prioridade_robusta_3_ou_4_cenarios"
] = (
    base_modelo[
        "quantidade_cenarios_estrategico"
    ]
    .ge(3)
    .fillna(False)
)


# ============================================================
# TABELA ESPECÍFICA DOS 225 ESTRATÉGICOS
# ============================================================

priorizacao_estrategica_final = (
    base_modelo.loc[
        base_modelo[
            "flag_prioridade_estrategica_base"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Índice apenas para ordenação exploratória DENTRO do
# quadrante estratégico.
#
# Não interpretar como potencial de crédito de carbono.
# ------------------------------------------------------------

priorizacao_estrategica_final[
    "indice_exploratorio_priorizacao"
] = (
    priorizacao_estrategica_final[
        [
            "score_relevancia_produtiva",
            "score_pressao_agroambiental"
        ]
    ]
    .mean(
        axis=1
    )
)


ORDEM_CONFIANCA_EXPORTACAO = {
    "Alta": 1,
    "Moderada": 2,
    "Limitada": 3
}


priorizacao_estrategica_final[
    "ordem_confianca"
] = (
    priorizacao_estrategica_final[
        "faixa_confianca_modelo"
    ]
    .map(
        ORDEM_CONFIANCA_EXPORTACAO
    )
)


priorizacao_estrategica_final = (
    priorizacao_estrategica_final
    .sort_values(
        by=[
            "ordem_confianca",
            "quantidade_cenarios_estrategico",
            "indice_exploratorio_priorizacao"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Ordem exploratória — NÃO é ranking de crédito de carbono
# ------------------------------------------------------------

priorizacao_estrategica_final[
    "ordem_exploratoria_quadrante"
] = (
    np.arange(
        1,
        len(
            priorizacao_estrategica_final
        ) + 1
    )
)


print("=" * 70)
print("SAÍDAS FINAIS PREPARADAS")
print("=" * 70)


print(
    "\nBase municipal:",
    base_modelo.shape
)


print(
    "Prioridade estratégica:",
    priorizacao_estrategica_final.shape
)


print(
    "\nEstratégicos robustos em 3 ou 4 cenários:"
)


print(
    priorizacao_estrategica_final[
        "flag_prioridade_robusta_3_ou_4_cenarios"
    ]
    .sum()
)


print(
    "\nDistribuição de confiança:"
)


display(
    priorizacao_estrategica_final[
        "faixa_confianca_modelo"
    ]
    .value_counts()
)

SAÍDAS FINAIS PREPARADAS

Base municipal: (1505, 93)
Prioridade estratégica: (225, 96)

Estratégicos robustos em 3 ou 4 cenários:
219

Distribuição de confiança:


faixa_confianca_modelo
Alta        119
Moderada     53
Limitada     53
Name: count, dtype: int64

In [86]:
# ============================================================
# EXPORTAÇÃO DAS BASES FINAIS — NOTEBOOK 11
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# Localizar raiz do projeto
# ------------------------------------------------------------

caminho_atual = Path.cwd()


candidatos_raiz = [
    caminho_atual,
    *caminho_atual.parents
]


RAIZ_PROJETO = next(
    (
        caminho
        for caminho in candidatos_raiz
        if (
            (caminho / "data").exists()
            and
            (caminho / "notebooks").exists()
        )
    ),
    None
)


if RAIZ_PROJETO is None:

    raise FileNotFoundError(
        "Não foi possível localizar automaticamente "
        "a raiz do projeto."
    )


PASTA_SAIDA_PRIORIZACAO = (
    RAIZ_PROJETO
    /
    "data"
    /
    "databases_curated"
    /
    "priorizacao_agroambiental"
)


PASTA_SAIDA_PRIORIZACAO.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("PASTA DE SAÍDA")
print("=" * 70)


print(
    PASTA_SAIDA_PRIORIZACAO
)


# ============================================================
# ARQUIVOS
# ============================================================

ARQUIVO_BASE_MODELO = (
    PASTA_SAIDA_PRIORIZACAO
    /
    "base_modelo_priorizacao_agroambiental_soja_centro_oeste_sul_municipio.csv"
)


ARQUIVO_ESTRATEGICOS = (
    PASTA_SAIDA_PRIORIZACAO
    /
    "municipios_prioridade_estrategica_soja_centro_oeste_sul.csv"
)


ARQUIVO_SENSIBILIDADE = (
    PASTA_SAIDA_PRIORIZACAO
    /
    "teste_sensibilidade_priorizacao_soja_centro_oeste_sul.csv"
)


ARQUIVO_RESUMO_SENSIBILIDADE = (
    PASTA_SAIDA_PRIORIZACAO
    /
    "resumo_sensibilidade_priorizacao_soja_centro_oeste_sul.csv"
)


# ============================================================
# EXPORTAR
# ============================================================

base_modelo.to_csv(
    ARQUIVO_BASE_MODELO,
    index=False,
    encoding="utf-8-sig"
)


priorizacao_estrategica_final.to_csv(
    ARQUIVO_ESTRATEGICOS,
    index=False,
    encoding="utf-8-sig"
)


resultado_sensibilidade.to_csv(
    ARQUIVO_SENSIBILIDADE,
    index=False,
    encoding="utf-8-sig"
)


estabilidade_cenarios.to_csv(
    ARQUIVO_RESUMO_SENSIBILIDADE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "\nArquivos exportados:"
)


for arquivo in [
    ARQUIVO_BASE_MODELO,
    ARQUIVO_ESTRATEGICOS,
    ARQUIVO_SENSIBILIDADE,
    ARQUIVO_RESUMO_SENSIBILIDADE
]:

    print(
        "->",
        arquivo.name
    )

PASTA DE SAÍDA
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\priorizacao_agroambiental

Arquivos exportados:
-> base_modelo_priorizacao_agroambiental_soja_centro_oeste_sul_municipio.csv
-> municipios_prioridade_estrategica_soja_centro_oeste_sul.csv
-> teste_sensibilidade_priorizacao_soja_centro_oeste_sul.csv
-> resumo_sensibilidade_priorizacao_soja_centro_oeste_sul.csv


In [87]:
# ============================================================
# RECARREGAR E VALIDAR ARQUIVOS EXPORTADOS
# ============================================================

base_modelo_recarregada = pd.read_csv(
    ARQUIVO_BASE_MODELO,
    dtype={
        "codigo_ibge": "string"
    }
)


estrategicos_recarregada = pd.read_csv(
    ARQUIVO_ESTRATEGICOS,
    dtype={
        "codigo_ibge": "string"
    }
)


sensibilidade_recarregada = pd.read_csv(
    ARQUIVO_SENSIBILIDADE,
    dtype={
        "codigo_ibge": "string"
    }
)


resumo_sensibilidade_recarregado = pd.read_csv(
    ARQUIVO_RESUMO_SENSIBILIDADE
)


print("=" * 70)
print("VALIDAÇÃO DOS ARQUIVOS EXPORTADOS")
print("=" * 70)


print(
    "\nBase modelo:",
    base_modelo_recarregada.shape
)


print(
    "Municípios:",
    base_modelo_recarregada[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "Duplicatas:",
    base_modelo_recarregada
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nEstratégicos:",
    estrategicos_recarregada.shape
)


print(
    "Municípios estratégicos únicos:",
    estrategicos_recarregada[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nSensibilidade:",
    sensibilidade_recarregada.shape
)


print(
    "\nResumo sensibilidade:",
    resumo_sensibilidade_recarregado.shape
)


# ------------------------------------------------------------
# Controles principais do modelo salvo
# ------------------------------------------------------------

print(
    "\nElegíveis salvos:"
)

print(
    base_modelo_recarregada[
        "elegivel_cruzamento_priorizacao"
    ]
    .sum()
)


print(
    "\nEstratégicos salvos:"
)

print(
    base_modelo_recarregada[
        "flag_prioridade_estrategica_base"
    ]
    .sum()
)


print(
    "\nEstratégicos robustos 3/4 ou 4/4:"
)

print(
    estrategicos_recarregada[
        "flag_prioridade_robusta_3_ou_4_cenarios"
    ]
    .sum()
)


# ------------------------------------------------------------
# Controle estrutural final
# ------------------------------------------------------------

assert len(
    base_modelo_recarregada
) == 1505


assert (
    base_modelo_recarregada[
        "codigo_ibge"
    ]
    .nunique()
    == 1505
)


assert (
    base_modelo_recarregada
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
    == 0
)


assert (
    base_modelo_recarregada[
        "elegivel_cruzamento_priorizacao"
    ]
    .sum()
    == 1400
)


assert (
    base_modelo_recarregada[
        "flag_prioridade_estrategica_base"
    ]
    .sum()
    == 225
)


assert (
    len(
        estrategicos_recarregada
    )
    == 225
)


print(
    "\nTodas as validações estruturais passaram."
)

VALIDAÇÃO DOS ARQUIVOS EXPORTADOS

Base modelo: (1505, 93)
Municípios: 1505
Duplicatas: 0

Estratégicos: (225, 96)
Municípios estratégicos únicos: 225

Sensibilidade: (1400, 16)

Resumo sensibilidade: (4, 7)

Elegíveis salvos:
1400

Estratégicos salvos:
225

Estratégicos robustos 3/4 ou 4/4:
219

Todas as validações estruturais passaram.


# Conclusão — Indicadores e Priorização Agroambiental

Este notebook estruturou uma camada analítica municipal para priorização agroambiental da produção de soja nas regiões Centro-Oeste e Sul do Brasil.

A análise partiu da base agroambiental anual integrada, contendo informações de produção agrícola, clima, carbono do solo, cobertura e uso da terra, emissões associadas a resíduos agrícolas de soja e contexto histórico de mudança de uso da terra.

## Unidade de análise

A base anual possui granularidade `município + ano`.

Para a priorização, foi criada uma camada específica com granularidade:

**1 linha = 1 município**

A transformação permitiu resumir o comportamento temporal observado no período disponível e evitar que municípios com maior quantidade de anos fossem indevidamente sobrerrepresentados.

## Universo municipal

A base municipal final contém:

- 1.505 municípios;
- 1.400 municípios elegíveis para o modelo completo;
- 105 municípios classificados como dados insuficientes para o cruzamento integral dos eixos.

Os municípios não elegíveis foram preservados na base e não receberam valores artificiais para os indicadores ausentes.

## Redução de redundâncias

Antes da construção dos scores, foram avaliadas relações estatísticas e matemáticas entre os indicadores.

Entre os principais resultados observados:

- emissões SEEG absolutas apresentaram relação praticamente perfeita com a escala de produção neste recorte;
- produção e área colhida apresentaram forte associação;
- carbono do solo médio e carbono do solo de 2024 apresentaram forte redundância;
- produção atual apresentou forte associação com área histórica absoluta de conversão para soja;
- SOC e Ctotal do BRLUC apresentaram redundância decorrente do componente Cveg constante;
- delta SOC e delta Ctotal do BRLUC foram matematicamente equivalentes no recorte analisado.

Variáveis redundantes foram mantidas como contexto quando úteis, mas não receberam pesos independentes no modelo.

## Arquitetura da priorização

A classificação foi estruturada em dois eixos independentes.

### Relevância produtiva

O eixo representa a importância relativa do município para a produção de soja e utiliza:

- escala média de produção;
- rendimento médio;
- participação relativa da soja no território.

Os três componentes receberam pesos iguais.

### Pressão agroambiental

O eixo representa sinais relativos de pressão ou vulnerabilidade associados ao sistema produtivo e utiliza quatro subdimensões:

1. instabilidade produtiva;
2. variabilidade climática;
3. mudanças ambientais recentes;
4. contexto histórico BRLUC.

As quatro subdimensões receberam pesos iguais.

Os indicadores internos de cada subdimensão também foram agregados de forma balanceada, evitando que um tema recebesse maior peso apenas por possuir maior quantidade de variáveis.

## Normalização

Os indicadores foram transformados para escala relativa de 0 a 100 utilizando posição percentual/ranking dentro do mesmo universo de 1.400 municípios elegíveis.

Essa estratégia reduz a influência de distribuições assimétricas e valores extremos.

A normalização foi aplicada respeitando a direção interpretativa de cada variável.

Exemplos:

- maior produção → maior relevância produtiva;
- maior variabilidade → maior pressão;
- variações ambientais mais negativas → maior pressão relativa.

## Quadrantes

Os municípios foram classificados utilizando as medianas dos dois scores como limiares.

Foram obtidos:

- 225 municípios em **Alta relevância + Alta pressão**;
- 475 municípios em **Alta relevância + Baixa pressão**;
- 475 municípios em **Baixa relevância + Alta pressão**;
- 225 municípios em **Baixa relevância + Baixa pressão**;
- 105 municípios com **dados insuficientes**.

O grupo **Alta relevância + Alta pressão** foi denominado quadrante estratégico para aprofundamento de diagnóstico agroambiental.

Essa classificação não representa certificação, elegibilidade ou quantidade potencial de créditos de carbono.

## Qualidade e confiança

A confiança da classificação foi avaliada separadamente dos scores ambientais.

Foram consideradas principalmente:

- cobertura temporal;
- pior qualidade climática espacial observada;
- disponibilidade do intervalo de confiança da taxa BRLUC;
- ocorrência de intervalo de confiança da taxa BRLUC cruzando zero.

A confiança não modifica os scores de relevância ou pressão.

Ela funciona como uma ressalva sobre a qualidade da evidência utilizada na classificação.

Entre os 225 municípios estratégicos:

- 119 apresentaram confiança alta;
- 53 apresentaram confiança moderada;
- 53 apresentaram confiança limitada.

## Sensibilidade aos pesos

Foram avaliados quatro cenários:

- cenário-base com pesos iguais;
- cenário com maior peso à escala produtiva;
- cenário com maior peso às mudanças ambientais;
- cenário com maior peso à variabilidade climática.

A retenção dos 225 municípios estratégicos do cenário-base variou entre aproximadamente 84,9% e 96,0% nos cenários alternativos.

As correlações de Spearman dos scores alternativos com o cenário-base permaneceram acima de 0,91.

Dos 225 municípios estratégicos do cenário-base:

- 171 permaneceram estratégicos em 4 de 4 cenários;
- 48 permaneceram em 3 de 4 cenários;
- 5 permaneceram em 2 de 4 cenários;
- 1 permaneceu em apenas 1 de 4 cenários.

Assim, 219 dos 225 municípios, aproximadamente 97,3%, permaneceram estratégicos em pelo menos três dos quatro cenários analisados.

A análise indica estabilidade da classificação frente a alterações razoáveis de ponderação, sem implicar que os pesos adotados sejam únicos ou definitivos.

## Interpretação

A priorização deve ser interpretada como uma ferramenta de triagem e apoio à decisão.

Um município classificado como **Alta relevância + Alta pressão** representa um território que combina elevada importância relativa para a soja com sinais comparativamente elevados de pressão agroambiental segundo os indicadores utilizados.

Isso pode justificar aprofundamento de análises, diagnóstico de práticas agrícolas, conservação do solo, uso da terra, resiliência climática e oportunidades de intervenção.

A classificação não significa que o município possua créditos de carbono disponíveis para comercialização.

Uma eventual avaliação de projetos de carbono exige etapas adicionais, incluindo definição de projeto e área, linha de base, adicionalidade, metodologia aplicável, monitoramento, relato e verificação (MRV), permanência, vazamento e demais requisitos do padrão ou mercado utilizado.

## Arquivos finais

Os produtos consolidados deste notebook foram exportados para:

`data/databases_curated/priorizacao_agroambiental/`

Arquivos:

`base_modelo_priorizacao_agroambiental_soja_centro_oeste_sul_municipio.csv`

Base municipal completa para análise, banco de dados e visualização.

`municipios_prioridade_estrategica_soja_centro_oeste_sul.csv`

Recorte dos 225 municípios classificados como Alta relevância + Alta pressão no cenário-base.

`teste_sensibilidade_priorizacao_soja_centro_oeste_sul.csv`

Resultados municipais dos cenários alternativos de ponderação.

`resumo_sensibilidade_priorizacao_soja_centro_oeste_sul.csv`

Resumo da estabilidade da classificação frente aos cenários avaliados.

Todos os arquivos foram recarregados após a exportação e passaram pelos controles estruturais definidos no notebook.